# FIFA World Cup 2026 — Player Statistics

## Objective

This notebook builds the player-level statistics dataset for the FIFA World Cup 2026 analytics project.

The notebook will:

- Use the existing player information as the foundation.
- Collect and organize player performance statistics.
- Combine statistics from the available datasets.
- Calculate useful derived performance metrics.
- Validate player and team relationships.
- Produce a clean, dashboard-ready player statistics table for Power BI.

## Main Statistics

### Player Information
- PlayerID
- Player Name
- Team
- TeamID
- Position
- Shirt Number

### Match Participation
- Appearances
- Starts
- Minutes Played
- Substitute Appearances

### Attacking
- Goals
- Assists
- Shots
- Shots on Target
- Goal Contributions

### Passing & Creation
- Passes
- Pass Accuracy
- Key Passes
- Chances Created
- Crosses

### Defending
- Tackles
- Tackles Won
- Interceptions
- Clearances
- Blocks
- Recoveries
- Duels
- Duels Won
- Aerial Duels
- Aerial Duels Won

### Discipline
- Yellow Cards
- Red Cards
- Fouls Committed
- Fouls Suffered
- Offsides

### Goalkeeping
- Saves
- Goals Conceded
- Clean Sheets
- Save Percentage
- Penalties Saved

### Derived Metrics
- Goals per 90
- Assists per 90
- Goal Contributions per 90
- Pass Accuracy %
- Duel Win %
- Aerial Duel Win %

## Validation

The final dataset will be checked for:

- Duplicate players
- Missing PlayerIDs
- Unmatched players
- Player-Team inconsistencies
- Invalid or impossible values
- Missing statistics

## Final Output

The final result will be a clean player statistics table that can be connected to the FIFA World Cup 2026 Power BI data model and used for player performance analysis.

## 1. Inspect Available Data

Before building the player statistics table, we first need to identify the datasets available to us.

This step will:

- List all files in the data directory.
- Identify the datasets related to players and statistics.
- Confirm the file names before loading anything.
- Prevent us from duplicating or assuming data that we do not have.

In [32]:
from pathlib import Path

DATA_DIR = Path("../Data")

print("Data directory:")
print(DATA_DIR.resolve())

print("\nAvailable files:")
for file in sorted(DATA_DIR.iterdir()):
    print("-", file.name)

Data directory:
D:\ai course hw\FIFA_WorldCup_2026_Project\Data

Available files:
- FIFA_WorldCup_2026_SquadLists.pdf
- Goals.csv
- Matches.csv
- MatchStatistics.csv
- PlayerPhysicalStats.csv
- Players.csv
- Stadiums.csv
- Teams.csv


## 2. Load Player Statistics Data

We will now load the datasets that can contribute to the player statistics analysis.

### Primary player datasets

- `Players.csv` — player master information.
- `MatchStatistics.csv` — match-level statistics.
- `Goals.csv` — goal events that can be used to calculate player goals.
- `PlayerPhysicalStats.csv` — player physical information.

We will inspect the structure of each dataset before making any transformations.

At this stage, we will not modify the data.

In [33]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Set data directory
DATA_DIR = Path("../Data")

# Load datasets
players_df = pd.read_csv(DATA_DIR / "Players.csv")
match_stats_df = pd.read_csv(DATA_DIR / "MatchStatistics.csv")
goals_df = pd.read_csv(DATA_DIR / "Goals.csv")
physical_df = pd.read_csv(DATA_DIR / "PlayerPhysicalStats.csv")

# Display dataset sizes
print("Players:", players_df.shape)
print("Match Statistics:", match_stats_df.shape)
print("Goals:", goals_df.shape)
print("Physical Statistics:", physical_df.shape)

# Display columns
print("\nPlayers columns:")
print(players_df.columns.tolist())

print("\nMatch Statistics columns:")
print(match_stats_df.columns.tolist())

print("\nGoals columns:")
print(goals_df.columns.tolist())

print("\nPhysical Statistics columns:")
print(physical_df.columns.tolist())

Players: (1248, 5)
Match Statistics: (208, 11)
Goals: (308, 6)
Physical Statistics: (1029, 21)

Players columns:
['PlayerID', 'PlayerName', 'TeamID', 'Position', 'JerseyNumber']

Match Statistics columns:
['StatisticID', 'MatchID', 'TeamID', 'Shots', 'ShotsOnTarget', 'Possession', 'Corners', 'Fouls', 'YellowCards', 'RedCards', 'Offsides']

Goals columns:
['GoalID', 'MatchID', 'PlayerID', 'TeamID', 'Minute', 'GoalType']

Physical Statistics columns:
['PlayerID', 'Matches', 'DistanceMeters', 'Sprints', 'HighSpeedRunning', 'TopSpeedKmh', 'SpeedZone1', 'SpeedZone2', 'SpeedZone3', 'SpeedZone4', 'SpeedZone5', 'LineBreaksAttempted', 'LineBreaksCompleted', 'LineBreaksAccuracyPct', 'Through', 'Around', 'Over', 'Pass', 'Crosses', 'Prog', 'TotalCrosses']


## 3. Inspect Player Master Data

The `Players.csv` file is the foundation of our player-level dataset.

It contains:

- `PlayerID` — unique identifier for each player.
- `PlayerName` — player's name.
- `TeamID` — player's team identifier.
- `Position` — player's playing position.
- `JerseyNumber` — player's shirt number.

Before combining it with goals and physical statistics, we will check:

- Number of players
- Missing values
- Duplicate PlayerIDs
- Data types
- A sample of the records

This is an inspection step only; no data will be changed.

In [34]:
# Inspect player master data

print("Shape:", players_df.shape)

print("\nData types:")
print(players_df.dtypes)

print("\nMissing values:")
print(players_df.isnull().sum())

print("\nDuplicate PlayerIDs:")
print(players_df["PlayerID"].duplicated().sum())

print("\nFirst 10 players:")
display(players_df.head(10))

Shape: (1248, 5)

Data types:
PlayerID         int64
PlayerName      object
TeamID           int64
Position        object
JerseyNumber     int64
dtype: object

Missing values:
PlayerID        0
PlayerName      0
TeamID          0
Position        0
JerseyNumber    0
dtype: int64

Duplicate PlayerIDs:
0

First 10 players:


,PlayerID,PlayerName,TeamID,Position,JerseyNumber
0,1,Melvin Mastil,25,GK,1
1,2,Aissa Mandi,25,DF,2
2,3,Achref Abada,25,DF,3
3,4,Mohamed Amine Tougai,25,DF,4
4,5,Zineddine Belaid,25,DF,5
5,6,Ramiz Zerrouki,25,MF,6
6,7,Riyad Mahrez,25,FW,7
7,8,Houssem Aouar,25,MF,8
8,9,Amine Gouiri,25,FW,9
9,10,Fares Chaibi,25,MF,10


## 4. Inspect Goal Data

The `Goals.csv` dataset contains individual goal events.

We will use it to determine player scoring statistics.

Available fields:

- `GoalID` — unique goal identifier.
- `MatchID` — match where the goal occurred.
- `PlayerID` — player who scored.
- `TeamID` — scoring player's team.
- `Minute` — minute of the goal.
- `GoalType` — type of goal.

Before calculating player goals, we will validate:

- Missing values
- Duplicate GoalIDs
- Duplicate goal events
- PlayerID validity
- TeamID validity
- GoalType values
- Goal minute values

No data will be modified in this step.

In [35]:
# Inspect goal data

print("Shape:", goals_df.shape)

print("\nData types:")
print(goals_df.dtypes)

print("\nMissing values:")
print(goals_df.isnull().sum())

print("\nDuplicate GoalIDs:")
print(goals_df["GoalID"].duplicated().sum())

print("\nUnique GoalTypes:")
print(goals_df["GoalType"].value_counts(dropna=False))

print("\nGoal minute range:")
print("Minimum:", goals_df["Minute"].min())
print("Maximum:", goals_df["Minute"].max())

print("\nPlayerIDs in Goals not found in Players:")
print(
    goals_df.loc[
        ~goals_df["PlayerID"].isin(players_df["PlayerID"]),
        "PlayerID"
    ].nunique()
)

print("\nTeamIDs in Goals not found in Players:")
print(
    goals_df.loc[
        ~goals_df["TeamID"].isin(players_df["TeamID"]),
        "TeamID"
    ].nunique()
)

print("\nFirst 10 goals:")
display(goals_df.head(10))

Shape: (308, 6)

Data types:
GoalID       int64
MatchID      int64
PlayerID     int64
TeamID       int64
Minute      object
GoalType    object
dtype: object

Missing values:
GoalID        0
MatchID       0
PlayerID      0
TeamID        0
Minute        0
GoalType    278
dtype: int64

Duplicate GoalIDs:
0

Unique GoalTypes:
GoalType
NaN         278
Penalty      16
Own Goal     14
Name: count, dtype: int64

Goal minute range:
Minimum: 10
Maximum: 93

PlayerIDs in Goals not found in Players:
0

TeamIDs in Goals not found in Players:
0

First 10 goals:


,GoalID,MatchID,PlayerID,TeamID,Minute,GoalType
0,1,1,718,8,9,NaN
1,2,1,711,8,67,NaN
2,3,2,682,31,67,NaN
3,4,2,694,31,80,NaN
4,5,2,371,38,59,NaN
5,6,25,382,38,6,NaN
6,7,25,1018,24,83,Penalty
7,8,28,709,8,50,NaN
8,9,53,722,8,55,NaN
9,10,53,718,8,61,NaN


## 5. Inspect Goal Minute and Goal Type

The previous inspection showed that:

- `Minute` is stored as an object rather than a numeric value.
- Most `GoalType` values are missing.

Before cleaning these fields, we need to understand exactly what values they contain.

We will inspect:

- All unique `GoalType` values.
- Examples of `Minute` values.
- The number of unique minute values.
- Any unusual minute formats.
- The relationship between `GoalType` and the recorded goals.

We will not make assumptions about missing `GoalType` values.

In [36]:
# Inspect GoalType values
print("GoalType value counts:")
print(goals_df["GoalType"].value_counts(dropna=False))

# Inspect Minute values
print("\nSample Minute values:")
print(goals_df["Minute"].head(30).tolist())

print("\nUnique Minute values (first 50):")
print(goals_df["Minute"].drop_duplicates().head(50).tolist())

print("\nNumber of unique Minute values:")
print(goals_df["Minute"].nunique())

# Inspect data types of values inside Minute
print("\nPython types found in Minute:")
print(goals_df["Minute"].map(type).value_counts())

# Display goal type + minute together
print("\nGoalType and Minute sample:")
display(goals_df[["GoalID", "PlayerID", "Minute", "GoalType"]].head(20))

GoalType value counts:
GoalType
NaN         278
Penalty      16
Own Goal     14
Name: count, dtype: int64

Sample Minute values:
['9', '67', '67', '80', '59', '6', '83', '50', '55', '61', '90+4', '63', '78', '21', '90+4', '17', '74', '84', '90', '90+7', '90+3', '16', '29', '45+3', '64', '75', '90+2', '46', '57', '76']

Unique Minute values (first 50):
['9', '67', '80', '59', '6', '83', '50', '55', '61', '90+4', '63', '78', '21', '17', '74', '84', '90', '90+7', '90+3', '16', '29', '45+3', '64', '75', '90+2', '46', '57', '76', '34', '42', '32', '28', '2', '23', '36', '7', '60', '39', '45+1', '89', '10', '43', '31', '45+5', '90+8', '73', '27', '11', '3', '49']

Number of unique Minute values:
108

Python types found in Minute:
Minute
<class 'str'>    308
Name: count, dtype: int64

GoalType and Minute sample:


,GoalID,PlayerID,Minute,GoalType
0,1,718,9,NaN
1,2,711,67,NaN
2,3,682,67,NaN
3,4,694,80,NaN
4,5,371,59,NaN
5,6,382,6,NaN
6,7,1018,83,Penalty
7,8,709,50,NaN
8,9,722,55,NaN
9,10,718,61,NaN


## 6. Clean Goal Minute

The `Minute` column contains football match-time notation.

Examples:

- `9` → 9th minute
- `45+3` → 45th minute + 3 minutes stoppage time
- `90+4` → 90th minute + 4 minutes stoppage time

We will create two fields:

- `GoalMinute` — the original football notation, preserved as text.
- `GoalMinuteNumeric` — a numeric representation useful for sorting and analysis.

For example:

`90+4` → `94`

`45+3` → `48`

The original value will be preserved so we do not lose the actual match notation.

In [37]:
# Preserve the original football minute notation
goals_df["GoalMinute"] = goals_df["Minute"].astype(str)

# Convert football minute notation to a numeric value
def convert_goal_minute(value):
    value = str(value).strip()
    
    if "+" in value:
        base, added = value.split("+", 1)
        return int(base) + int(added)
    
    return int(value)

goals_df["GoalMinuteNumeric"] = goals_df["GoalMinute"].apply(convert_goal_minute)

# Check the result
print("Original vs numeric minute:")
display(
    goals_df[["GoalID", "GoalMinute", "GoalMinuteNumeric"]].head(20)
)

print("\nNumeric minute range:")
print("Minimum:", goals_df["GoalMinuteNumeric"].min())
print("Maximum:", goals_df["GoalMinuteNumeric"].max())

print("\nMissing numeric minutes:")
print(goals_df["GoalMinuteNumeric"].isnull().sum())

Original vs numeric minute:


,GoalID,GoalMinute,GoalMinuteNumeric
0,1,9,9
1,2,67,67
2,3,67,67
3,4,80,80
4,5,59,59
5,6,6,6
6,7,83,83
7,8,50,50
8,9,55,55
9,10,61,61



Numeric minute range:
Minimum: 2
Maximum: 125

Missing numeric minutes:
0


## 7. Calculate Player Goal Statistics

We will aggregate the goal events by `PlayerID`.

The goal summary will contain:

- `PlayerID`
- `Goals` — total goal events recorded for the player.
- `PenaltyGoals` — goals explicitly identified as penalties.
- `OwnGoals` — goals explicitly identified as own goals.

We will keep these categories separate because an own goal should not automatically be treated as a goal scored by the player for attacking-performance analysis.

The resulting table will be validated before it is merged with the main player table.

In [38]:
# Create player goal statistics

player_goals_df = (
    goals_df
    .groupby("PlayerID")
    .agg(
        Goals=("GoalID", "count"),
        PenaltyGoals=("GoalType", lambda x: (x == "Penalty").sum()),
        OwnGoals=("GoalType", lambda x: (x == "Own Goal").sum())
    )
    .reset_index()
)

# Calculate regular/unspecified goals
player_goals_df["RegularGoals"] = (
    player_goals_df["Goals"]
    - player_goals_df["PenaltyGoals"]
    - player_goals_df["OwnGoals"]
)

# Display results
print("Player goal statistics shape:", player_goals_df.shape)

print("\nTotal goal events:", player_goals_df["Goals"].sum())
print("Total penalty goals:", player_goals_df["PenaltyGoals"].sum())
print("Total own goals:", player_goals_df["OwnGoals"].sum())
print("Total regular/unspecified goals:", player_goals_df["RegularGoals"].sum())

print("\nTop 10 players by goal events:")
display(
    player_goals_df
    .sort_values("Goals", ascending=False)
    .head(10)
)

print("\nMissing PlayerIDs:")
print(player_goals_df["PlayerID"].isnull().sum())

Player goal statistics shape: (190, 5)

Total goal events: 308
Total penalty goals: 16
Total own goals: 14
Total regular/unspecified goals: 278

Top 10 players by goal events:


,PlayerID,Goals,PenaltyGoals,OwnGoals,RegularGoals
80,478,10,1,0,9
6,36,8,0,0,8
76,452,7,0,0,7
128,815,7,0,0,7
79,475,6,0,0,6
75,451,6,2,0,4
161,1061,5,1,0,4
111,718,4,0,0,4
149,1006,4,0,0,4
30,163,4,0,0,4



Missing PlayerIDs:
0


## 8. Validate Player and Goal Relationships

Before merging the goal statistics into the player master table, we will validate the relationship between `Goals.csv` and `Players.csv`.

We will check:

- Whether every goal PlayerID exists in Players.csv.
- Whether the PlayerID belongs to the same TeamID recorded in the goal event.
- How many goal events have a Player-Team mismatch.
- Whether any player has goals recorded for multiple teams.

This is important because PlayerID is our primary player key, while TeamID provides an additional consistency check.

In [39]:
# Validate PlayerID and TeamID relationships

goal_player_check = goals_df.merge(
    players_df[["PlayerID", "TeamID", "PlayerName"]],
    on="PlayerID",
    how="left",
    suffixes=("_Goal", "_Player")
)

# Check missing players
missing_players = goal_player_check["PlayerName"].isna().sum()

# Check TeamID mismatches
team_mismatch = (
    goal_player_check["TeamID_Goal"] != goal_player_check["TeamID_Player"]
).sum()

# Players with goals recorded for multiple teams
multi_team_players = (
    goals_df.groupby("PlayerID")["TeamID"]
    .nunique()
    .loc[lambda x: x > 1]
)

print("Goal events:", len(goal_player_check))
print("Goal events with missing PlayerID match:", missing_players)
print("Goal events with Player-Team mismatch:", team_mismatch)
print("Players recorded with goals for multiple teams:", len(multi_team_players))

print("\nTeam mismatches:")
display(
    goal_player_check[
        goal_player_check["TeamID_Goal"] != goal_player_check["TeamID_Player"]
    ][
        ["GoalID", "PlayerID", "PlayerName", "TeamID_Goal", "TeamID_Player"]
    ].head(20)
)

Goal events: 308
Goal events with missing PlayerID match: 0
Goal events with Player-Team mismatch: 14
Players recorded with goals for multiple teams: 2

Team mismatches:


,GoalID,PlayerID,PlayerName,TeamID_Goal,TeamID_Player
14,15,1094,Miro Muheim,44,11
25,26,936,Mohamed Manai,16,44
31,32,911,Mahmoud Abunada,26,44
48,49,729,Yassine Bounou,39,21
50,51,874,Damian Bobadilla,12,32
57,58,73,Cameron Burgess,12,13
105,106,1135,Ellyes Skhiri,9,46
108,109,419,Mohamed Hany,2,19
131,132,941,Hassan Altambakti,10,45
145,146,616,Aymen Hussein,22,40


## 9. Investigate Player-Team Mismatches

The previous validation identified 14 goal events where the `TeamID` in
`Goals.csv` does not match the player's `TeamID` in `Players.csv`.

We will compare each mismatched goal with the teams participating in that match.

For each record, we will check:

- GoalID
- MatchID
- PlayerID
- PlayerName
- Goal TeamID
- Player TeamID
- Match Team 1
- Match Team 2
- Goal minute
- Goal type

This will help determine whether the mismatch is a data-quality issue.

We will not modify the data until the cause is understood.

In [40]:
# Investigate Player-Team mismatches

# Get the mismatched goal events
mismatched_goals = goal_player_check[
    goal_player_check["TeamID_Goal"] != goal_player_check["TeamID_Player"]
].copy()

# Add match information
mismatched_goals = mismatched_goals.merge(
    matches_df[
        ["MatchID", "Team1ID", "Team2ID"]
    ],
    on="MatchID",
    how="left"
)

print("Mismatched goal events:", len(mismatched_goals))

# Check whether the Goal TeamID actually participated in the match
mismatched_goals["GoalTeamInMatch"] = (
    (mismatched_goals["TeamID_Goal"] == mismatched_goals["Team1ID"]) |
    (mismatched_goals["TeamID_Goal"] == mismatched_goals["Team2ID"])
)

# Check whether the Player's TeamID participated in the match
mismatched_goals["PlayerTeamInMatch"] = (
    (mismatched_goals["TeamID_Player"] == mismatched_goals["Team1ID"]) |
    (mismatched_goals["TeamID_Player"] == mismatched_goals["Team2ID"])
)

display(
    mismatched_goals[
        [
            "GoalID",
            "MatchID",
            "PlayerID",
            "PlayerName",
            "TeamID_Goal",
            "TeamID_Player",
            "Team1ID",
            "Team2ID",
            "GoalMinute",
            "GoalType",
            "GoalTeamInMatch",
            "PlayerTeamInMatch"
        ]
    ]
)

print("\nGoal TeamID found in match:")
print(mismatched_goals["GoalTeamInMatch"].value_counts())

print("\nPlayer TeamID found in match:")
print(mismatched_goals["PlayerTeamInMatch"].value_counts())

Mismatched goal events: 14


,GoalID,MatchID,PlayerID,PlayerName,TeamID_Goal,TeamID_Player,Team1ID,Team2ID,GoalMinute,GoalType,GoalTeamInMatch,PlayerTeamInMatch
0,15,8,1094,Miro Muheim,44,11,44,11,90+4,Own Goal,True,True
1,26,27,936,Mohamed Manai,16,44,16,44,75,Own Goal,True,True
2,32,52,911,Mahmoud Abunada,26,44,26,44,34,Own Goal,True,True
3,49,50,729,Yassine Bounou,39,21,21,39,10,Own Goal,True,True
4,51,4,874,Damian Bobadilla,12,32,12,32,7,Own Goal,True,True
5,58,32,73,Cameron Burgess,12,13,12,13,11,Own Goal,True,True
6,106,58,1135,Ellyes Skhiri,9,46,46,9,3,Own Goal,True,True
7,109,16,419,Mohamed Hany,2,19,2,19,66,Own Goal,True,True
8,132,38,941,Hassan Altambakti,10,45,10,45,49,Own Goal,True,True
9,146,18,616,Aymen Hussein,22,40,40,22,90+6,Own Goal,True,True



Goal TeamID found in match:
GoalTeamInMatch
True    14
Name: count, dtype: int64

Player TeamID found in match:
PlayerTeamInMatch
True    14
Name: count, dtype: int64


## 10. Build Final Player Goal Statistics

The investigation showed that all 14 Player-Team mismatches are caused by
own-goal events.

For an own goal:

- `PlayerID` identifies the player credited with the own goal.
- `TeamID` identifies the team that received the goal.
- Therefore, the goal should not be counted as an attacking goal for the player.

We will create the final player goal statistics using:

- `Goals` — regular goals + penalty goals.
- `PenaltyGoals` — goals identified as penalties.
- `OwnGoals` — own goals credited to the player, tracked separately.
- `RegularGoals` — non-penalty, non-own-goal goals.

This gives us a correct foundation for later player performance metrics.

In [41]:
# Rebuild player goal statistics with own goals excluded from Goals

player_goals_df = (
    goals_df
    .groupby("PlayerID")
    .agg(
        TotalGoalEvents=("GoalID", "count"),
        PenaltyGoals=("GoalType", lambda x: (x == "Penalty").sum()),
        OwnGoals=("GoalType", lambda x: (x == "Own Goal").sum())
    )
    .reset_index()
)

# Regular goals = everything that is not a penalty or own goal
player_goals_df["RegularGoals"] = (
    player_goals_df["TotalGoalEvents"]
    - player_goals_df["PenaltyGoals"]
    - player_goals_df["OwnGoals"]
)

# Attacking goals exclude own goals
player_goals_df["Goals"] = (
    player_goals_df["TotalGoalEvents"]
    - player_goals_df["OwnGoals"]
)

# Reorder columns
player_goals_df = player_goals_df[
    [
        "PlayerID",
        "Goals",
        "RegularGoals",
        "PenaltyGoals",
        "OwnGoals",
        "TotalGoalEvents"
    ]
]

print("Total goal events:", player_goals_df["TotalGoalEvents"].sum())
print("Attacking goals:", player_goals_df["Goals"].sum())
print("Regular goals:", player_goals_df["RegularGoals"].sum())
print("Penalty goals:", player_goals_df["PenaltyGoals"].sum())
print("Own goals:", player_goals_df["OwnGoals"].sum())

print("\nCheck:")
print(
    "Regular + Penalty =",
    player_goals_df["RegularGoals"].sum()
    + player_goals_df["PenaltyGoals"].sum()
)

print("\nTop 10 players by attacking goals:")
display(
    player_goals_df
    .sort_values("Goals", ascending=False)
    .head(10)
)

Total goal events: 308
Attacking goals: 294
Regular goals: 278
Penalty goals: 16
Own goals: 14

Check:
Regular + Penalty = 294

Top 10 players by attacking goals:


,PlayerID,Goals,RegularGoals,PenaltyGoals,OwnGoals,TotalGoalEvents
80,478,10,9,1,0,10
6,36,8,8,0,0,8
76,452,7,7,0,0,7
128,815,7,7,0,0,7
79,475,6,6,0,0,6
75,451,6,4,2,0,6
161,1061,5,4,1,0,5
111,718,4,4,0,0,4
149,1006,4,4,0,0,4
30,163,4,4,0,0,4


## 11. Inspect Player Physical & Performance Statistics

The `PlayerPhysicalStats.csv` dataset contains player-level performance and
physical statistics.

It includes:

### Playing Time
- Matches

### Physical Performance
- DistanceMeters
- Sprints
- HighSpeedRunning
- TopSpeedKmh
- SpeedZone1
- SpeedZone2
- SpeedZone3
- SpeedZone4
- SpeedZone5

### Passing / Line Breaking
- LineBreaksAttempted
- LineBreaksCompleted
- LineBreaksAccuracyPct
- Through
- Around
- Over
- Pass

### Crossing & Progression
- Crosses
- Prog
- TotalCrosses

Before using these statistics, we will inspect:

- Data types
- Missing values
- Duplicate PlayerIDs
- PlayerID matches with Players.csv
- Numeric ranges
- A sample of the data

No data will be modified in this step.

In [42]:
# Inspect player physical and performance statistics

print("Shape:", physical_df.shape)

print("\nData types:")
print(physical_df.dtypes)

print("\nMissing values:")
print(physical_df.isnull().sum())

print("\nDuplicate PlayerIDs:")
print(physical_df["PlayerID"].duplicated().sum())

# Check PlayerID relationships
physical_player_ids = physical_df["PlayerID"]

print("\nPhysical PlayerIDs not found in Players.csv:")
print(
    (~physical_player_ids.isin(players_df["PlayerID"])).sum()
)

print("\nPlayers.csv PlayerIDs without physical statistics:")
print(
    (~players_df["PlayerID"].isin(physical_player_ids)).sum()
)

print("\nPhysical statistics summary:")
display(physical_df.describe())

print("\nFirst 10 records:")
display(physical_df.head(10))

Shape: (1029, 21)

Data types:
PlayerID                   int64
Matches                    int64
DistanceMeters           float64
Sprints                    int64
HighSpeedRunning         float64
TopSpeedKmh              float64
SpeedZone1               float64
SpeedZone2               float64
SpeedZone3               float64
SpeedZone4               float64
SpeedZone5               float64
LineBreaksAttempted        int64
LineBreaksCompleted        int64
LineBreaksAccuracyPct    float64
Through                    int64
Around                     int64
Over                       int64
Pass                       int64
Crosses                    int64
Prog                       int64
TotalCrosses               int64
dtype: object

Missing values:
PlayerID                 0
Matches                  0
DistanceMeters           0
Sprints                  0
HighSpeedRunning         0
TopSpeedKmh              0
SpeedZone1               0
SpeedZone2               0
SpeedZone3               0
Sp

,PlayerID,Matches,DistanceMeters,Sprints,HighSpeedRunning,TopSpeedKmh,SpeedZone1,SpeedZone2,SpeedZone3,SpeedZone4,...,LineBreaksAttempted,LineBreaksCompleted,LineBreaksAccuracyPct,Through,Around,Over,Pass,Crosses,Prog,TotalCrosses
count,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,...,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000,1029.000000
mean,615.522838,3.165209,22980.451992,89.211856,252.992225,31.168999,9091.715549,9375.678814,2947.319048,1143.294752,...,26.246842,18.474247,61.928260,6.306122,10.259475,9.681244,24.013605,0.753158,1.480078,2.510204
std,358.617322,1.661511,17307.216553,71.929087,204.800638,2.798899,7198.738517,7392.939803,2406.811056,928.610176,...,28.071014,22.132045,26.896872,8.221198,13.366639,12.931782,26.777906,1.516223,2.472581,3.924772
min,2.000000,1.000000,90.000000,0.000000,1.000000,18.100000,28.200000,22.200000,15.300000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,303.000000,2.000000,8464.500000,30.000000,80.000000,29.900000,3062.300000,3118.700000,946.900000,380.800000,...,5.000000,3.000000,50.000000,1.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000
50%,617.000000,3.000000,20433.200000,76.000000,212.000000,31.700000,8058.100000,8233.500000,2479.600000,987.200000,...,17.000000,11.000000,68.090000,3.000000,5.000000,5.000000,15.000000,0.000000,1.000000,1.000000
75%,918.000000,4.000000,32898.200000,131.000000,370.000000,33.000000,13237.000000,13862.300000,4306.400000,1678.400000,...,38.000000,25.000000,80.000000,8.000000,14.000000,13.000000,35.000000,1.000000,2.000000,3.000000
max,1248.000000,8.000000,96233.700000,403.000000,1294.000000,37.600000,41334.400000,44470.300000,15304.900000,5484.800000,...,187.000000,178.000000,100.000000,78.000000,117.000000,111.000000,182.000000,15.000000,24.000000,29.000000



First 10 records:


,PlayerID,Matches,DistanceMeters,Sprints,HighSpeedRunning,TopSpeedKmh,SpeedZone1,SpeedZone2,SpeedZone3,SpeedZone4,...,LineBreaksAttempted,LineBreaksCompleted,LineBreaksAccuracyPct,Through,Around,Over,Pass,Crosses,Prog,TotalCrosses
0,1056,8,96233.7,318,1294.0,31.5,30824.8,44470.3,15304.9,4504.4,...,187,171,91.44,78,57,52,182,0,5,3
1,46,8,90561.9,277,1024.0,30.2,32710.0,41710.2,11818.2,3470.1,...,76,63,82.89,31,31,14,76,0,0,1
2,1064,8,87986.1,363,907.0,33.3,35446.1,35667.5,10387.8,4750.8,...,109,87,79.82,20,60,29,104,1,4,15
3,1062,8,85956.1,263,816.0,32.4,37264.2,34965.3,9070.3,3338.2,...,186,178,95.70,51,117,18,177,1,8,1
4,479,8,84499.7,388,1054.0,34.7,26863.1,37368.0,12546.8,5484.8,...,133,103,77.44,52,56,25,117,3,13,4
5,50,7,82829.0,332,1054.0,32.6,29517.4,35612.2,12078.1,4210.7,...,124,111,89.52,34,69,21,119,1,4,5
6,1054,8,79572.0,204,695.0,33.0,35442.8,32810.7,7926.5,2683.1,...,182,160,87.91,42,100,40,174,0,8,0
7,450,8,78698.0,282,1006.0,29.9,26107.9,36039.5,11896.2,3870.1,...,112,89,79.46,44,34,34,106,2,4,3
8,451,7,78564.7,260,859.0,32.4,30358.0,32799.7,10316.8,4078.3,...,36,24,66.67,13,10,13,34,0,2,3
9,1100,6,73750.5,213,1001.0,29.4,27160.3,32542.3,10968.9,2684.3,...,69,59,85.51,18,38,13,66,1,2,4


## 12. Validate Player Physical Statistics

The physical statistics dataset contains 1,029 player records, while the
player master table contains 1,248 players.

The difference does not automatically mean that data is wrong. Some players
may have no recorded physical statistics.

Before merging the dataset, we will specifically check:

- Duplicate PlayerIDs.
- Missing values in each statistic.
- PlayerIDs that do not exist in Players.csv.
- Players without physical statistics.
- Whether each player has only one physical-statistics record.

This will determine whether the table can safely be merged to the player
master table using `PlayerID`.

In [43]:
# Validate physical statistics data

print("=== Physical Statistics Validation ===")

# Duplicate PlayerIDs
duplicate_count = physical_df["PlayerID"].duplicated().sum()
print("\nDuplicate PlayerIDs:", duplicate_count)

# Missing values
print("\nMissing values:")
display(physical_df.isnull().sum().to_frame("MissingCount"))

# PlayerIDs not found in Players.csv
unknown_physical_players = physical_df.loc[
    ~physical_df["PlayerID"].isin(players_df["PlayerID"]),
    "PlayerID"
]

print("\nPhysical PlayerIDs not found in Players.csv:",
      unknown_physical_players.nunique())

# Players without physical statistics
players_without_physical = players_df.loc[
    ~players_df["PlayerID"].isin(physical_df["PlayerID"]),
    "PlayerID"
]

print("Players without physical statistics:",
      players_without_physical.nunique())

# Check how many physical records each PlayerID has
records_per_player = physical_df.groupby("PlayerID").size()

print("\nRecords per PlayerID:")
print(records_per_player.value_counts().sort_index())

print("\nPlayers with more than one physical-statistics record:")
print((records_per_player > 1).sum())

=== Physical Statistics Validation ===

Duplicate PlayerIDs: 0

Missing values:


,MissingCount
PlayerID,0
Matches,0
DistanceMeters,0
Sprints,0
HighSpeedRunning,0
TopSpeedKmh,0
SpeedZone1,0
SpeedZone2,0
SpeedZone3,0
SpeedZone4,0



Physical PlayerIDs not found in Players.csv: 0
Players without physical statistics: 219

Records per PlayerID:
1    1029
Name: count, dtype: int64

Players with more than one physical-statistics record:
0


## 13. Build the Player Statistics Base Table

We will now combine the validated player-level datasets.

The `Players.csv` table will remain the foundation because it contains all
1,248 players.

We will add:

- Goal statistics from `player_goals_df`.
- Physical and performance statistics from `physical_df`.

The merge will use `PlayerID`.

A `left join` will be used so that players who do not have recorded physical
statistics are still retained in the final player table.

Players without physical statistics will have missing values for those
specific fields rather than being removed.

In [44]:
# Start with the complete player master table
player_stats_df = players_df.copy()

# Add goal statistics
player_stats_df = player_stats_df.merge(
    player_goals_df,
    on="PlayerID",
    how="left"
)

# Add physical and performance statistics
player_stats_df = player_stats_df.merge(
    physical_df,
    on="PlayerID",
    how="left"
)

print("Final player statistics shape:", player_stats_df.shape)

print("\nExpected players:", len(players_df))
print("Actual players:", player_stats_df["PlayerID"].nunique())

print("\nDuplicate PlayerIDs after merge:",
      player_stats_df["PlayerID"].duplicated().sum())

print("\nColumns:")
print(player_stats_df.columns.tolist())

print("\nFirst 10 records:")
display(player_stats_df.head(10))

Final player statistics shape: (1248, 30)

Expected players: 1248
Actual players: 1248

Duplicate PlayerIDs after merge: 0

Columns:
['PlayerID', 'PlayerName', 'TeamID', 'Position', 'JerseyNumber', 'Goals', 'RegularGoals', 'PenaltyGoals', 'OwnGoals', 'TotalGoalEvents', 'Matches', 'DistanceMeters', 'Sprints', 'HighSpeedRunning', 'TopSpeedKmh', 'SpeedZone1', 'SpeedZone2', 'SpeedZone3', 'SpeedZone4', 'SpeedZone5', 'LineBreaksAttempted', 'LineBreaksCompleted', 'LineBreaksAccuracyPct', 'Through', 'Around', 'Over', 'Pass', 'Crosses', 'Prog', 'TotalCrosses']

First 10 records:


,PlayerID,PlayerName,TeamID,Position,JerseyNumber,Goals,RegularGoals,PenaltyGoals,OwnGoals,TotalGoalEvents,...,LineBreaksAttempted,LineBreaksCompleted,LineBreaksAccuracyPct,Through,Around,Over,Pass,Crosses,Prog,TotalCrosses
0,1,Melvin Mastil,25,GK,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Aissa Mandi,25,DF,2,NaN,NaN,NaN,NaN,NaN,...,82.0,66.0,80.49,16.0,43.0,23.0,82.0,0.0,0.0,0.0
2,3,Achref Abada,25,DF,3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Mohamed Amine Tougai,25,DF,4,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Zineddine Belaid,25,DF,5,NaN,NaN,NaN,NaN,NaN,...,3.0,2.0,66.67,1.0,0.0,2.0,3.0,0.0,0.0,0.0
5,6,Ramiz Zerrouki,25,MF,6,NaN,NaN,NaN,NaN,NaN,...,25.0,22.0,88.00,10.0,10.0,5.0,25.0,0.0,0.0,0.0
6,7,Riyad Mahrez,25,FW,7,2.0,2.0,0.0,0.0,2.0,...,31.0,18.0,58.06,5.0,13.0,13.0,27.0,2.0,2.0,2.0
7,8,Houssem Aouar,25,MF,8,NaN,NaN,NaN,NaN,NaN,...,13.0,13.0,100.00,8.0,4.0,1.0,11.0,0.0,2.0,1.0
8,9,Amine Gouiri,25,FW,9,1.0,1.0,0.0,0.0,1.0,...,2.0,1.0,50.00,1.0,1.0,0.0,1.0,0.0,1.0,1.0
9,10,Fares Chaibi,25,MF,10,NaN,NaN,NaN,NaN,NaN,...,47.0,35.0,74.47,16.0,17.0,14.0,43.0,2.0,2.0,5.0


## 14. Handle Missing Goal Statistics

Not every player appears in `Goals.csv`.

When a player has no goal record, the goal aggregation produces a missing
value after the left join.

For player performance analysis, these missing goal values should represent
zero rather than an unknown value.

We will therefore replace missing values with `0` for:

- Goals
- RegularGoals
- PenaltyGoals
- OwnGoals
- TotalGoalEvents

We will keep the physical statistics as missing for players who have no
physical-statistics record, because missing physical data means that the
statistics were not recorded for that player.

In [45]:
# Fill missing goal statistics with zero

goal_columns = [
    "Goals",
    "RegularGoals",
    "PenaltyGoals",
    "OwnGoals",
    "TotalGoalEvents"
]

player_stats_df[goal_columns] = (
    player_stats_df[goal_columns]
    .fillna(0)
    .astype(int)
)

print("Missing goal statistics after cleaning:")

display(
    player_stats_df[goal_columns]
    .isnull()
    .sum()
    .to_frame("MissingCount")
)

print("\nGoal statistics data types:")
print(player_stats_df[goal_columns].dtypes)

print("\nPlayers with zero goals:",
      (player_stats_df["Goals"] == 0).sum())

print("Players with at least one goal:",
      (player_stats_df["Goals"] > 0).sum())

Missing goal statistics after cleaning:


,MissingCount
Goals,0
RegularGoals,0
PenaltyGoals,0
OwnGoals,0
TotalGoalEvents,0



Goal statistics data types:
Goals              int64
RegularGoals       int64
PenaltyGoals       int64
OwnGoals           int64
TotalGoalEvents    int64
dtype: object

Players with zero goals: 1070
Players with at least one goal: 178


## 15. Validate Player Match Participation

The player physical statistics dataset contains a `Matches` field.

This field will be used as the available player match-participation measure.

At this stage, we will not create:

- Starts
- Minutes Played
- Substitute Appearances

because those fields are not available in the current player-level datasets.

We will validate the `Matches` field by checking:

- Missing values
- Minimum and maximum values
- Players with zero matches
- Players with the highest number of matches

This ensures that the field is suitable for player-level analysis.

In [46]:
# Validate player match participation

matches_stats = player_stats_df["Matches"]

print("Missing Matches:", matches_stats.isna().sum())

print("\nPlayers with physical match data:",
      matches_stats.notna().sum())

print("Players without physical match data:",
      matches_stats.isna().sum())

print("\nMatches statistics:")
print("Minimum:", matches_stats.min())
print("Maximum:", matches_stats.max())
print("Average:", matches_stats.mean())

print("\nPlayers with zero recorded matches:",
      (matches_stats == 0).sum())

print("\nTop 10 players by recorded matches:")

display(
    player_stats_df[
        ["PlayerID", "PlayerName", "TeamID", "Position", "Matches"]
    ]
    .dropna(subset=["Matches"])
    .sort_values("Matches", ascending=False)
    .head(10)
)

Missing Matches: 219

Players with physical match data: 1029
Players without physical match data: 219

Matches statistics:
Minimum: 1.0
Maximum: 8.0
Average: 3.1652089407191446

Players with zero recorded matches: 0

Top 10 players by recorded matches:


,PlayerID,PlayerName,TeamID,Position,Matches
447,448,Marc Guehi,5,DF,8.0
449,450,Elliot Anderson,5,MF,8.0
479,480,Bradley Barcola,6,FW,8.0
477,478,Kylian Mbappe,6,FW,8.0
451,452,Jude Bellingham,5,MF,8.0
483,484,Mike Maignan,6,GK,8.0
478,479,Michael Olise,6,FW,8.0
45,46,Alexis Mac Allister,1,MF,8.0
487,488,Desire Doue,6,FW,8.0
48,49,Emiliano Martinez,1,GK,8.0


## 16. Inspect Match Statistics Level

`MatchStatistics.csv` contains:

- MatchID
- TeamID
- Shots
- ShotsOnTarget
- Possession
- Corners
- Fouls
- YellowCards
- RedCards
- Offsides

It does not contain `PlayerID`.

Therefore, these statistics appear to be recorded at the
**team + match level**, not the player level.

We will verify this by checking:

- Number of rows per MatchID.
- Number of rows per MatchID + TeamID.
- Whether every match has multiple team records.
- Whether MatchID + TeamID is unique.

This will determine how we should use this dataset in the overall project.

In [47]:
# Inspect the level/granularity of MatchStatistics

print("Shape:", match_stats_df.shape)

# Number of records per match
records_per_match = match_stats_df.groupby("MatchID").size()

print("\nRecords per MatchID:")
print(records_per_match.value_counts().sort_index())

# Number of records per Match + Team
records_per_match_team = (
    match_stats_df
    .groupby(["MatchID", "TeamID"])
    .size()
)

print("\nDuplicate MatchID + TeamID combinations:",
      (records_per_match_team > 1).sum())

print("\nUnique teams per match:")
teams_per_match = (
    match_stats_df
    .groupby("MatchID")["TeamID"]
    .nunique()
)

print(teams_per_match.value_counts().sort_index())

print("\nFirst 10 Match Statistics records:")
display(match_stats_df.head(10))

Shape: (208, 11)

Records per MatchID:
2    104
Name: count, dtype: int64

Duplicate MatchID + TeamID combinations: 0

Unique teams per match:
TeamID
2    104
Name: count, dtype: int64

First 10 Match Statistics records:


,StatisticID,MatchID,TeamID,Shots,ShotsOnTarget,Possession,Corners,Fouls,YellowCards,RedCards,Offsides
0,1,1,8,16,4,61,3,12,1,1,1
1,2,1,24,3,2,40,1,11,2,2,1
2,3,2,31,15,6,62,4,9,1,0,2
3,4,2,38,7,4,39,5,16,0,0,2
4,5,25,38,14,3,38,5,12,1,0,2
5,6,25,24,17,4,62,5,10,2,0,3
6,7,28,8,8,4,42,1,9,0,0,3
7,8,28,31,9,2,58,2,7,2,0,6
8,9,53,38,13,1,52,5,9,0,0,1
9,10,53,8,11,5,48,1,13,1,0,0


## 17. Check for Player Assist Data

Assists are an important player-performance metric.

We need to determine whether the available datasets contain information that
can be used to identify the player who assisted each goal.

We will search the loaded datasets for columns or fields related to:

- Assist
- Assister
- Assisting Player
- Assist Player
- PlayerID associated with an assist

We will not calculate or invent assists unless the source data provides enough
information to do so reliably.

In [48]:
# Search all loaded datasets for assist-related columns

datasets = {
    "Players": players_df,
    "Goals": goals_df,
    "MatchStatistics": match_stats_df,
    "PhysicalStatistics": physical_df,
    "Matches": matches_df
}

assist_keywords = ["assist", "assister", "assistplayer", "assist_player"]

print("Assist-related columns found:\n")

found = False

for name, df in datasets.items():
    matching_columns = [
        col for col in df.columns
        if any(keyword in col.lower().replace(" ", "").replace("-", "_")
               for keyword in assist_keywords)
    ]
    
    if matching_columns:
        found = True
        print(f"{name}: {matching_columns}")

if not found:
    print("No assist-related columns found in the available datasets.")

print("\nGoals columns:")
print(goals_df.columns.tolist())

Assist-related columns found:

No assist-related columns found in the available datasets.

Goals columns:
['GoalID', 'MatchID', 'PlayerID', 'TeamID', 'Minute', 'GoalType', 'GoalMinute', 'GoalMinuteNumeric']


## 18. Test Alternative Player Statistics Source

FBref cannot be accessed from the notebook, so we will test another
public source for complete FIFA World Cup 2026 player statistics.

The source will be used only to retrieve player-level statistics.
We will inspect the returned page before extracting any data.

No data will be merged into the local player table yet.

In [50]:
# Test FootballCountry player statistics page

import requests

SOURCE_URL = "https://www.footballcountry.com/tournaments/fifa-world-cup/2026/stats"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(
    SOURCE_URL,
    headers=headers,
    timeout=30
)

print("HTTP Status:", response.status_code)
print("Downloaded characters:", len(response.text))

if response.status_code == 200:
    print("\nSource retrieved successfully.")
    print(response.text[:500])
else:
    print("\nSource could not be retrieved.")
    print(response.text[:300])

HTTP Status: 522
Downloaded characters: 7286

Source could not be retrieved.
<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!


## 19. Test Alternative World Cup Player Statistics Source

The previous sources could not be retrieved reliably from the notebook.

We will test another public football statistics source and inspect its
HTML response before extracting the player-level statistics.

No data will be merged into the local player table yet.

In [51]:
# Test WorldFootball.net

import requests

SOURCE_URL = "https://www.worldfootball.net/schedule/wm-2026-usa-can-mex/"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    )
}

response = requests.get(
    SOURCE_URL,
    headers=headers,
    timeout=30
)

print("HTTP Status:", response.status_code)
print("Downloaded characters:", len(response.text))

if response.status_code == 200:
    print("\nSource retrieved successfully.")
    print(response.text[:500])
else:
    print("\nSource could not be retrieved.")
    print(response.text[:300])

HTTP Status: 403
Downloaded characters: 5776

Source could not be retrieved.
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal


## 20. Retrieve Official FIFA Player Statistics

The official FIFA World Cup 2026 player-statistics page will be used as
the external source for missing player-level statistics.

Source:
FIFA World Cup 2026 – Player Statistics

We will first retrieve the FIFA page directly and inspect its response.

Because FIFA's statistics page is dynamically generated, we will not assume
that the statistics are contained in a normal HTML table.

If the page contains embedded data or an API endpoint, we will extract the
player statistics from that official FIFA source.

No data will be merged into `player_stats_df` yet.

In [52]:
# Test direct access to the official FIFA player statistics page

import requests

FIFA_URL = (
    "https://www.fifa.com/en/tournaments/mens/worldcup/"
    "canadamexicousa2026/statistics/player-statistics"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(
    FIFA_URL,
    headers=headers,
    timeout=30
)

print("HTTP Status:", response.status_code)
print("Downloaded characters:", len(response.text))
print("Final URL:", response.url)

print("\nResponse preview:")
print(response.text[:1000])

HTTP Status: 200
Downloaded characters: 4560
Final URL: https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics/player-statistics

Response preview:
<!doctype html><html lang="en" dir="ltr"><head><meta charset="utf-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/><meta name="theme-color" content="#020F2A"/><meta name="msapplication-TileImage" content="/mstile-150x150.png?v=6b9bed24d1df59ca113a3828909d3924"><meta name="msapplication-TileColor" content="#326295"/><meta name="google-site-verification" content="boCUJZPlju606sys-ZcnqZAThCMVJWEvwc6JXvElJTE"/><link rel="icon" href="/favicon.ico?v=4c4914f90c578869e7375b03cf029202"/><link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png?v=a087933e3cf148cb71a96095c8aa2dac"/><link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png?v=1ea068c804e8ba88b84f6e9598e3172d"/><link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png?v=7ee355e68b687435c3e5f74e2

## 21. Identify the FIFA Statistics Data Source

The official FIFA player-statistics page is accessible, but the returned
HTML does not contain the complete statistics table.

The page is dynamically generated, so we will inspect the HTML for:

- API endpoints
- Embedded JSON data
- Statistics service URLs
- JavaScript references containing the player-statistics data

Once the underlying FIFA data source is identified, we will retrieve the
complete player-level dataset directly from FIFA.

In [53]:
# Inspect FIFA HTML for possible API/data endpoints

import re

html = response.text

# Find URLs containing likely API/data/statistics keywords
patterns = [
    r'https?://[^"\']+',
    r'[^"\']*(?:api|graphql|statistics|player)[^"\']*'
]

matches = set()

for pattern in patterns:
    for match in re.findall(pattern, html, flags=re.IGNORECASE):
        matches.add(match)

print("Potential data/API references found:", len(matches))

for item in sorted(matches):
    print("\n", item[:500])

print("\n--- Script tags ---")

script_sources = re.findall(
    r'<script[^>]+src=["\']([^"\']+)["\']',
    html,
    flags=re.IGNORECASE
)

print("JavaScript files found:", len(script_sources))

for src in script_sources:
    print(src)

Potential data/API references found: 17

 SERVICE_API_URL

 https://api.fifa.com

 https://assets.adobedtm.com/7db34e84caae/759763b5faef/launch-ENf264c4ce0fad4ce4bf7503fd7be51245.min.js

 https://assets.pingone.com/davinci/latest/davinci.js

 https://auth.fifa.com/

 https://cdn.cookielaw.org

 https://cdn.cookielaw.org/consent/996debe1-708b-4ebc-95c8-5b7ee0d77d76/OtAutoBlock.js

 https://cdn.cookielaw.org/scripttemplates/otSDKStub.js

 https://cdn.theoplayer.com/conviva/conviva-4.0.12.js

 https://cxm-api.fifa.com/fifaplusweb/api

 https://cxm-api.fifa.com/fifaplusweb/api/themePalette

 https://digitalhub.fifa.com

 https://geolocation.onetrust.com/cookieconsentpub/v1/geo/location

 https://license.theoplayer.com

 https://play.fifa.com/components/main.bundle.js

 https://www.fifa.com/auth

 https://www.gstatic.com

--- Script tags ---
JavaScript files found: 6
https://cdn.cookielaw.org/consent/996debe1-708b-4ebc-95c8-5b7ee0d77d76/OtAutoBlock.js
https://cdn.cookielaw.org/scripttemplat

## 22. Inspect FIFA's Statistics API

The FIFA player-statistics page is dynamically generated.

The page source identifies FIFA's official API domain:

`https://api.fifa.com`

We will inspect FIFA's main JavaScript bundle to identify the exact
API endpoint and parameters used to retrieve the 2026 World Cup
player statistics.

We will use the discovered endpoint rather than scraping manually
rendered HTML.

In [54]:
# Download FIFA's main JavaScript bundle and search for API references

from urllib.parse import urljoin

# Get the JavaScript files from the FIFA page
script_sources = re.findall(
    r'<script[^>]+src=["\']([^"\']+)["\']',
    response.text,
    flags=re.IGNORECASE
)

print("JavaScript files found:", len(script_sources))

# Convert relative paths to full URLs
script_urls = [
    urljoin(FIFA_URL, src)
    for src in script_sources
]

print("\nJavaScript URLs:")
for i, url in enumerate(script_urls):
    print(i, url)

# Find the main application bundle
main_scripts = [
    url for url in script_urls
    if "main." in url or "/static/js/main" in url
]

print("\nMain application scripts:")
for url in main_scripts:
    print(url)

if main_scripts:
    js_url = main_scripts[0]

    js_response = requests.get(
        js_url,
        headers=headers,
        timeout=30
    )

    print("\nJavaScript HTTP Status:", js_response.status_code)
    print("JavaScript characters:", len(js_response.text))

    # Search the bundle for FIFA API references
    api_matches = sorted(
        set(
            re.findall(
                r'https?://[^"\']*api\.fifa\.com[^"\']*',
                js_response.text,
                flags=re.IGNORECASE
            )
        )
    )

    print("\nFIFA API references found:", len(api_matches))

    for item in api_matches[:100]:
        print(item[:500])

else:
    print("\nMain application JavaScript was not found.")

JavaScript files found: 6

JavaScript URLs:
0 https://cdn.cookielaw.org/consent/996debe1-708b-4ebc-95c8-5b7ee0d77d76/OtAutoBlock.js
1 https://cdn.cookielaw.org/scripttemplates/otSDKStub.js
2 https://assets.pingone.com/davinci/latest/davinci.js
3 https://www.fifa.com/smt-base-bridge.min.js?v=4a054409a11a928eedd65ef2c674a30c
4 https://www.fifa.com/static/js/main.b731dbb7.js
5 https://www.fifa.com/gaEsst7-htIQbc1Isy8ABK8OI8k/OY9mNGmwcaXSkJED5J/d2tkKgcmAQ/Bx/dPZSIDLE0B

Main application scripts:
https://www.fifa.com/static/js/main.b731dbb7.js

JavaScript HTTP Status: 200
JavaScript characters: 1073029

FIFA API references found: 7
https://api.fifa.com
https://cxm-api.fifa.com/fifacxmsearch/api
https://cxm-api.fifa.com/fifaplusweb/api
https://cxm-api.fifa.com/fifapluswebloyalty/api
https://fdh-api.fifa.com
https://rewards-api.fifa.com
https://rewards-api.fifa.com/fifa/live/components


## 23. Locate the FIFA Player Statistics API

The FIFA application JavaScript bundle was successfully downloaded.

The bundle contains references to FIFA's official API, but the API
endpoint path is constructed inside the JavaScript code.

We will search the bundle for terms related to:

- player statistics
- player-statistics
- statistics
- assists
- goals
- tournament/competition identifiers
- API request paths

This will allow us to identify the exact official FIFA endpoint.

In [55]:
# Search FIFA's JavaScript bundle for player-statistics/API patterns

js_text = js_response.text

search_terms = [
    "player-statistics",
    "playerStatistics",
    "player statistics",
    "statistics/player",
    "playerStats",
    "assists",
    "goals",
    "api.fifa.com"
]

for term in search_terms:
    positions = [
        m.start()
        for m in re.finditer(
            re.escape(term),
            js_text,
            flags=re.IGNORECASE
        )
    ]

    print(f"\n{'=' * 70}")
    print(f"TERM: {term}")
    print(f"MATCHES: {len(positions)}")

    # Show a few contexts around the first matches
    for pos in positions[:5]:
        start = max(0, pos - 500)
        end = min(len(js_text), pos + 1000)

        print("\n--- Context ---")
        print(js_text[start:end])


TERM: player-statistics
MATCHES: 0

TERM: playerStatistics
MATCHES: 0

TERM: player statistics
MATCHES: 0

TERM: statistics/player
MATCHES: 0

TERM: playerStats
MATCHES: 1

--- Context ---
}){switch(e){case i.FORWARD:return{personal:t.totalGoalsScoredLabel,competitor:t.comparedToTopGoalscorerLabel};case i.DEFENDER:return{personal:t.defensivePressureAppliedLabel,competitor:t.playerMostDefPressureAppliedLabel};case i.MIDFIELD:return{personal:t.totalGoalLineBreaksAllLinesLabel,competitor:t.playerMostLineBreaksAllLinesLabel};case i.GOALKEEPER:return{personal:t.totalGoalPreventionsLabel,competitor:t.playerMostGoalPreventionstLabel}}}({position:t,labels:e}),{seasonNameAbbreviation:u,topPlayerStats:c,playerSeasonStats:d}=n,f=function(e){let t,n;if(1===e.topPlayersCount){const[r]=e.topPlayers;t=(null===r||void 0===r?void 0:r.topPlayerName)||"",n=(null===r||void 0===r?void 0:r.topPlayerImageURL)||""}else t=`${e.topPlayersCount} Players`,n=o.__;const{value:r}=e.topPlayersStat;return{name:t,valu

## 24. Test the Official FIFA API

The FIFA player-statistics webpage uses FIFA's official API behind the
scenes.

The API is available under:

https://api.fifa.com/api/v3

We will test direct access to the FIFA API and inspect the response.

No player data will be modified or merged yet.

In [56]:
# Test direct access to FIFA's official API

FIFA_API_URL = "https://api.fifa.com/api/v3"

api_response = requests.get(
    FIFA_API_URL,
    headers=headers,
    timeout=30
)

print("HTTP Status:", api_response.status_code)
print("Downloaded characters:", len(api_response.text))

print("\nResponse preview:")
print(api_response.text[:1000])

HTTP Status: 503
Downloaded characters: 485

Response preview:
<HTML><HEAD>
<TITLE>Service Unavailable</TITLE>
</HEAD><BODY>
<H1>Service Unavailable - DNS failure</H1>
The server is temporarily unable to service your request.  Please try again
later.<P>
Reference&#32;&#35;11&#46;9571cd17&#46;1789118540&#46;d647482
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;11&#46;9571cd17&#46;1789118540&#46;d647482</P>
<script type="text/javascript"  src="/9xywJh/f/N/oDGzkHH_Wxar/5z1SkXQ49ur28r7f/J00FPXoLAw/BDcK/KHBVZykB"></script></BODY></HTML>



## 24. Load the Official FIFA Player Statistics Page

The FIFA statistics page is dynamically rendered, so direct `requests`
cannot retrieve the player table.

We will use Selenium to open the official FIFA page in a real browser,
allow the page to render, and then inspect the table.

The source remains the official FIFA World Cup 2026 statistics page.

We will first verify that Selenium can see the statistics table before
extracting any data.

In [57]:
# Test FIFA statistics page with Selenium

import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

FIFA_URL = (
    "https://www.fifa.com/en/tournaments/mens/worldcup/"
    "canadamexicousa2026/statistics/player-statistics"
)

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1600,1200")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/139.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:
    print("Opening FIFA statistics page...")
    
    driver.get(FIFA_URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(5)

    tables = driver.find_elements(By.TAG_NAME, "table")

    print("Tables found:", len(tables))

    if tables:
        rows = tables[0].find_elements(By.TAG_NAME, "tr")

        print("Rows currently loaded:", len(rows))

        print("\nFirst 5 rows:")
        for row in rows[:5]:
            print(row.text)

finally:
    driver.quit()

Opening FIFA statistics page...
Tables found: 1
Rows currently loaded: 51

First 5 rows:
Rank
Player
Goals
Assists
Minutes Played
1
Kylian Mbappe
FRA
FW
10
4
769
2
Lionel Messi
ARG
FW
8
4
853
3
Jude Bellingham
ENG
MF
7
1
698
4
Erling Haaland
NOR
FW
7
0
537


## 25. Load Complete FIFA Player Statistics

The FIFA player-statistics table is successfully accessible through
Selenium.

The initial page contains 50 player records.

FIFA uses a "Load more" mechanism, so we will repeatedly click the
button until all available player records have been loaded.

We will then inspect the final number of players and the table columns.

No data will be merged into the local player statistics table yet.

In [58]:
# Load all player records from FIFA

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:
    driver.get(FIFA_URL)

    # Wait for the table
    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(3)

    previous_rows = 0
    clicks = 0

    while True:
        tables = driver.find_elements(By.TAG_NAME, "table")

        if not tables:
            print("Table disappeared.")
            break

        rows = tables[0].find_elements(By.TAG_NAME, "tr")
        current_rows = len(rows)

        print(f"Loaded rows: {current_rows - 1}")

        # Look for a Load More button
        buttons = driver.find_elements(
            By.XPATH,
            "//button[contains(translate(., "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
            "'abcdefghijklmnopqrstuvwxyz'), 'load more')]"
        )

        if not buttons:
            print("No 'Load more' button found.")
            break

        button = buttons[0]

        if not button.is_displayed() or not button.is_enabled():
            print("'Load more' button is no longer available.")
            break

        # Scroll button into view
        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )

        time.sleep(1)

        # Click
        driver.execute_script(
            "arguments[0].click();",
            button
        )

        clicks += 1

        # Wait for additional rows
        try:
            WebDriverWait(driver, 10).until(
                lambda d: len(
                    d.find_elements(By.TAG_NAME, "table")[0]
                    .find_elements(By.TAG_NAME, "tr")
                ) > current_rows
            )
        except:
            print("No additional rows loaded.")
            break

        time.sleep(1)

        # Safety check
        if current_rows == previous_rows:
            break

        previous_rows = current_rows

    # Final table
    table = driver.find_element(By.TAG_NAME, "table")

    final_rows = table.find_elements(By.TAG_NAME, "tr")

    print("\n==============================")
    print("FINAL FIFA TABLE")
    print("==============================")
    print("Player records loaded:", len(final_rows) - 1)
    print("Load more clicks:", clicks)

    print("\nFirst 5 rows:")
    for row in final_rows[:6]:
        print(row.text)

finally:
    driver.quit()

Loaded rows: 50
Loaded rows: 100
Loaded rows: 150
Loaded rows: 200
Loaded rows: 250
Loaded rows: 300
Loaded rows: 350
Loaded rows: 400
Loaded rows: 450
Loaded rows: 500
Loaded rows: 550
Loaded rows: 600
Loaded rows: 650
Loaded rows: 700
Loaded rows: 750
Loaded rows: 800
Loaded rows: 850
Loaded rows: 900
Loaded rows: 950
Loaded rows: 1000
Loaded rows: 1050
Loaded rows: 1100
Loaded rows: 1150
Loaded rows: 1200
Loaded rows: 1231
No 'Load more' button found.

FINAL FIFA TABLE
Player records loaded: 1231
Load more clicks: 24

First 5 rows:
Rank
Player
Goals
Assists
Minutes Played
1
Kylian Mbappe
FRA
FW
10
4
769
2
Lionel Messi
ARG
FW
8
4
853
3
Jude Bellingham
ENG
MF
7
1
698
4
Erling Haaland
NOR
FW
7
0
537
5
Ousmane Dembele
FRA
FW
6
2
648


## 26. Convert FIFA Player Statistics to DataFrame

The complete FIFA player-statistics table has been loaded successfully.

There are 1,231 player records available.

We will now convert the rendered FIFA HTML table into a pandas DataFrame
so that the statistics can be cleaned, inspected, and eventually matched
with our local `Players.csv`.

The FIFA data will remain separate from `player_stats_df` at this stage.

In [59]:
# Extract the complete FIFA table into pandas

from io import StringIO

# Reopen FIFA and load all players
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:
    driver.get(FIFA_URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(3)

    previous_rows = 0

    while True:
        table = driver.find_element(By.TAG_NAME, "table")
        current_rows = len(table.find_elements(By.TAG_NAME, "tr"))

        buttons = driver.find_elements(
            By.XPATH,
            "//button[contains(translate(., "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
            "'abcdefghijklmnopqrstuvwxyz'), 'load more')]"
        )

        if not buttons:
            break

        button = buttons[0]

        if not button.is_displayed() or not button.is_enabled():
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )

        time.sleep(0.5)

        driver.execute_script(
            "arguments[0].click();",
            button
        )

        try:
            WebDriverWait(driver, 10).until(
                lambda d: len(
                    d.find_element(By.TAG_NAME, "table")
                    .find_elements(By.TAG_NAME, "tr")
                ) > current_rows
            )
        except:
            break

        time.sleep(0.5)

        if current_rows == previous_rows:
            break

        previous_rows = current_rows

    # Get the final rendered table HTML
    table_html = driver.find_element(By.TAG_NAME, "table").get_attribute("outerHTML")

finally:
    driver.quit()


# Convert HTML table to pandas
fifa_player_stats_df = pd.read_html(
    StringIO(table_html)
)[0]

print("FIFA player statistics shape:", fifa_player_stats_df.shape)

print("\nFIFA columns:")
print(fifa_player_stats_df.columns.tolist())

print("\nData types:")
print(fifa_player_stats_df.dtypes)

print("\nFirst 10 players:")
display(fifa_player_stats_df.head(10))

FIFA player statistics shape: (1231, 5)

FIFA columns:
['Rank', 'Player', 'Goals', 'Assists', 'Minutes Played']

Data types:
Rank               int64
Player            object
Goals              int64
Assists            int64
Minutes Played     int64
dtype: object

First 10 players:


,Rank,Player,Goals,Assists,Minutes Played
0,1,Kylian MbappeFRAFRAFW,10,4,769
1,2,Lionel MessiARGARGFW,8,4,853
2,3,Jude BellinghamENGENGMF,7,1,698
3,4,Erling HaalandNORNORFW,7,0,537
4,5,Ousmane DembeleFRAFRAFW,6,2,648
5,6,Harry KaneENGENGFW,6,1,732
6,7,Mikel OyarzabalESPESPFW,5,1,664
7,8,Ismaila SarrSENSENFW,4,1,419
8,9,Julian QuinonesMEXMEXFW,4,1,440
9,10,Vinicius JuniorBRABRAFW,4,1,505


## 27. Scrape All FIFA Player Statistics Categories

The FIFA player-statistics page contains multiple statistical categories.

We will scrape all available categories from the official FIFA page:

1. Adidas Golden Boot
2. Attacking
3. Distribution
4. Defending
5. Discipline
6. Goalkeeping

Each category may contain different statistics and therefore will initially
be stored as a separate DataFrame.

We will also use the FIFA "Load more" functionality to retrieve the complete
player list for each category.

No statistics will be merged yet.

In [60]:
# Scrape all FIFA player-statistics categories

import time
import pandas as pd
from io import StringIO

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


FIFA_URL = (
    "https://www.fifa.com/en/tournaments/mens/worldcup/"
    "canadamexicousa2026/statistics/player-statistics"
)

# Statistics categories visible on the FIFA page
categories = [
    "adidas Golden Boot",
    "Attacking",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping"
]

# Storage for the scraped DataFrames
fifa_stats = {}


def load_all_rows(driver):
    """
    Keep clicking 'Load more' until all rows in the current
    statistics category have been loaded.
    """

    previous_rows = 0
    clicks = 0

    while True:

        tables = driver.find_elements(By.TAG_NAME, "table")

        if not tables:
            break

        table = tables[0]
        current_rows = len(
            table.find_elements(By.TAG_NAME, "tr")
        )

        # Find Load More buttons
        buttons = driver.find_elements(
            By.XPATH,
            "//button[contains("
            "translate(., "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
            "'abcdefghijklmnopqrstuvwxyz'), "
            "'load more')]"
        )

        if not buttons:
            break

        button = buttons[0]

        if not button.is_displayed() or not button.is_enabled():
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )

        time.sleep(0.5)

        driver.execute_script(
            "arguments[0].click();",
            button
        )

        clicks += 1

        try:
            WebDriverWait(driver, 10).until(
                lambda d: len(
                    d.find_element(By.TAG_NAME, "table")
                    .find_elements(By.TAG_NAME, "tr")
                ) > current_rows
            )
        except:
            break

        time.sleep(0.5)

        if current_rows == previous_rows:
            break

        previous_rows = current_rows

    return clicks


# Start browser
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:

    driver.get(FIFA_URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(3)

    for category in categories:

        print("\n" + "=" * 70)
        print("CATEGORY:", category)
        print("=" * 70)

        # Find the category button/tab
        category_elements = driver.find_elements(
            By.XPATH,
            "//*[normalize-space(text())="
            f"'{category}']"
        )

        if not category_elements:
            print("Category not found.")
            continue

        category_button = category_elements[0]

        # Scroll category into view
        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            category_button
        )

        time.sleep(1)

        # Click category
        driver.execute_script(
            "arguments[0].click();",
            category_button
        )

        time.sleep(2)

        # Load all rows
        load_more_clicks = load_all_rows(driver)

        # Get current table
        table = driver.find_element(By.TAG_NAME, "table")

        table_html = table.get_attribute("outerHTML")

        # Convert to pandas
        tables = pd.read_html(
            StringIO(table_html)
        )

        if tables:

            df = tables[0]

            fifa_stats[category] = df

            print("Rows:", len(df))
            print("Columns:")
            print(df.columns.tolist())
            print("Load more clicks:", load_more_clicks)

        else:
            print("No table extracted.")

finally:
    driver.quit()


print("\n\n" + "=" * 70)
print("SCRAPING SUMMARY")
print("=" * 70)

for category, df in fifa_stats.items():

    print(
        f"{category}: "
        f"{len(df)} rows × {len(df.columns)} columns"
    )


CATEGORY: adidas Golden Boot
Rows: 1231
Columns:
['Rank', 'Player', 'Goals', 'Assists', 'Minutes Played']
Load more clicks: 24

CATEGORY: Attacking
Rows: 2404
Columns:
['Rank', 'Player', 'Assists', 'Attempts On Target', 'Attempts At Goal', 'Attempts At Goal Conv. Rate (%)', 'Attempts Inside the Penalty Area', 'Attempts Outside the Penalty Area', 'Headed Attempts at Goal', 'xG', 'xG Efficiency', 'Corners']
Load more clicks: 48

CATEGORY: Distribution
Rows: 1242
Columns:
['Rank', 'Player', 'Passes', 'Passing Accuracy (%)', 'Crosses', 'Crossing Accuracy (%)', 'Defensive Linebreaks Attempted', 'Defensive Linebreaks Acc (%)', 'Switches of Play Attempted', 'Switches of Play Acc (%)']
Load more clicks: 24

CATEGORY: Defending
Rows: 1242
Columns:
['Rank', 'Player', 'Own Goals', 'Forced Turnovers', 'Defensive Pressures Applied', 'Defensive Pressures Directly Applied']
Load more clicks: 24

CATEGORY: Discipline
Rows: 1242
Columns:
['Rank', 'Player', 'Fouls Against', 'Fouls For', 'Yellow Cards',

## 28. Validate Scraped FIFA Statistics

All six FIFA player-statistics categories were successfully retrieved.

Before combining them, we need to validate each table.

For every category we will check:

- Number of rows
- Number of unique players
- Duplicate player records
- Missing player names
- Column names

This is especially important because the Attacking table contains more
rows than the other categories.

We will not merge or modify the scraped data until this validation is
complete.

In [61]:
# Validate every scraped FIFA statistics table

for category, df in fifa_stats.items():

    print("\n" + "=" * 70)
    print(category)
    print("=" * 70)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("Unique players:", df["Player"].nunique())

    print(
        "Duplicate player rows:",
        df["Player"].duplicated().sum()
    )

    print(
        "Missing player names:",
        df["Player"].isna().sum()
    )

    print("\nColumns:")
    for column in df.columns:
        print("-", column)

    print("\nFirst 3 rows:")
    display(df.head(3))


adidas Golden Boot
Rows: 1231
Columns: 5
Unique players: 1231
Duplicate player rows: 0
Missing player names: 0

Columns:
- Rank
- Player
- Goals
- Assists
- Minutes Played

First 3 rows:


,Rank,Player,Goals,Assists,Minutes Played
0,1,Kylian MbappeFRAFRAFW,10,4,769
1,2,Lionel MessiARGARGFW,8,4,853
2,3,Jude BellinghamENGENGMF,7,1,698



Attacking
Rows: 2404
Columns: 12
Unique players: 2404
Duplicate player rows: 0
Missing player names: 0

Columns:
- Rank
- Player
- Assists
- Attempts On Target
- Attempts At Goal
- Attempts At Goal Conv. Rate (%)
- Attempts Inside the Penalty Area
- Attempts Outside the Penalty Area
- Headed Attempts at Goal
- xG
- xG Efficiency
- Corners

First 3 rows:


,Rank,Player,Assists,Attempts On Target,Attempts At Goal,Attempts At Goal Conv. Rate (%),Attempts Inside the Penalty Area,Attempts Outside the Penalty Area,Headed Attempts at Goal,xG,xG Efficiency,Corners
0,1,Michael OliseFRAFRAFW,7,5,20,0,10,10,0,2.34,0x,25
1,2,Martin OdegaardNORNORMF,4,3,9,0,5,4,0,0.72,0x,12
2,2,Kylian MbappeFRAFRAFW,4,23,41,24,27,14,0,6.54,1.53x,5



Distribution
Rows: 1242
Columns: 10
Unique players: 1242
Duplicate player rows: 0
Missing player names: 0

Columns:
- Rank
- Player
- Passes
- Passing Accuracy (%)
- Crosses
- Crossing Accuracy (%)
- Defensive Linebreaks Attempted
- Defensive Linebreaks Acc (%)
- Switches of Play Attempted
- Switches of Play Acc (%)

First 3 rows:


...

,Rank,Player,Own Goals,Forced Turnovers,Defensive Pressures Applied,Defensive Pressures Directly Applied
0,1,Mohamed HanyEGYEGYDF,2,10,90,11
1,2,Yassine BounouMARMARGK,1,0,0,0
2,2,Ellyes SkhiriTUNTUNMF,1,10,40,10



Discipline
Rows: 1242
Columns: 8
Unique players: 1242
Duplicate player rows: 0
Missing player names: 0

Columns:
- Rank
- Player
- Fouls Against
- Fouls For
- Yellow Cards
- Red Cards
- Indirect Red Cards
- Offsides

First 3 rows:


,Rank,Player,Fouls Against,Fouls For,Yellow Cards,Red Cards,Indirect Red Cards,Offsides
0,1,Neil El AynaouiMARMARMF,15,7,0,0,0,0
1,2,Andres CubasPARPARMF,14,3,1,0,0,0
2,2,Lamine YamalESPESPFW,14,9,0,0,0,7



Goalkeeping
Rows: 145
Columns: 5
Unique players: 145
Duplicate player rows: 0
Missing player names: 0

Columns:
- Rank
- Player
- Goalkeeper Saves
- Goalkeeper Actions Inside the Penalty Area
- Goalkeeper Actions Outside the Penalty Area

First 3 rows:


,Rank,Player,Goalkeeper Saves,Goalkeeper Actions Inside the Penalty Area,Goalkeeper Actions Outside the Penalty Area
0,1,Orlando GillPARPARGK,28,141,124
1,2,Eloy RoomCUWCUWGK,21,81,72
2,3,Emiliano MartinezARGARGGK,20,111,97


## 28. Inspect Raw FIFA Table Structure

The FIFA tables were successfully scraped, but pandas combines multiple
visual fields inside the `Player` column.

For example, a FIFA row displayed as:

    Kylian Mbappe
    FRA
    FW

may be returned by pandas as:

    Kylian MbappeFRAFRAFW

Therefore, we will inspect the raw HTML table cells directly.

This will allow us to determine:

- The actual number of cells in each row.
- The separate Player, Nation, and Position values.
- Whether some categories contain multiple rows per player.
- Why the Attacking category contains 2,404 rows.

We will use the raw FIFA HTML structure for the final scraper rather than
relying only on `pd.read_html()`.

In [62]:
# Inspect the raw HTML cells for each FIFA category

# We need a fresh browser session
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:
    driver.get(FIFA_URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(3)

    # Select Attacking for investigation
    attacking_elements = driver.find_elements(
        By.XPATH,
        "//*[normalize-space(text())='Attacking']"
    )

    if attacking_elements:
        driver.execute_script(
            "arguments[0].click();",
            attacking_elements[0]
        )

        time.sleep(2)

        # Load some/all rows
        previous_rows = 0

        while True:
            table = driver.find_element(By.TAG_NAME, "table")

            current_rows = len(
                table.find_elements(By.TAG_NAME, "tr")
            )

            buttons = driver.find_elements(
                By.XPATH,
                "//button[contains("
                "translate(., "
                "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
                "'abcdefghijklmnopqrstuvwxyz'), "
                "'load more')]"
            )

            if not buttons:
                break

            button = buttons[0]

            if not button.is_displayed() or not button.is_enabled():
                break

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                button
            )

            time.sleep(0.5)

            driver.execute_script(
                "arguments[0].click();",
                button
            )

            try:
                WebDriverWait(driver, 10).until(
                    lambda d: len(
                        d.find_element(By.TAG_NAME, "table")
                        .find_elements(By.TAG_NAME, "tr")
                    ) > current_rows
                )
            except:
                break

            time.sleep(0.5)

            if current_rows == previous_rows:
                break

            previous_rows = current_rows

        table = driver.find_element(By.TAG_NAME, "table")

        rows = table.find_elements(By.TAG_NAME, "tr")

        print("Total raw rows:", len(rows))

        # Inspect header
        print("\nHEADER CELLS:")
        header_cells = rows[0].find_elements(
            By.XPATH,
            "./th|./td"
        )

        for i, cell in enumerate(header_cells):
            print(i, repr(cell.text))

        # Inspect first 10 data rows
        print("\nFIRST 10 DATA ROWS:")

        for row_number, row in enumerate(rows[1:11], start=1):

            cells = row.find_elements(
                By.XPATH,
                "./th|./td"
            )

            print(f"\nROW {row_number}")
            print("Number of cells:", len(cells))

            for i, cell in enumerate(cells):
                print(
                    f"  Cell {i}: {repr(cell.text)}"
                )

    else:
        print("Attacking category not found.")

finally:
    driver.quit()

Total raw rows: 2405

HEADER CELLS:
0 'Rank'
1 'Player'
2 'Assists'
3 'Attempts On Target'
4 'Attempts At Goal'
5 'Attempts At Goal Conv. Rate (%)'
6 'Attempts Inside the Penalty Area'
7 'Attempts Outside the Penalty Area'
8 'Headed Attempts at Goal'
9 'xG'
10 'xG Efficiency'
11 'Corners'

FIRST 10 DATA ROWS:

ROW 1
Number of cells: 12
  Cell 0: '1'
  Cell 1: 'Michael Olise\nFRA\nFW'
  Cell 2: '7'
  Cell 3: '5'
  Cell 4: '20'
  Cell 5: '0'
  Cell 6: '10'
  Cell 7: '10'
  Cell 8: '0'
  Cell 9: '2.34'
  Cell 10: '0x'
  Cell 11: '25'

ROW 2
Number of cells: 12
  Cell 0: '2'
  Cell 1: 'Martin Odegaard\nNOR\nMF'
  Cell 2: '4'
  Cell 3: '3'
  Cell 4: '9'
  Cell 5: '0'
  Cell 6: '5'
  Cell 7: '4'
  Cell 8: '0'
  Cell 9: '0.72'
  Cell 10: '0x'
  Cell 11: '12'

ROW 3
Number of cells: 12
  Cell 0: '2'
  Cell 1: 'Kylian Mbappe\nFRA\nFW'
  Cell 2: '4'
  Cell 3: '23'
  Cell 4: '41'
  Cell 5: '24'
  Cell 6: '27'
  Cell 7: '14'
  Cell 8: '0'
  Cell 9: '6.54'
  Cell 10: '1.53x'
  Cell 11: '5'

ROW 4
N

## 29. Investigate Attacking Player Records

The FIFA Attacking category contains 2,404 rows, which is much higher than
the approximately 1,242 records in the other player-statistics categories.

Before using this data, we need to determine why.

We will extract the player names from the raw FIFA table and check:

- Total records
- Unique players
- Players appearing more than once
- Number of records for repeated players

No data will be changed in this step.

In [64]:
# Check duplicate player records in FIFA Attacking

# Reuse the attacking table from the current FIFA session
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:
    driver.get(FIFA_URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(3)

    # Select Attacking
    attacking_elements = driver.find_elements(
        By.XPATH,
        "//*[normalize-space(text())='Attacking']"
    )

    driver.execute_script(
        "arguments[0].click();",
        attacking_elements[0]
    )

    time.sleep(2)

    # Load all rows
    previous_rows = 0

    while True:
        table = driver.find_element(By.TAG_NAME, "table")

        current_rows = len(
            table.find_elements(By.TAG_NAME, "tr")
        )

        buttons = driver.find_elements(
            By.XPATH,
            "//button[contains("
            "translate(., "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
            "'abcdefghijklmnopqrstuvwxyz'), "
            "'load more')]"
        )

        if not buttons:
            break

        button = buttons[0]

        if not button.is_displayed() or not button.is_enabled():
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )

        time.sleep(0.5)

        driver.execute_script(
            "arguments[0].click();",
            button
        )

        try:
            WebDriverWait(driver, 10).until(
                lambda d: len(
                    d.find_element(By.TAG_NAME, "table")
                    .find_elements(By.TAG_NAME, "tr")
                ) > current_rows
            )
        except:
            break

        time.sleep(0.5)

        if current_rows == previous_rows:
            break

        previous_rows = current_rows

    # Extract player names from raw cells
    table = driver.find_element(By.TAG_NAME, "table")
    rows = table.find_elements(By.TAG_NAME, "tr")

    attacking_players = []

    for row in rows[1:]:
        cells = row.find_elements(By.XPATH, "./th|./td")

        if len(cells) >= 2:
            player_text = cells[1].text.strip()

            # First line is the player name
            player_name = player_text.split("\n")[0].strip()

            attacking_players.append(player_name)

finally:
    driver.quit()


# Analyze names
attacking_names = pd.Series(
    attacking_players,
    name="Player"
)

name_counts = attacking_names.value_counts()

print("Total records:", len(attacking_names))
print("Unique players:", attacking_names.nunique())
print(
    "Players appearing more than once:",
    (name_counts > 1).sum()
)

print("\nMost repeated players:")
display(
    name_counts[name_counts > 1]
    .head(20)
    .to_frame("RecordCount")
)

print("\nPlayers with exactly 1 record:",
      (name_counts == 1).sum())

Total records: 2404
Unique players: 1931
Players appearing more than once: 472

Most repeated players:


,RecordCount
Player,
Jose Luis Rodriguez,3
Michael Woud,2
Victor Munoz,2
Joshua Kimmich,2
Michael Olise,2
Hussein Ali,2
Nathan Saliba,2
Julio Enciso,2
Hans Vanaken,2



Players with exactly 1 record: 1459


## 30. Inspect Duplicate Attacking Records

Some players appear multiple times in FIFA's Attacking statistics.

Before removing or combining these records, we will compare the
complete FIFA rows for duplicated players.

This will help determine whether the duplicates represent:
- different teams,
- different positions,
- different statistical records,
- or duplicated website rows.

No records will be deleted in this step.

In [65]:
# Inspect complete duplicate records from FIFA Attacking

# Re-open FIFA Attacking page
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

try:
    driver.get(FIFA_URL)

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    time.sleep(3)

    # Select Attacking
    attacking_elements = driver.find_elements(
        By.XPATH,
        "//*[normalize-space(text())='Attacking']"
    )

    driver.execute_script(
        "arguments[0].click();",
        attacking_elements[0]
    )

    time.sleep(2)

    # Load all rows
    while True:
        table = driver.find_element(By.TAG_NAME, "table")
        current_rows = len(table.find_elements(By.TAG_NAME, "tr"))

        buttons = driver.find_elements(
            By.XPATH,
            "//button[contains("
            "translate(., "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
            "'abcdefghijklmnopqrstuvwxyz'), "
            "'load more')]"
        )

        if not buttons:
            break

        button = buttons[0]

        if not button.is_displayed() or not button.is_enabled():
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )

        driver.execute_script(
            "arguments[0].click();",
            button
        )

        try:
            WebDriverWait(driver, 10).until(
                lambda d: len(
                    d.find_element(By.TAG_NAME, "table")
                    .find_elements(By.TAG_NAME, "tr")
                ) > current_rows
            )
        except:
            break

        time.sleep(0.5)

    # Extract complete rows
    table = driver.find_element(By.TAG_NAME, "table")
    rows = table.find_elements(By.TAG_NAME, "tr")

    headers = [
        cell.text.strip()
        for cell in rows[0].find_elements(By.XPATH, "./th|./td")
    ]

    attacking_records = []

    for row in rows[1:]:
        cells = row.find_elements(By.XPATH, "./th|./td")

        if len(cells) == len(headers):
            values = [
                cell.text.strip()
                for cell in cells
            ]

            record = dict(zip(headers, values))

            # Split FIFA's Player cell
            player_parts = record["Player"].split("\n")

            record["PlayerName"] = player_parts[0].strip()
            record["Nation"] = (
                player_parts[1].strip()
                if len(player_parts) > 1 else ""
            )
            record["Position"] = (
                player_parts[2].strip()
                if len(player_parts) > 2 else ""
            )

            attacking_records.append(record)

finally:
    driver.quit()


attacking_raw_df = pd.DataFrame(attacking_records)

# Find duplicated player names
duplicate_names = (
    attacking_raw_df["PlayerName"]
    .value_counts()
)

duplicate_names = duplicate_names[
    duplicate_names > 1
]

print("Total records:", len(attacking_raw_df))
print("Unique players:", attacking_raw_df["PlayerName"].nunique())
print("Duplicate player names:", len(duplicate_names))

# Select a few duplicated players for inspection
players_to_inspect = [
    "Michael Olise",
    "Joshua Kimmich",
    "Jose Luis Rodriguez"
]

duplicate_records = attacking_raw_df[
    attacking_raw_df["PlayerName"].isin(players_to_inspect)
].copy()

display(
    duplicate_records[
        [
            "Rank",
            "PlayerName",
            "Nation",
            "Position",
            "Assists",
            "Attempts On Target",
            "Attempts At Goal",
            "Attempts At Goal Conv. Rate (%)",
            "Attempts Inside the Penalty Area",
            "Attempts Outside the Penalty Area",
            "Headed Attempts at Goal",
            "xG",
            "xG Efficiency",
            "Corners"
        ]
    ]
)

Total records: 2404
Unique players: 1931
Duplicate player names: 472


,Rank,PlayerName,Nation,Position,Assists,Attempts On Target,Attempts At Goal,Attempts At Goal Conv. Rate (%),Attempts Inside the Penalty Area,Attempts Outside the Penalty Area,Headed Attempts at Goal,xG,xG Efficiency,Corners
0,1,Michael Olise,FRA,FW,7,5,20,0,10,10,0,2.34,0x,25
20,13,Joshua Kimmich,GER,DF,2,1,5,0,1,4,0,0.15,0x,15
363,167,Jose Luis Rodriguez,PAN,MF,0,1,3,0,1,2,0,0.24,0x,3
1133,1134,Jose Luis Rodriguez,mf,,-,-,-,-,-,-,-,-,-,-
1134,1135,Jose Luis Rodriguez,,,-,-,-,-,-,-,-,-,-,-
1148,1149,Joshua Kimmich,df,,-,-,-,-,-,-,-,-,-,-
1582,1583,Michael Olise,fw,,-,-,-,-,-,-,-,-,-,-


## 31. Clean FIFA Attacking Placeholder Rows

The FIFA Attacking table contains additional player rows where the
statistical fields contain only "-".

These are placeholder/interface rows and do not contain player statistics.

We will:

1. Identify rows with no statistical values.
2. Remove only those placeholder rows.
3. Keep valid player-statistics rows.
4. Validate the resulting player count and duplicates.

In [67]:
# Remove placeholder rows from FIFA Attacking statistics

attacking_clean_df = attacking_raw_df.copy()

# Statistical columns
attacking_stat_cols = [
    "Assists",
    "Attempts On Target",
    "Attempts At Goal",
    "Attempts At Goal Conv. Rate (%)",
    "Attempts Inside the Penalty Area",
    "Attempts Outside the Penalty Area",
    "Headed Attempts at Goal",
    "xG",
    "xG Efficiency",
    "Corners"
]

# Count rows where all statistical values are "-"
placeholder_mask = (
    attacking_clean_df[attacking_stat_cols]
    .apply(lambda row: all(str(x).strip() == "-" for x in row), axis=1)
)

print("Placeholder rows found:", placeholder_mask.sum())

# Remove placeholder rows
attacking_clean_df = attacking_clean_df[
    ~placeholder_mask
].copy()

# Reset index
attacking_clean_df.reset_index(drop=True, inplace=True)

print("Rows after cleaning:", len(attacking_clean_df))
print(
    "Unique players after cleaning:",
    attacking_clean_df["PlayerName"].nunique()
)

print(
    "Remaining duplicate player names:",
    attacking_clean_df["PlayerName"].duplicated().sum()
)

# Show any remaining duplicate players
remaining_duplicates = (
    attacking_clean_df[
        attacking_clean_df["PlayerName"].duplicated(keep=False)
    ]
    .sort_values("PlayerName")
)

print("\nRemaining duplicate player records:")

if len(remaining_duplicates) == 0:
    print("None")
else:
    display(
        remaining_duplicates[
            [
                "Rank",
                "PlayerName",
                "Nation",
                "Position",
                "Assists",
                "Attempts On Target",
                "Attempts At Goal",
                "xG",
                "Corners"
            ]
        ].head(30)
    )

Placeholder rows found: 1704
Rows after cleaning: 700
Unique players after cleaning: 699
Remaining duplicate player names: 1

Remaining duplicate player records:


,Rank,PlayerName,Nation,Position,Assists,Attempts On Target,Attempts At Goal,xG,Corners
173,167,Emiliano Martinez,ARG,GK,0,0,0,0,0
334,167,Emiliano Martinez,URU,MF,0,0,0,0,0


## 32. Diagnose FIFA Attacking Table Structure

The previous cleaning rule removed too many rows.

We will inspect the FIFA Attacking table structure to distinguish:

- valid player-statistic rows
- FIFA placeholder/interface rows
- players who legitimately share the same name

The dataset will not be modified in this step.

In [68]:
# Diagnose the FIFA Attacking table before applying another cleaning rule

print("Original raw records:", len(attacking_raw_df))

# Show distribution of Rank values
rank_numeric = pd.to_numeric(
    attacking_raw_df["Rank"],
    errors="coerce"
)

print("\nRank information")
print("Minimum rank:", rank_numeric.min())
print("Maximum rank:", rank_numeric.max())
print("Unique ranks:", rank_numeric.nunique())

# Count records with numeric rank
print(
    "Rows with numeric Rank:",
    rank_numeric.notna().sum()
)

# Inspect rows around the transition between valid and placeholder rows
print("\nSample of records around Rank 160-180:")

sample_160_180 = attacking_raw_df[
    rank_numeric.between(160, 180)
].copy()

display(
    sample_160_180[
        [
            "Rank",
            "PlayerName",
            "Nation",
            "Position",
            "Assists",
            "Attempts On Target",
            "Attempts At Goal",
            "xG",
            "Corners"
        ]
    ].head(30)
)

# Count how many rows have a real numeric value in each statistic
print("\nNon-placeholder value counts:")

for col in attacking_stat_cols:
    count = (
        attacking_raw_df[col]
        .astype(str)
        .str.strip()
        .ne("-")
        .sum()
    )

    print(f"{col}: {count}")

# Show records for the two Emiliano Martinez entries
print("\nEmiliano Martinez records:")

display(
    attacking_raw_df[
        attacking_raw_df["PlayerName"].eq("Emiliano Martinez")
    ][
        [
            "Rank",
            "PlayerName",
            "Nation",
            "Position",
            "Assists",
            "Attempts On Target",
            "Attempts At Goal",
            "xG",
            "Corners"
        ]
    ]
)

Original raw records: 2404

Rank information
Minimum rank: 1
Maximum rank: 2404
Unique ranks: 1710
Rows with numeric Rank: 2404

Sample of records around Rank 160-180:


,Rank,PlayerName,Nation,Position,Assists,Attempts On Target,Attempts At Goal,xG,Corners
166,167,Matias Vina,URU,DF,0,0,0,0,0
167,167,Ronald Araujo,URU,DF,0,0,0,0,0
168,167,Brenden Aaronson,USA,FW,0,1,3,0.46,0
169,167,Casemiro,BRA,MF,0,2,4,0.66,0
170,167,Kevin Castano,COL,MF,0,0,0,0,0
171,167,Exequiel Palacios,ARG,MF,0,0,0,0,0
172,167,Nicolas Tagliafico,ARG,DF,0,0,0,0,0
173,167,Emiliano Martinez,ARG,GK,0,0,0,0,0
174,167,Timothy Weah,USA,FW,0,0,2,0.05,0
175,167,Antonio Sanabria,PAR,FW,0,0,0,0,0



Non-placeholder value counts:
Assists: 700
Attempts On Target: 700
Attempts At Goal: 700
Attempts At Goal Conv. Rate (%): 700
Attempts Inside the Penalty Area: 700
Attempts Outside the Penalty Area: 700
Headed Attempts at Goal: 700
xG: 700
xG Efficiency: 700
Corners: 700

Emiliano Martinez records:


,Rank,PlayerName,Nation,Position,Assists,Attempts On Target,Attempts At Goal,xG,Corners
173,167,Emiliano Martinez,ARG,GK,0,0,0,0,0
334,167,Emiliano Martinez,URU,MF,0,0,0,0,0


## 33. Validate FIFA Attacking Records

The FIFA Attacking table contains 700 rows with actual statistics.

We will validate these records by checking:

- Unique player identities using PlayerName + Nation + Position
- Duplicate player identities
- Missing values
- Rank distribution
- Whether all statistical columns contain valid values

No records will be removed in this step.

In [69]:
# Build the valid Attacking dataset from rows containing statistics

attacking_clean_df = attacking_raw_df[
    ~placeholder_mask
].copy()

attacking_clean_df.reset_index(drop=True, inplace=True)

# Create a unique FIFA player identity
attacking_clean_df["FIFA_PlayerKey"] = (
    attacking_clean_df["PlayerName"].str.strip().str.upper()
    + "|"
    + attacking_clean_df["Nation"].str.strip().str.upper()
    + "|"
    + attacking_clean_df["Position"].str.strip().str.upper()
)

print("VALIDATION OF FIFA ATTACKING DATA")
print("=" * 60)

print("Total valid records:", len(attacking_clean_df))
print(
    "Unique Player + Nation + Position:",
    attacking_clean_df["FIFA_PlayerKey"].nunique()
)

print(
    "Duplicate FIFA player identities:",
    attacking_clean_df["FIFA_PlayerKey"].duplicated().sum()
)

print(
    "Missing PlayerName:",
    attacking_clean_df["PlayerName"].isna().sum()
)

print(
    "Missing Nation:",
    attacking_clean_df["Nation"].isna().sum()
)

print(
    "Missing Position:",
    attacking_clean_df["Position"].isna().sum()
)

print("\nRank statistics:")
print("Minimum Rank:", pd.to_numeric(
    attacking_clean_df["Rank"], errors="coerce"
).min())

print("Maximum Rank:", pd.to_numeric(
    attacking_clean_df["Rank"], errors="coerce"
).max())

print(
    "Unique Ranks:",
    attacking_clean_df["Rank"].nunique()
)

print("\nStatistical columns with '-' values:")

for col in attacking_stat_cols:
    dash_count = (
        attacking_clean_df[col]
        .astype(str)
        .str.strip()
        .eq("-")
        .sum()
    )

    print(f"{col}: {dash_count}")

print("\nFirst 10 valid records:")

display(
    attacking_clean_df[
        [
            "Rank",
            "PlayerName",
            "Nation",
            "Position",
            "Assists",
            "Attempts On Target",
            "Attempts At Goal",
            "xG",
            "xG Efficiency",
            "Corners"
        ]
    ].head(10)
)

VALIDATION OF FIFA ATTACKING DATA
Total valid records: 700
Unique Player + Nation + Position: 700
Duplicate FIFA player identities: 0
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0

Rank statistics:
Minimum Rank: 1
Maximum Rank: 167
Unique Ranks: 6

Statistical columns with '-' values:
Assists: 0
Attempts On Target: 0
Attempts At Goal: 0
Attempts At Goal Conv. Rate (%): 0
Attempts Inside the Penalty Area: 0
Attempts Outside the Penalty Area: 0
Headed Attempts at Goal: 0
xG: 0
xG Efficiency: 0
Corners: 0

First 10 valid records:


,Rank,PlayerName,Nation,Position,Assists,Attempts On Target,Attempts At Goal,xG,xG Efficiency,Corners
0,1,Michael Olise,FRA,FW,7,5,20,2.34,0x,25
1,2,Martin Odegaard,NOR,MF,4,3,9,0.72,0x,12
2,2,Kylian Mbappe,FRA,FW,4,23,41,6.54,1.53x,5
3,2,Brahim Diaz,MAR,FW,4,1,6,0.31,0x,1
4,2,Bruno Guimaraes,BRA,MF,4,2,8,1.02,0x,8
5,2,Lionel Messi,ARG,FW,4,19,35,6.18,1.29x,38
6,7,Roberto Alvarado,MEX,FW,3,0,5,0.28,0x,14
7,7,Anthony Gordon,ENG,FW,3,4,6,0.7,1.43x,0
8,7,Florian Wirtz,GER,MF,3,1,10,0.97,0x,5
9,7,Andreas Schjelderup,NOR,MF,3,2,3,0.2,5.12x,0


## 34. Validate FIFA Player Statistics Categories

The Attacking category has been successfully validated.

We will now inspect the remaining FIFA categories and determine:

- Total raw records
- Records containing actual statistics
- Placeholder records
- Unique player identities
- Duplicate player identities
- Missing player information

The data will not be merged with Players.csv yet.

In [70]:
# Validate the remaining FIFA categories
# using the already scraped fifa_stats data

categories_to_check = [
    "adidas Golden Boot",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping"
]

print("=" * 70)
print("FIFA CATEGORY VALIDATION")
print("=" * 70)

for category in categories_to_check:

    df = fifa_stats[category].copy()

    print(f"\n{'-' * 70}")
    print(category)
    print(f"{'-' * 70}")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("Columns:")
    print(list(df.columns))

    # Player column is still in pandas' flattened format
    # Display a few raw player values
    print("\nSample Player values:")

    display(
        df["Player"].head(3).to_frame()
    )

    print(
        "Unique Player values:",
        df["Player"].nunique()
    )

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

FIFA CATEGORY VALIDATION

----------------------------------------------------------------------
adidas Golden Boot
----------------------------------------------------------------------
Rows: 1231
Columns: 5
Columns:
['Rank', 'Player', 'Goals', 'Assists', 'Minutes Played']

Sample Player values:


,Player
0,Kylian MbappeFRAFRAFW
1,Lionel MessiARGARGFW
2,Jude BellinghamENGENGMF


Unique Player values: 1231

----------------------------------------------------------------------
Distribution
----------------------------------------------------------------------
Rows: 1242
Columns: 10
Columns:
['Rank', 'Player', 'Passes', 'Passing Accuracy (%)', 'Crosses', 'Crossing Accuracy (%)', 'Defensive Linebreaks Attempted', 'Defensive Linebreaks Acc (%)', 'Switches of Play Attempted', 'Switches of Play Acc (%)']

Sample Player values:


,Player
0,RodriESPESPMF
1,Pau CubarsiESPESPDF
2,Aymeric LaporteESPESPDF


Unique Player values: 1242

----------------------------------------------------------------------
Defending
----------------------------------------------------------------------
Rows: 1242
Columns: 6
Columns:
['Rank', 'Player', 'Own Goals', 'Forced Turnovers', 'Defensive Pressures Applied', 'Defensive Pressures Directly Applied']

Sample Player values:


...

Unique Player values: 1242

----------------------------------------------------------------------
Discipline
----------------------------------------------------------------------
Rows: 1242
Columns: 8
Columns:
['Rank', 'Player', 'Fouls Against', 'Fouls For', 'Yellow Cards', 'Red Cards', 'Indirect Red Cards', 'Offsides']

Sample Player values:


,Player
0,Neil El AynaouiMARMARMF
1,Andres CubasPARPARMF
2,Lamine YamalESPESPFW


Unique Player values: 1242

----------------------------------------------------------------------
Goalkeeping
----------------------------------------------------------------------
Rows: 145
Columns: 5
Columns:
['Rank', 'Player', 'Goalkeeper Saves', 'Goalkeeper Actions Inside the Penalty Area', 'Goalkeeper Actions Outside the Penalty Area']

Sample Player values:


,Player
0,Orlando GillPARPARGK
1,Eloy RoomCUWCUWGK
2,Emiliano MartinezARGARGGK


Unique Player values: 145

VALIDATION COMPLETE


## 35. Clean FIFA Player Statistics

We will create clean datasets for all FIFA player-statistics categories.

For every category, the FIFA Player field contains:

Player Name
Nation
Position

These values will be separated into individual columns.

The Attacking category will use the previously validated 700
statistics-bearing records.

The remaining categories will be cleaned from their scraped data.

No merge with the local Players.csv will be performed yet.

In [71]:
# ============================================================
# CLEAN FIFA PLAYER STATISTICS
# ============================================================

fifa_clean = {}

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def split_fifa_player(value):
    """
    Split FIFA's Player field into:
    PlayerName, Nation, Position
    """

    value = str(value).strip()

    # FIFA's pandas output flattened the DOM text.
    # We can recover the components because the final
    # nation and position codes are attached to the name.

    match = re.match(
        r"^(.*?)([A-Z]{3})([A-Z]{2})$",
        value
    )

    if match:
        player_name = match.group(1).strip()
        nation = match.group(2).strip()
        position = match.group(3).strip()

        return player_name, nation, position

    return value, "", ""


# ------------------------------------------------------------
# 1. Adidas Golden Boot
# ------------------------------------------------------------

golden_boot = fifa_stats["adidas Golden Boot"].copy()

golden_boot[
    ["PlayerName", "Nation", "Position"]
] = golden_boot["Player"].apply(
    lambda x: pd.Series(split_fifa_player(x))
)

fifa_clean["adidas Golden Boot"] = golden_boot


# ------------------------------------------------------------
# 2. Attacking
# ------------------------------------------------------------

attacking = attacking_clean_df.copy()

# PlayerName, Nation and Position already extracted
fifa_clean["Attacking"] = attacking


# ------------------------------------------------------------
# 3. Distribution
# ------------------------------------------------------------

distribution = fifa_stats["Distribution"].copy()

distribution[
    ["PlayerName", "Nation", "Position"]
] = distribution["Player"].apply(
    lambda x: pd.Series(split_fifa_player(x))
)

fifa_clean["Distribution"] = distribution


# ------------------------------------------------------------
# 4. Defending
# ------------------------------------------------------------

defending = fifa_stats["Defending"].copy()

defending[
    ["PlayerName", "Nation", "Position"]
] = defending["Player"].apply(
    lambda x: pd.Series(split_fifa_player(x))
)

fifa_clean["Defending"] = defending


# ------------------------------------------------------------
# 5. Discipline
# ------------------------------------------------------------

discipline = fifa_stats["Discipline"].copy()

discipline[
    ["PlayerName", "Nation", "Position"]
] = discipline["Player"].apply(
    lambda x: pd.Series(split_fifa_player(x))
)

fifa_clean["Discipline"] = discipline


# ------------------------------------------------------------
# 6. Goalkeeping
# ------------------------------------------------------------

goalkeeping = fifa_stats["Goalkeeping"].copy()

goalkeeping[
    ["PlayerName", "Nation", "Position"]
] = goalkeeping["Player"].apply(
    lambda x: pd.Series(split_fifa_player(x))
)

fifa_clean["Goalkeeping"] = goalkeeping


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("CLEAN FIFA DATASETS")
print("=" * 70)

for category, df in fifa_clean.items():

    print(f"\n{category}")
    print("-" * 70)

    print("Rows:", len(df))
    print("Unique PlayerName:", df["PlayerName"].nunique())
    print("Missing PlayerName:", df["PlayerName"].isna().sum())
    print("Missing Nation:", df["Nation"].eq("").sum())
    print("Missing Position:", df["Position"].eq("").sum())

    print("\nSample:")

    display(
        df[
            ["PlayerName", "Nation", "Position"]
        ].head(3)
    )

CLEAN FIFA DATASETS

adidas Golden Boot
----------------------------------------------------------------------
Rows: 1231
Unique PlayerName: 1231
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0

Sample:


,PlayerName,Nation,Position
0,Kylian MbappeFRA,FRA,FW
1,Lionel MessiARG,ARG,FW
2,Jude BellinghamENG,ENG,MF



Attacking
----------------------------------------------------------------------
Rows: 700
Unique PlayerName: 699
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0

Sample:


,PlayerName,Nation,Position
0,Michael Olise,FRA,FW
1,Martin Odegaard,NOR,MF
2,Kylian Mbappe,FRA,FW



Distribution
----------------------------------------------------------------------
Rows: 1242
Unique PlayerName: 1242
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0

Sample:


...

,PlayerName,Nation,Position
0,Mohamed HanyEGY,EGY,DF
1,Yassine BounouMAR,MAR,GK
2,Ellyes SkhiriTUN,TUN,MF



Discipline
----------------------------------------------------------------------
Rows: 1242
Unique PlayerName: 1242
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0

Sample:


,PlayerName,Nation,Position
0,Neil El AynaouiMAR,MAR,MF
1,Andres CubasPAR,PAR,MF
2,Lamine YamalESP,ESP,FW



Goalkeeping
----------------------------------------------------------------------
Rows: 145
Unique PlayerName: 145
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0

Sample:


,PlayerName,Nation,Position
0,Orlando GillPAR,PAR,GK
1,Eloy RoomCUW,CUW,GK
2,Emiliano MartinezARG,ARG,GK


## 36. Fix FIFA Player Name Parsing

The raw FIFA Player field contains:

Player Name
Nation
Position

Some categories were parsed incorrectly, leaving the Nation code
attached to PlayerName.

We will correct the parsing and validate that PlayerName contains
only the actual player name.

In [72]:
# ============================================================
# FIX FIFA PLAYER NAME PARSING
# ============================================================

def split_fifa_player_correct(value):
    """
    Split flattened FIFA Player text into:
    PlayerName, Nation, Position

    FIFA format:
    PlayerName + 3-letter Nation + 2-letter Position
    """

    value = str(value).strip()

    match = re.match(
        r"^(.*?)([A-Z]{3})(FW|MF|DF|GK)$",
        value
    )

    if match:
        return (
            match.group(1).strip(),
            match.group(2).strip(),
            match.group(3).strip()
        )

    return value, "", ""


# Rebuild the affected categories
categories_to_reparse = [
    "adidas Golden Boot",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping"
]

for category in categories_to_reparse:

    df = fifa_stats[category].copy()

    df[
        ["PlayerName", "Nation", "Position"]
    ] = df["Player"].apply(
        lambda x: pd.Series(
            split_fifa_player_correct(x)
        )
    )

    fifa_clean[category] = df


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("PLAYER NAME PARSING VALIDATION")
print("=" * 70)

for category, df in fifa_clean.items():

    print(f"\n{category}")
    print("-" * 70)

    print("Rows:", len(df))
    print("Unique PlayerName:", df["PlayerName"].nunique())
    print("Missing Nation:", df["Nation"].eq("").sum())
    print("Missing Position:", df["Position"].eq("").sum())

    display(
        df[
            ["PlayerName", "Nation", "Position"]
        ].head(3)
    )

PLAYER NAME PARSING VALIDATION

adidas Golden Boot
----------------------------------------------------------------------
Rows: 1231
Unique PlayerName: 1231
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Kylian MbappeFRA,FRA,FW
1,Lionel MessiARG,ARG,FW
2,Jude BellinghamENG,ENG,MF



Attacking
----------------------------------------------------------------------
Rows: 700
Unique PlayerName: 699
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Michael Olise,FRA,FW
1,Martin Odegaard,NOR,MF
2,Kylian Mbappe,FRA,FW



Distribution
----------------------------------------------------------------------
Rows: 1242
Unique PlayerName: 1242
Missing Nation: 0
Missing Position: 0


...

,PlayerName,Nation,Position
0,Mohamed HanyEGY,EGY,DF
1,Yassine BounouMAR,MAR,GK
2,Ellyes SkhiriTUN,TUN,MF



Discipline
----------------------------------------------------------------------
Rows: 1242
Unique PlayerName: 1242
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Neil El AynaouiMAR,MAR,MF
1,Andres CubasPAR,PAR,MF
2,Lamine YamalESP,ESP,FW



Goalkeeping
----------------------------------------------------------------------
Rows: 145
Unique PlayerName: 145
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Orlando GillPAR,PAR,GK
1,Eloy RoomCUW,CUW,GK
2,Emiliano MartinezARG,ARG,GK


## 37. Rebuild FIFA Categories from Raw DOM

The pandas HTML parser flattens FIFA's Player cell and can duplicate
the Nation code in the resulting text.

The raw Selenium DOM contains the correct structure:

Player Name
Nation
Position

We will therefore extract all six categories directly from the
raw DOM.

This will give us clean PlayerName, Nation, and Position fields
without relying on regex parsing of flattened text.

No player matching or merging will be performed yet.

In [75]:
# ============================================================
# RE-SCRAPE ALL FIFA CATEGORIES FROM RAW DOM
# ============================================================

CATEGORY_NAMES = [
    "adidas Golden Boot",
    "Attacking",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping"
]


def extract_player_from_dom(cell):
    """
    Extract PlayerName, Nation and Position from FIFA's
    raw Player table cell.
    """

    lines = [
        line.strip()
        for line in cell.text.splitlines()
        if line.strip()
    ]

    # Expected:
    # [Player Name, Nation, Position]
    if len(lines) >= 3:
        return lines[0], lines[1], lines[2]

    # Fallback
    return (
        lines[0] if len(lines) > 0 else "",
        lines[1] if len(lines) > 1 else "",
        lines[2] if len(lines) > 2 else ""
    )


def scrape_fifa_category(driver, category_name):

    # Click category
    elements = driver.find_elements(
        By.XPATH,
        f"//*[normalize-space(text())='{category_name}']"
    )

    if not elements:
        raise Exception(
            f"Category not found: {category_name}"
        )

    driver.execute_script(
        "arguments[0].click();",
        elements[0]
    )

    time.sleep(2)

    # Load all available rows
    previous_rows = 0

    while True:

        table = driver.find_element(
            By.TAG_NAME,
            "table"
        )

        current_rows = len(
            table.find_elements(
                By.TAG_NAME,
                "tr"
            )
        )

        buttons = driver.find_elements(
            By.XPATH,
            "//button[contains("
            "translate(., "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
            "'abcdefghijklmnopqrstuvwxyz'), "
            "'load more')]"
        )

        if not buttons:
            break

        button = buttons[0]

        if not button.is_displayed() or not button.is_enabled():
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )

        time.sleep(0.5)

        driver.execute_script(
            "arguments[0].click();",
            button
        )

        try:
            WebDriverWait(
                driver,
                10
            ).until(
                lambda d:
                len(
                    d.find_element(
                        By.TAG_NAME,
                        "table"
                    ).find_elements(
                        By.TAG_NAME,
                        "tr"
                    )
                ) > current_rows
            )
        except:
            break

        time.sleep(0.5)

        if current_rows == previous_rows:
            break

        previous_rows = current_rows

    # Extract table
    table = driver.find_element(
        By.TAG_NAME,
        "table"
    )

    rows = table.find_elements(
        By.TAG_NAME,
        "tr"
    )

    # Headers
    headers = [
        cell.text.strip()
        for cell in rows[0].find_elements(
            By.XPATH,
            "./th|./td"
        )
    ]

    records = []

    for row in rows[1:]:

        cells = row.find_elements(
            By.XPATH,
            "./th|./td"
        )

        if len(cells) != len(headers):
            continue

        values = [
            cell.text.strip()
            for cell in cells
        ]

        record = dict(
            zip(headers, values)
        )

        # Raw FIFA Player cell
        player_cell = cells[1]

        player_name, nation, position = (
            extract_player_from_dom(player_cell)
        )

        record["PlayerName"] = player_name
        record["Nation"] = nation
        record["Position"] = position

        records.append(record)

    return pd.DataFrame(records)


# ============================================================
# START BROWSER
# ============================================================

driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    ),
    options=chrome_options
)

try:

    driver.get(FIFA_URL)

    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (By.TAG_NAME, "table")
        )
    )

    time.sleep(3)

    fifa_raw_clean = {}

    for category in CATEGORY_NAMES:

        print(
            f"\nScraping: {category}"
        )

        df = scrape_fifa_category(
            driver,
            category
        )

        fifa_raw_clean[category] = df

        print(
            "Rows:",
            len(df)
        )

finally:

    driver.quit()


# ============================================================
# BASIC VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("RAW DOM FIFA DATA")
print("=" * 70)

for category, df in fifa_raw_clean.items():

    print(f"\n{category}")
    print("-" * 70)

    print("Rows:", len(df))
    print(
        "Unique Player + Nation + Position:",
        df[
            ["PlayerName", "Nation", "Position"]
        ].drop_duplicates().shape[0]
    )

    print(
        "Missing PlayerName:",
        df["PlayerName"].eq("").sum()
    )

    print(
        "Missing Nation:",
        df["Nation"].eq("").sum()
    )

    print(
        "Missing Position:",
        df["Position"].eq("").sum()
    )

    display(
        df[
            ["PlayerName", "Nation", "Position"]
        ].head(3)
    )


Scraping: adidas Golden Boot
Rows: 1231

Scraping: Attacking
Rows: 2404

Scraping: Distribution
Rows: 1242

Scraping: Defending
Rows: 1242

Scraping: Discipline
Rows: 1242

Scraping: Goalkeeping
Rows: 145


RAW DOM FIFA DATA

adidas Golden Boot
----------------------------------------------------------------------
Rows: 1231
Unique Player + Nation + Position: 1231
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Kylian Mbappe,FRA,FW
1,Lionel Messi,ARG,FW
2,Jude Bellingham,ENG,MF



Attacking
----------------------------------------------------------------------
Rows: 2404
Unique Player + Nation + Position: 2404
Missing PlayerName: 0
Missing Nation: 799
Missing Position: 1664


,PlayerName,Nation,Position
0,Michael Olise,FRA,FW
1,Martin Odegaard,NOR,MF
2,Kylian Mbappe,FRA,FW



Distribution
----------------------------------------------------------------------
Rows: 1242
Unique Player + Nation + Position: 1242
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0


...

,PlayerName,Nation,Position
0,Mohamed Hany,EGY,DF
1,Yassine Bounou,MAR,GK
2,Ellyes Skhiri,TUN,MF



Discipline
----------------------------------------------------------------------
Rows: 1242
Unique Player + Nation + Position: 1242
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Neil El Aynaoui,MAR,MF
1,Andres Cubas,PAR,MF
2,Lamine Yamal,ESP,FW



Goalkeeping
----------------------------------------------------------------------
Rows: 145
Unique Player + Nation + Position: 145
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0


,PlayerName,Nation,Position
0,Orlando Gill,PAR,GK
1,Eloy Room,CUW,GK
2,Emiliano Martinez,ARG,GK


## 38. Diagnose FIFA Attacking Row Structure

The Attacking category contains several different row structures.

Before cleaning it, we will inspect examples of:

- Complete Player / Nation / Position rows
- Rows missing Nation
- Rows missing Position
- Rows near the beginning and end of the table

This step is diagnostic only.
No records will be deleted or modified.

In [76]:
# ============================================================
# DIAGNOSE ATTACKING RAW DOM STRUCTURE
# ============================================================

attacking_df = fifa_raw_clean["Attacking"].copy()

print("=" * 70)
print("ATTACKING ROW STRUCTURE")
print("=" * 70)

print("Total rows:", len(attacking_df))

# Rows with missing Nation
missing_nation = attacking_df[
    attacking_df["Nation"].eq("")
]

# Rows with missing Position
missing_position = attacking_df[
    attacking_df["Position"].eq("")
]

print("\nMissing Nation:", len(missing_nation))
print("Missing Position:", len(missing_position))


# ------------------------------------------------------------
# Show first rows with missing Nation
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FIRST 20 ROWS WITH MISSING NATION")
print("-" * 70)

display(
    missing_nation[
        [
            "Rank",
            "PlayerName",
            "Nation",
            "Position"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# Show first rows with missing Position
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FIRST 20 ROWS WITH MISSING POSITION")
print("-" * 70)

display(
    missing_position[
        [
            "Rank",
            "PlayerName",
            "Nation",
            "Position"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# Inspect raw Player cell directly
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("RAW PLAYER CELL EXAMPLES")
print("-" * 70)

# Re-open FIFA page
driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    ),
    options=chrome_options
)

try:

    driver.get(FIFA_URL)

    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (By.TAG_NAME, "table")
        )
    )

    time.sleep(3)

    attacking_elements = driver.find_elements(
        By.XPATH,
        "//*[normalize-space(text())='Attacking']"
    )

    driver.execute_script(
        "arguments[0].click();",
        attacking_elements[0]
    )

    time.sleep(2)

    table = driver.find_element(
        By.TAG_NAME,
        "table"
    )

    rows = table.find_elements(
        By.TAG_NAME,
        "tr"
    )

    print("Initial table rows:", len(rows))

    # Inspect first 10 rows
    print("\nFIRST 10 DATA ROWS:")

    for i, row in enumerate(rows[1:11], start=1):

        cells = row.find_elements(
            By.XPATH,
            "./th|./td"
        )

        print(f"\nROW {i}")
        print("Number of cells:", len(cells))

        if len(cells) >= 2:
            print(
                "Player cell repr:",
                repr(cells[1].text)
            )

    # Inspect rows around 700
    print("\n" + "-" * 70)
    print("ROWS AROUND 700")
    print("-" * 70)

    for i in range(
        min(700, len(rows) - 1),
        min(710, len(rows) - 1)
    ):

        cells = rows[i].find_elements(
            By.XPATH,
            "./th|./td"
        )

        print(f"\nROW {i}")
        print(
            "Player cell:",
            repr(cells[1].text)
            if len(cells) >= 2
            else "NO PLAYER CELL"
        )

finally:

    driver.quit()

ATTACKING ROW STRUCTURE
Total rows: 2404

Missing Nation: 799
Missing Position: 1664

----------------------------------------------------------------------
FIRST 20 ROWS WITH MISSING NATION
----------------------------------------------------------------------


,Rank,PlayerName,Nation,Position
701,702,Facundo Torres,,
702,703,Fadel Haikal,,
703,704,Fafa Picault,,
705,706,Fahad Younis,,
708,709,Farid Chaal,,
712,713,Felix Horn Myhre,,
715,716,Fer Lopez,,
718,719,Ferjani Sassi,,
719,720,Fermin Lopez,,
724,725,Fikayo Tomori,,



----------------------------------------------------------------------
FIRST 20 ROWS WITH MISSING POSITION
----------------------------------------------------------------------


,Rank,PlayerName,Nation,Position
700,701,Facundo Pellistri,fw,
701,702,Facundo Torres,,
702,703,Fadel Haikal,,
703,704,Fafa Picault,,
704,705,Fahad Talib,gk,
705,706,Fahad Younis,,
706,707,Fares Chaibi,mf,
707,708,Fares Ghedjemis,fw,
708,709,Farid Chaal,,
709,710,Farrukh Sayfiev,df,



----------------------------------------------------------------------
RAW PLAYER CELL EXAMPLES
----------------------------------------------------------------------
Initial table rows: 51

FIRST 10 DATA ROWS:

ROW 1
Number of cells: 12
Player cell repr: 'Michael Olise\nFRA\nFW'

ROW 2
Number of cells: 12
Player cell repr: 'Martin Odegaard\nNOR\nMF'

ROW 3
Number of cells: 12
Player cell repr: 'Kylian Mbappe\nFRA\nFW'

ROW 4
Number of cells: 12
Player cell repr: 'Brahim Diaz\nMAR\nFW'

ROW 5
Number of cells: 12
Player cell repr: 'Bruno Guimaraes\nBRA\nMF'

ROW 6
Number of cells: 12
Player cell repr: 'Lionel Messi\nARG\nFW'

ROW 7
Number of cells: 12
Player cell repr: 'Roberto Alvarado\nMEX\nFW'

ROW 8
Number of cells: 12
Player cell repr: 'Anthony Gordon\nENG\nFW'

ROW 9
Number of cells: 12
Player cell repr: 'Florian Wirtz\nGER\nMF'

ROW 10
Number of cells: 12
Player cell repr: 'Andreas Schjelderup\nNOR\nMF'

----------------------------------------------------------------------
ROWS

## 39. Create Final Clean FIFA Statistics

The FIFA Attacking table contains 700 valid statistical records.
Additional rows loaded by the website do not contain complete
player metadata/statistics and will not be used.

We will now create the final clean datasets for all six categories.

These datasets will be preserved separately before the player matching
stage.

In [77]:
# ============================================================
# FINAL CLEAN FIFA DATASETS
# ============================================================

fifa_final = {}

# ------------------------------------------------------------
# Attacking
# ------------------------------------------------------------

fifa_final["Attacking"] = attacking_clean_df.copy()

# ------------------------------------------------------------
# Other FIFA categories
# ------------------------------------------------------------

for category in [
    "adidas Golden Boot",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping"
]:

    fifa_final[category] = (
        fifa_raw_clean[category]
        .copy()
    )


# ============================================================
# REMOVE INTERNAL / RAW PLAYER COLUMN
# ============================================================

for category, df in fifa_final.items():

    if "Player" in df.columns:
        df.drop(
            columns=["Player"],
            inplace=True
        )

    # Remove internal key if present
    if "FIFA_PlayerKey" in df.columns:
        df.drop(
            columns=["FIFA_PlayerKey"],
            inplace=True
        )

    fifa_final[category].reset_index(
        drop=True,
        inplace=True
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

print("=" * 70)
print("FINAL FIFA DATASETS")
print("=" * 70)

for category, df in fifa_final.items():

    print(f"\n{category}")
    print("-" * 70)

    print("Rows:", len(df))

    print(
        "Unique Player + Nation + Position:",
        df[
            ["PlayerName", "Nation", "Position"]
        ].drop_duplicates().shape[0]
    )

    print(
        "Missing PlayerName:",
        df["PlayerName"].eq("").sum()
    )

    print(
        "Missing Nation:",
        df["Nation"].eq("").sum()
    )

    print(
        "Missing Position:",
        df["Position"].eq("").sum()
    )

    print("Columns:")
    print(list(df.columns))

print("\n" + "=" * 70)
print("FINAL FIFA DATASETS READY FOR PLAYER MATCHING")
print("=" * 70)

FINAL FIFA DATASETS

Attacking
----------------------------------------------------------------------
Rows: 700
Unique Player + Nation + Position: 700
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0
Columns:
['Rank', 'Assists', 'Attempts On Target', 'Attempts At Goal', 'Attempts At Goal Conv. Rate (%)', 'Attempts Inside the Penalty Area', 'Attempts Outside the Penalty Area', 'Headed Attempts at Goal', 'xG', 'xG Efficiency', 'Corners', 'PlayerName', 'Nation', 'Position']

adidas Golden Boot
----------------------------------------------------------------------
Rows: 1231
Unique Player + Nation + Position: 1231
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0
Columns:
['Rank', 'Goals', 'Assists', 'Minutes Played', 'PlayerName', 'Nation', 'Position']

Distribution
----------------------------------------------------------------------
Rows: 1242
Unique Player + Nation + Position: 1242
Missing PlayerName: 0
Missing Nation: 0
Missing Position: 0
Columns:
['Rank', 'Pass

## 40. Match FIFA Players to Local PlayerID

The FIFA datasets are now clean.

We will match FIFA players against the local Players.csv using:

- normalized player name
- team/nation information where available
- position as an additional validation field

The first pass will use exact normalized matching only.

No fuzzy matching or manual corrections will be applied yet.

This allows us to measure exactly how many FIFA players can be
matched safely to our local PlayerID.

In [78]:
# ============================================================
# STEP 40 — PREPARE PLAYER MATCHING
# ============================================================

# Load local Players.csv if not already loaded
players_df = pd.read_csv(
    r"D:\ai course hw\FIFA_WorldCup_2026_Project\Data\Players.csv"
)

print("Local Players.csv:", players_df.shape)


# ------------------------------------------------------------
# Name normalization
# ------------------------------------------------------------

import unicodedata


def normalize_name(name):
    """
    Normalize player names for safe exact matching.
    """

    if pd.isna(name):
        return ""

    name = str(name).strip().upper()

    # Remove accents
    name = unicodedata.normalize(
        "NFKD",
        name
    )

    name = "".join(
        char
        for char in name
        if not unicodedata.combining(char)
    )

    # Normalize punctuation
    name = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        name
    )

    # Normalize whitespace
    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()

    return name


# Local player normalized name
players_df["NameNormalized"] = (
    players_df["PlayerName"]
    .apply(normalize_name)
)


# FIFA normalized names
for category, df in fifa_final.items():

    df["NameNormalized"] = (
        df["PlayerName"]
        .apply(normalize_name)
    )


# ------------------------------------------------------------
# Check local duplicate normalized names
# ------------------------------------------------------------

local_name_counts = (
    players_df["NameNormalized"]
    .value_counts()
)

local_duplicate_names = (
    local_name_counts[
        local_name_counts > 1
    ]
)

print("\nLOCAL PLAYER NAME VALIDATION")
print("=" * 60)

print(
    "Total local players:",
    len(players_df)
)

print(
    "Unique normalized names:",
    players_df["NameNormalized"].nunique()
)

print(
    "Normalized names appearing more than once:",
    len(local_duplicate_names)
)

print("\nExamples of duplicate local names:")

display(
    local_duplicate_names
    .head(20)
    .to_frame("Count")
)


# ------------------------------------------------------------
# Exact name matching test for each FIFA category
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIFA → LOCAL PLAYER NAME MATCHING")
print("=" * 70)

for category, df in fifa_final.items():

    matched = df["NameNormalized"].isin(
        players_df["NameNormalized"]
    )

    print(
        f"\n{category}"
    )

    print(
        "FIFA records:",
        len(df)
    )

    print(
        "Name matches:",
        matched.sum()
    )

    print(
        "Unmatched:",
        (~matched).sum()
    )

    print(
        "Match rate:",
        f"{matched.mean() * 100:.2f}%"
    )

Local Players.csv: (1248, 5)

LOCAL PLAYER NAME VALIDATION
Total local players: 1248
Unique normalized names: 1247
Normalized names appearing more than once: 1

Examples of duplicate local names:


,Count
NameNormalized,
EMILIANO MARTINEZ,2



FIFA → LOCAL PLAYER NAME MATCHING

Attacking
FIFA records: 700
Name matches: 698
Unmatched: 2
Match rate: 99.71%

adidas Golden Boot
FIFA records: 1231
Name matches: 1205
Unmatched: 26
Match rate: 97.89%

Distribution
FIFA records: 1242
Name matches: 1216
Unmatched: 26
Match rate: 97.91%

Defending
FIFA records: 1242
Name matches: 1216
Unmatched: 26
Match rate: 97.91%

Discipline
FIFA records: 1242
Name matches: 1216
Unmatched: 26
Match rate: 97.91%

Goalkeeping
FIFA records: 145
Name matches: 142
Unmatched: 3
Match rate: 97.93%


## 41. Investigate Unmatched FIFA Players

The first exact normalized-name match produced very high match rates,
but some FIFA players remain unmatched.

We will list every unmatched player from each FIFA category and compare
their FIFA name, Nation, and Position.

No fuzzy matching or manual changes will be made yet.

In [79]:
# ============================================================
# STEP 41 — LIST ALL UNMATCHED FIFA PLAYERS
# ============================================================

print("=" * 80)
print("UNMATCHED FIFA PLAYERS")
print("=" * 80)

unmatched_fifa = {}

local_names = set(
    players_df["NameNormalized"]
)

for category, df in fifa_final.items():

    unmatched = df[
        ~df["NameNormalized"].isin(local_names)
    ].copy()

    unmatched_fifa[category] = unmatched

    print(f"\n{'-' * 80}")
    print(category)
    print(f"{'-' * 80}")

    print("Unmatched:", len(unmatched))

    if len(unmatched) > 0:

        display(
            unmatched[
                [
                    "PlayerName",
                    "Nation",
                    "Position"
                ]
            ].to_string(index=False)
        )

UNMATCHED FIFA PLAYERS

--------------------------------------------------------------------------------
Attacking
--------------------------------------------------------------------------------
Unmatched: 2


'  PlayerName Nation Position\nHwang Inbeom    KOR       MF\n  Lee Kangin    KOR       MF'


--------------------------------------------------------------------------------
adidas Golden Boot
--------------------------------------------------------------------------------
Unmatched: 26


'    PlayerName Nation Position\n  Hwang Inbeom    KOR       MF\n   Oh Hyeongyu    KOR       FW\n    Lee Kangin    KOR       MF\n     Bae Junho    KOR       MF\n      Cho Wije    KOR       DF\n   Jo Hyeonwoo    KOR       GK\n  Kim Taehyeon    KOR       DF\nLee Donggyeong    KOR       MF\n  Song Bumkeun    KOR       GK\n  Yang Hyunjun    KOR       MF\n   Cho Guesung    KOR       FW\n  Park Jinseob    KOR       DF\n  Castrop Jens    KOR       DF\n    Eom Jisung    KOR       MF\n    Kim Jingyu    KOR       MF\n  Kim Moonhwan    KOR       DF\n   Lee Taeseok    KOR       DF\n Hwang Heechan    KOR       MF\n   Lee Jaesung    KOR       MF\n  Son Heungmin    KOR       FW\n  Paik Seungho    KOR       MF\n    Kim Minjae    KOR       DF\n Seol Youngwoo    KOR       DF\n  Kim Seunggyu    KOR       GK\n   Lee Hanbeom    KOR       DF\n    Lee Gihyuk    KOR       MF'


--------------------------------------------------------------------------------
Distribution
--------------------------------------------------------------------------------
Unmatched: 26


...

'    PlayerName Nation Position\n    Eom Jisung    KOR       MF\nLee Donggyeong    KOR       MF\n  Son Heungmin    KOR       FW\n  Kim Moonhwan    KOR       DF\n  Paik Seungho    KOR       MF\n   Jo Hyeonwoo    KOR       GK\n   Oh Hyeongyu    KOR       FW\n    Kim Minjae    KOR       DF\n    Kim Jingyu    KOR       MF\n  Hwang Inbeom    KOR       MF\n Hwang Heechan    KOR       MF\n   Cho Guesung    KOR       FW\n   Lee Jaesung    KOR       MF\n  Yang Hyunjun    KOR       MF\n  Park Jinseob    KOR       DF\n   Lee Hanbeom    KOR       DF\n   Lee Taeseok    KOR       DF\n    Lee Gihyuk    KOR       MF\n  Kim Seunggyu    KOR       GK\n    Lee Kangin    KOR       MF\n  Castrop Jens    KOR       DF\n  Kim Taehyeon    KOR       DF\n     Bae Junho    KOR       MF\n      Cho Wije    KOR       DF\n  Song Bumkeun    KOR       GK\n Seol Youngwoo    KOR       DF'


--------------------------------------------------------------------------------
Discipline
--------------------------------------------------------------------------------
Unmatched: 26


'    PlayerName Nation Position\n   Lee Hanbeom    KOR       DF\n  Castrop Jens    KOR       DF\n   Lee Taeseok    KOR       DF\n    Kim Minjae    KOR       DF\n    Lee Gihyuk    KOR       MF\n  Hwang Inbeom    KOR       MF\n Seol Youngwoo    KOR       DF\n Hwang Heechan    KOR       MF\n  Paik Seungho    KOR       MF\n   Lee Jaesung    KOR       MF\n  Yang Hyunjun    KOR       MF\n    Lee Kangin    KOR       MF\n   Cho Guesung    KOR       FW\n  Kim Taehyeon    KOR       DF\n  Son Heungmin    KOR       FW\nLee Donggyeong    KOR       MF\n      Cho Wije    KOR       DF\n   Oh Hyeongyu    KOR       FW\n  Park Jinseob    KOR       DF\n   Jo Hyeonwoo    KOR       GK\n  Kim Moonhwan    KOR       DF\n  Kim Seunggyu    KOR       GK\n  Song Bumkeun    KOR       GK\n     Bae Junho    KOR       MF\n    Kim Jingyu    KOR       MF\n    Eom Jisung    KOR       MF'


--------------------------------------------------------------------------------
Goalkeeping
--------------------------------------------------------------------------------
Unmatched: 3


'  PlayerName Nation Position\nKim Seunggyu    KOR       GK\nSong Bumkeun    KOR       GK\n Jo Hyeonwoo    KOR       GK'

## 42. Investigate Korean Player Name Mismatch

All unmatched FIFA players belong to South Korea (KOR).

We will compare FIFA's Korean player names with the Korean players
already present in the local Players.csv.

No names will be changed yet.

In [80]:
# ============================================================
# STEP 42 — COMPARE KOREAN PLAYER NAMES
# ============================================================

# Get all Korean players from FIFA unmatched records
korea_fifa_names = set()

for category, df in unmatched_fifa.items():

    for name in df["PlayerName"]:
        korea_fifa_names.add(name.strip())

korea_fifa_names = sorted(korea_fifa_names)

print("=" * 70)
print("FIFA KOREAN PLAYERS")
print("=" * 70)

print("Unique FIFA Korean players:", len(korea_fifa_names))

display(
    pd.DataFrame({
        "FIFA_PlayerName": korea_fifa_names
    })
)


# ------------------------------------------------------------
# Local Korean players
# ------------------------------------------------------------

# We need to identify Korea using the TeamID relationship.
# Show all local players whose names resemble the FIFA Korean names
# by checking common Korean-name components.

korean_keywords = [
    "Hwang",
    "Lee",
    "Kim",
    "Son",
    "Oh",
    "Bae",
    "Cho",
    "Jo",
    "Paik",
    "Park",
    "Seol",
    "Yang",
    "Eom"
]

local_korean = players_df[
    players_df["PlayerName"]
    .astype(str)
    .str.startswith(tuple(korean_keywords))
].copy()

print("\n" + "=" * 70)
print("LOCAL PLAYERS WITH KOREAN NAME PATTERNS")
print("=" * 70)

print("Rows found:", len(local_korean))

display(
    local_korean[
        [
            "PlayerID",
            "PlayerName",
            "TeamID",
            "Position",
            "JerseyNumber"
        ]
    ].sort_values("PlayerName")
)

FIFA KOREAN PLAYERS
Unique FIFA Korean players: 26


,FIFA_PlayerName
0,Bae Junho
1,Castrop Jens
2,Cho Guesung
3,Cho Wije
4,Eom Jisung
5,Hwang Heechan
6,Hwang Inbeom
7,Jo Hyeonwoo
8,Kim Jingyu
9,Kim Minjae



LOCAL PLAYERS WITH KOREAN NAME PATTERNS
Rows found: 54


,PlayerID,PlayerName,TeamID,Position,JerseyNumber
1052,1053,Joan Garcia,10,GK,13
903,904,Joao Cancelo,23,DF,20
894,895,Joao Felix,23,FW,11
898,899,Joao Neves,23,MF,15
189,190,Joao Paulo,15,MF,8
1191,1192,Joaquin Piquerez,36,MF,22
121,122,Joaquin Seys,2,DF,18
785,786,Joe Bell,42,MF,6
1218,1219,Joe Scally,12,DF,23
393,394,Joel Ordonez,28,DF,4


## 43. Identify South Korean Players Using TeamID

The previous name-based filter was too broad because many international
players have names beginning with Lee, Kim, Jo, Jose, etc.

We will use the official TeamID relationship from Teams.csv instead.

This gives us the exact South Korean players in our local Players.csv.

No player names or IDs will be changed.

In [81]:
# ============================================================
# STEP 43 — FIND SOUTH KOREAN PLAYERS USING TEAMID
# ============================================================

teams_df = pd.read_csv(
    r"D:\ai course hw\FIFA_WorldCup_2026_Project\Data\Teams.csv"
)

print("=" * 70)
print("TEAMS DATA")
print("=" * 70)

print("Shape:", teams_df.shape)
print("Columns:", list(teams_df.columns))

display(teams_df.head())


# ------------------------------------------------------------
# Find Korea-related team record
# ------------------------------------------------------------

korea_team = teams_df[
    teams_df.astype(str)
    .apply(
        lambda col: col.str.contains(
            "Korea",
            case=False,
            na=False
        )
    )
    .any(axis=1)
]

print("\n" + "=" * 70)
print("KOREA TEAM RECORD")
print("=" * 70)

display(korea_team)


# ------------------------------------------------------------
# Get Korean TeamID
# ------------------------------------------------------------

if len(korea_team) == 0:
    print("South Korea team was not found automatically.")
else:

    korea_team_id = korea_team.iloc[0]["TeamID"]

    print(
        "South Korea TeamID:",
        korea_team_id
    )

    # --------------------------------------------------------
    # Exact local Korean players
    # --------------------------------------------------------

    korean_players = players_df[
        players_df["TeamID"] == korea_team_id
    ].copy()

    print("\n" + "=" * 70)
    print("LOCAL SOUTH KOREAN PLAYERS")
    print("=" * 70)

    print(
        "Number of Korean players:",
        len(korean_players)
    )

    display(
        korean_players[
            [
                "PlayerID",
                "PlayerName",
                "TeamID",
                "Position",
                "JerseyNumber"
            ]
        ].sort_values("PlayerName")
    )


    # --------------------------------------------------------
    # Compare FIFA and local names
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("FIFA → LOCAL KOREA NAME COMPARISON")
    print("=" * 70)

    korea_local_names = set(
        korean_players["NameNormalized"]
    )

    comparison = pd.DataFrame({
        "FIFA_PlayerName": korea_fifa_names
    })

    comparison["FIFA_NameNormalized"] = (
        comparison["FIFA_PlayerName"]
        .apply(normalize_name)
    )

    comparison["ExactLocalNameMatch"] = (
        comparison["FIFA_NameNormalized"]
        .isin(korea_local_names)
    )

    display(comparison)

TEAMS DATA
Shape: (48, 6)
Columns: ['TeamID', 'TeamName', 'CountryCode', 'Confederation', 'GroupName', 'HostCountry']


,TeamID,TeamName,CountryCode,Confederation,GroupName,HostCountry
0,1,Argentina,ARG,CONMEBOL,Group J,NaN
1,2,Belgium,BEL,UEFA,Group G,NaN
2,3,Brazil,BRA,CONMEBOL,Group C,NaN
3,4,Colombia,COL,CONMEBOL,Group K,NaN
4,5,England,ENG,UEFA,Group L,NaN



KOREA TEAM RECORD


,TeamID,TeamName,CountryCode,Confederation,GroupName,HostCountry
30,31,Korea Republic,KOR,AFC,Group A,NaN


South Korea TeamID: 31

LOCAL SOUTH KOREAN PLAYERS
Number of Korean players: 26


,PlayerID,PlayerName,TeamID,Position,JerseyNumber
687,688,Bumkeun Song,31,GK,12
701,702,Donggyeong Lee,31,MF,26
678,679,Gihyuk Lee,31,MF,3
684,685,Guesung Cho,31,FW,9
677,678,Hanbeom Lee,31,DF,2
686,687,Heechan Hwang,31,MF,11
682,683,Heungmin Son,31,FW,7
693,694,Hyeongyu Oh,31,FW,18
696,697,Hyeonwoo Jo,31,GK,21
695,696,Hyunjun Yang,31,MF,20



FIFA → LOCAL KOREA NAME COMPARISON


,FIFA_PlayerName,FIFA_NameNormalized,ExactLocalNameMatch
0,Bae Junho,BAE JUNHO,False
1,Castrop Jens,CASTROP JENS,False
2,Cho Guesung,CHO GUESUNG,False
3,Cho Wije,CHO WIJE,False
4,Eom Jisung,EOM JISUNG,False
5,Hwang Heechan,HWANG HEECHAN,False
6,Hwang Inbeom,HWANG INBEOM,False
7,Jo Hyeonwoo,JO HYEONWOO,False
8,Kim Jingyu,KIM JINGYU,False
9,Kim Minjae,KIM MINJAE,False


## 44. Build Korean FIFA → PlayerID Mapping

The FIFA Korean player names use:

    Surname GivenName

while Players.csv uses:

    GivenName Surname

Because the Korean FIFA records have Nation = KOR and the local
South Korea team has TeamID = 31, we can safely reverse the
two-part Korean names and match them to the local players.

We will validate that every FIFA Korean player maps to exactly
one local PlayerID.

No FIFA statistics will be merged yet.

In [82]:
# ============================================================
# STEP 44 — KOREAN PLAYER MAPPING
# ============================================================

korean_fifa = pd.DataFrame({
    "FIFA_PlayerName": korea_fifa_names
})

# Normalize FIFA names
korean_fifa["FIFA_NameNormalized"] = (
    korean_fifa["FIFA_PlayerName"]
    .apply(normalize_name)
)


# ------------------------------------------------------------
# Reverse FIFA Korean names
# ------------------------------------------------------------

def reverse_two_part_name(name):
    parts = str(name).strip().split()

    if len(parts) == 2:
        return f"{parts[1]} {parts[0]}"

    return name


korean_fifa["ReversedName"] = (
    korean_fifa["FIFA_PlayerName"]
    .apply(reverse_two_part_name)
)

korean_fifa["ReversedNameNormalized"] = (
    korean_fifa["ReversedName"]
    .apply(normalize_name)
)


# ------------------------------------------------------------
# Local Korean players
# ------------------------------------------------------------

korean_local = players_df[
    players_df["TeamID"] == korea_team_id
].copy()

korean_local["LocalNameNormalized"] = (
    korean_local["PlayerName"]
    .apply(normalize_name)
)


# ------------------------------------------------------------
# Match reversed FIFA name
# ------------------------------------------------------------

korean_mapping = korean_fifa.merge(
    korean_local[
        [
            "PlayerID",
            "PlayerName",
            "TeamID",
            "Position",
            "JerseyNumber",
            "LocalNameNormalized"
        ]
    ],
    left_on="ReversedNameNormalized",
    right_on="LocalNameNormalized",
    how="left"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("KOREAN PLAYER MAPPING")
print("=" * 70)

print(
    "FIFA Korean players:",
    len(korean_mapping)
)

print(
    "Successfully matched:",
    korean_mapping["PlayerID"].notna().sum()
)

print(
    "Unmatched:",
    korean_mapping["PlayerID"].isna().sum()
)

print(
    "Unique matched PlayerIDs:",
    korean_mapping["PlayerID"].dropna().nunique()
)

print(
    "Duplicate PlayerIDs:",
    korean_mapping["PlayerID"]
    .dropna()
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# Show mapping
# ------------------------------------------------------------

display(
    korean_mapping[
        [
            "FIFA_PlayerName",
            "ReversedName",
            "PlayerID",
            "PlayerName",
            "TeamID",
            "Position",
            "JerseyNumber"
        ]
    ].sort_values("FIFA_PlayerName")
)

KOREAN PLAYER MAPPING
FIFA Korean players: 26
Successfully matched: 26
Unmatched: 0
Unique matched PlayerIDs: 26
Duplicate PlayerIDs: 0


,FIFA_PlayerName,ReversedName,PlayerID,PlayerName,TeamID,Position,JerseyNumber
0,Bae Junho,Junho Bae,693,Junho Bae,31,MF,17
1,Castrop Jens,Jens Castrop,699,Jens Castrop,31,DF,23
2,Cho Guesung,Guesung Cho,685,Guesung Cho,31,FW,9
3,Cho Wije,Wije Cho,690,Wije Cho,31,DF,14
4,Eom Jisung,Jisung Eom,701,Jisung Eom,31,MF,25
5,Hwang Heechan,Heechan Hwang,687,Heechan Hwang,31,MF,11
6,Hwang Inbeom,Inbeom Hwang,682,Inbeom Hwang,31,MF,6
7,Jo Hyeonwoo,Hyeonwoo Jo,697,Hyeonwoo Jo,31,GK,21
8,Kim Jingyu,Jingyu Kim,700,Jingyu Kim,31,MF,24
9,Kim Minjae,Minjae Kim,680,Minjae Kim,31,DF,4


## 45. Analyze All Remaining Unmatched FIFA Players

The Korean naming mismatch has been fully resolved.

We will now identify the remaining unmatched FIFA players across
all categories.

Players already matching by normalized name will be excluded.

For each remaining player, we will show:

- FIFA PlayerName
- FIFA Nation
- FIFA Position
- Whether the same name exists locally
- Potential local name candidates

No PlayerID assignments will be made yet.

In [83]:
# ============================================================
# STEP 45 — REMAINING UNMATCHED PLAYERS
# ============================================================

# Collect unique unmatched FIFA players from all categories
all_unmatched = []

for category, df in fifa_final.items():

    unmatched = df[
        ~df["NameNormalized"].isin(
            players_df["NameNormalized"]
        )
    ].copy()

    if len(unmatched) > 0:

        temp = unmatched[
            [
                "PlayerName",
                "Nation",
                "Position"
            ]
        ].copy()

        temp["Category"] = category

        all_unmatched.append(temp)


all_unmatched_df = pd.concat(
    all_unmatched,
    ignore_index=True
)

# Remove duplicate player/category combinations
all_unmatched_df = (
    all_unmatched_df
    .drop_duplicates()
    .sort_values(
        ["Nation", "PlayerName", "Category"]
    )
    .reset_index(drop=True)
)


# ============================================================
# EXCLUDE KOREAN PLAYERS
# ============================================================

remaining_unmatched_df = all_unmatched_df[
    all_unmatched_df["Nation"].ne("KOR")
].copy()


print("=" * 70)
print("REMAINING UNMATCHED FIFA PLAYERS")
print("=" * 70)

print(
    "Unique unmatched players:",
    remaining_unmatched_df[
        ["PlayerName", "Nation", "Position"]
    ].drop_duplicates().shape[0]
)

print(
    "Countries involved:",
    remaining_unmatched_df["Nation"].nunique()
)

print(
    "Total unmatched category records:",
    len(remaining_unmatched_df)
)


# ============================================================
# DISPLAY PLAYERS
# ============================================================

unique_remaining = (
    remaining_unmatched_df[
        [
            "PlayerName",
            "Nation",
            "Position"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["Nation", "PlayerName"]
    )
    .reset_index(drop=True)
)

display(unique_remaining)


# ============================================================
# CHECK WHETHER NAME EXISTS LOCALLY WITHOUT NORMALIZATION
# ============================================================

local_names = set(
    players_df["PlayerName"]
    .astype(str)
    .str.upper()
)

unique_remaining["ExactTextMatchLocal"] = (
    unique_remaining["PlayerName"]
    .str.upper()
    .isin(local_names)
)

print("\nExact text matches among unmatched players:")

display(
    unique_remaining[
        unique_remaining["ExactTextMatchLocal"]
    ]
)

REMAINING UNMATCHED FIFA PLAYERS
Unique unmatched players: 0
Countries involved: 0
Total unmatched category records: 0


,PlayerName,Nation,Position



Exact text matches among unmatched players:


,PlayerName,Nation,Position,ExactTextMatchLocal


## 46. Build Master FIFA → PlayerID Mapping

All FIFA players can now be matched to the local Players.csv.

We will create one master mapping containing:

- FIFA PlayerName
- FIFA Nation
- FIFA Position
- Local PlayerName
- PlayerID
- TeamID

The Korean players will use the validated reversed-name mapping.

All other players will use exact normalized-name matching.

This mapping will become the key used to merge all FIFA statistics
into the project.

In [84]:
# ============================================================
# STEP 46 — MASTER FIFA → PLAYERID MAPPING
# ============================================================

# ------------------------------------------------------------
# Start with all FIFA player identities
# ------------------------------------------------------------

all_fifa_players = []

for category, df in fifa_final.items():

    temp = df[
        [
            "PlayerName",
            "Nation",
            "Position",
            "NameNormalized"
        ]
    ].copy()

    all_fifa_players.append(temp)


fifa_player_master = pd.concat(
    all_fifa_players,
    ignore_index=True
)

# Keep one identity per FIFA player
fifa_player_master = (
    fifa_player_master[
        [
            "PlayerName",
            "Nation",
            "Position",
            "NameNormalized"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Standard exact name mapping
# ------------------------------------------------------------

local_lookup = (
    players_df[
        [
            "PlayerID",
            "PlayerName",
            "TeamID",
            "Position",
            "NameNormalized"
        ]
    ]
    .copy()
)

# Remove the ambiguous duplicate name from the automatic lookup.
# Emiliano Martinez will be resolved using Nation/TeamID below.

unique_local_names = (
    local_lookup
    .groupby("NameNormalized")
    .filter(
        lambda x: len(x) == 1
    )
)

name_lookup = (
    unique_local_names
    .set_index("NameNormalized")
)


# Map unique names
fifa_player_master["PlayerID"] = (
    fifa_player_master["NameNormalized"]
    .map(name_lookup["PlayerID"])
)

fifa_player_master["LocalPlayerName"] = (
    fifa_player_master["NameNormalized"]
    .map(name_lookup["PlayerName"])
)

fifa_player_master["TeamID"] = (
    fifa_player_master["NameNormalized"]
    .map(name_lookup["TeamID"])
)


# ------------------------------------------------------------
# Resolve Emiliano Martinez using Nation
# ------------------------------------------------------------

emiliano_mask = (
    fifa_player_master["NameNormalized"]
    == "EMILIANO MARTINEZ"
)

print(
    "Emiliano Martinez FIFA records:",
    emiliano_mask.sum()
)

if emiliano_mask.sum() > 0:

    emiliano_local = local_lookup[
        local_lookup["NameNormalized"]
        == "EMILIANO MARTINEZ"
    ]

    print("\nLocal Emiliano Martinez records:")
    display(emiliano_local)

    # Match FIFA Nation to local TeamID via Teams.csv
    for idx in fifa_player_master[
        emiliano_mask
    ].index:

        fifa_nation = (
            fifa_player_master
            .loc[idx, "Nation"]
            .strip()
            .upper()
        )

        matching_team = teams_df[
            teams_df["CountryCode"]
            .astype(str)
            .str.upper()
            .eq(fifa_nation)
        ]

        if len(matching_team) == 1:

            team_id = matching_team.iloc[0]["TeamID"]

            candidate = emiliano_local[
                emiliano_local["TeamID"]
                == team_id
            ]

            if len(candidate) == 1:

                fifa_player_master.loc[
                    idx,
                    "PlayerID"
                ] = candidate.iloc[0]["PlayerID"]

                fifa_player_master.loc[
                    idx,
                    "LocalPlayerName"
                ] = candidate.iloc[0]["PlayerName"]

                fifa_player_master.loc[
                    idx,
                    "TeamID"
                ] = candidate.iloc[0]["TeamID"]


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MASTER FIFA → PLAYERID MAPPING")
print("=" * 70)

print(
    "Unique FIFA player identities:",
    len(fifa_player_master)
)

print(
    "Matched PlayerIDs:",
    fifa_player_master["PlayerID"].notna().sum()
)

print(
    "Unmatched PlayerIDs:",
    fifa_player_master["PlayerID"].isna().sum()
)

print(
    "Unique PlayerIDs:",
    fifa_player_master["PlayerID"]
    .dropna()
    .nunique()
)

print(
    "Duplicate PlayerID mappings:",
    fifa_player_master[
        "PlayerID"
    ]
    .dropna()
    .duplicated()
    .sum()
)

print("\nEmiliano Martinez mapping:")

display(
    fifa_player_master[
        emiliano_mask
    ][
        [
            "PlayerName",
            "Nation",
            "Position",
            "PlayerID",
            "LocalPlayerName",
            "TeamID"
        ]
    ]
)

print("\nFirst 20 mappings:")

display(
    fifa_player_master[
        [
            "PlayerName",
            "Nation",
            "Position",
            "PlayerID",
            "LocalPlayerName",
            "TeamID"
        ]
    ].head(20)
)

Emiliano Martinez FIFA records: 2

Local Emiliano Martinez records:


,PlayerID,PlayerName,TeamID,Position,NameNormalized
48,49,Emiliano Martinez,1,GK,EMILIANO MARTINEZ
1184,1185,Emiliano Martinez,36,MF,EMILIANO MARTINEZ



MASTER FIFA → PLAYERID MAPPING
Unique FIFA player identities: 1242
Matched PlayerIDs: 1216
Unmatched PlayerIDs: 26
Unique PlayerIDs: 1216
Duplicate PlayerID mappings: 0

Emiliano Martinez mapping:


,PlayerName,Nation,Position,PlayerID,LocalPlayerName,TeamID
173,Emiliano Martinez,ARG,GK,49.0,Emiliano Martinez,1.0
334,Emiliano Martinez,URU,MF,1185.0,Emiliano Martinez,36.0



First 20 mappings:


,PlayerName,Nation,Position,PlayerID,LocalPlayerName,TeamID
0,Michael Olise,FRA,FW,479.0,Michael Olise,6.0
1,Martin Odegaard,NOR,MF,816.0,Martin Odegaard,22.0
2,Kylian Mbappe,FRA,FW,478.0,Kylian Mbappe,6.0
3,Brahim Diaz,MAR,FW,738.0,Brahim Diaz,21.0
4,Bruno Guimaraes,BRA,MF,164.0,Bruno Guimaraes,3.0
5,Lionel Messi,ARG,FW,36.0,Lionel Messi,1.0
6,Roberto Alvarado,MEX,FW,727.0,Roberto Alvarado,8.0
7,Anthony Gordon,ENG,FW,460.0,Anthony Gordon,5.0
8,Florian Wirtz,GER,MF,511.0,Florian Wirtz,7.0
9,Andreas Schjelderup,NOR,MF,827.0,Andreas Schjelderup,22.0


## Step 47 — Resolve Korean Republic FIFA Player Names

FIFA displays several Korea Republic players as `Surname GivenName`,
while Players.csv stores them as `GivenName Surname`.

We will use the validated Korean player mapping to resolve these
26 remaining FIFA identities.

In [86]:
# ================================================================
# STEP 47 — FIX KOREA REPUBLIC PLAYER MAPPINGS
# ================================================================

# Start from the existing master mapping
fifa_player_master_fixed = fifa_player_master.copy()

# ------------------------------------------------
# Inspect the Korean mapping columns first
# ------------------------------------------------

print("Korean mapping columns:")
print(korean_mapping.columns.tolist())

print("\nKorean mapping sample:")
display(korean_mapping.head())

# ------------------------------------------------
# Find the columns containing the FIFA name and
# local PlayerID automatically
# ------------------------------------------------

print("\nUnmatched Korean FIFA players:")

korea_unmatched = fifa_player_master[
    (fifa_player_master["Nation"] == "KOR") &
    (fifa_player_master["PlayerID"].isna())
].copy()

display(
    korea_unmatched[
        ["PlayerName", "Nation", "Position"]
    ].sort_values("PlayerName")
)

# ------------------------------------------------
# Build mapping using the previously validated
# Korean name reversal.
#
# korean_mapping was created from:
# FIFA surname + given name
#          ↓
# local given name + surname
# ------------------------------------------------

# Detect likely columns
fifa_name_col = [
    c for c in korean_mapping.columns
    if "FIFA" in c.upper() and "NAME" in c.upper()
][0]

player_id_col = "PlayerID"

# Use PlayerName if it exists in korean_mapping;
# otherwise use the detected FIFA-name column.
if "PlayerName" in korean_mapping.columns:
    korean_fifa_name_col = "PlayerName"
else:
    korean_fifa_name_col = fifa_name_col

# ------------------------------------------------
# Create a simple FIFA-name → PlayerID dictionary
# ------------------------------------------------

korea_id_lookup = dict(
    zip(
        korean_mapping[korean_fifa_name_col],
        korean_mapping[player_id_col]
    )
)

# ------------------------------------------------
# Apply mapping
# ------------------------------------------------

fifa_player_master_fixed.loc[
    (fifa_player_master_fixed["Nation"] == "KOR") &
    (fifa_player_master_fixed["PlayerID"].isna()),
    "PlayerID"
] = (
    fifa_player_master_fixed.loc[
        (fifa_player_master_fixed["Nation"] == "KOR") &
        (fifa_player_master_fixed["PlayerID"].isna()),
        "PlayerName"
    ]
    .map(korea_id_lookup)
)

# ------------------------------------------------
# Final validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FIFA → PLAYERID MAPPING VALIDATION")
print("=" * 70)

print(
    f"Unique FIFA player identities: "
    f"{len(fifa_player_master_fixed)}"
)

print(
    f"Matched PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].notna().sum()}"
)

print(
    f"Unmatched PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].isna().sum()}"
)

print(
    f"Unique PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].dropna().nunique()}"
)

duplicate_playerids = (
    fifa_player_master_fixed["PlayerID"]
    .dropna()
    .duplicated()
    .sum()
)

print(
    f"Duplicate PlayerID mappings: "
    f"{duplicate_playerids}"
)

# ------------------------------------------------
# Show remaining unmatched players
# ------------------------------------------------

remaining_unmatched = fifa_player_master_fixed[
    fifa_player_master_fixed["PlayerID"].isna()
][
    ["PlayerName", "Nation", "Position"]
]

print("\nRemaining unmatched players:")

if len(remaining_unmatched) == 0:
    print("NONE — all FIFA players successfully mapped!")
else:
    display(remaining_unmatched)

# ------------------------------------------------
# Show Korean mappings after correction
# ------------------------------------------------

korea_validation = fifa_player_master_fixed[
    fifa_player_master_fixed["Nation"] == "KOR"
][
    ["PlayerName", "Nation", "Position", "PlayerID"]
].sort_values("PlayerName")

print("\nKorea Republic mappings:")
display(korea_validation)

Korean mapping columns:
['FIFA_PlayerName', 'FIFA_NameNormalized', 'ReversedName', 'ReversedNameNormalized', 'PlayerID', 'PlayerName', 'TeamID', 'Position', 'JerseyNumber', 'LocalNameNormalized']

Korean mapping sample:


,FIFA_PlayerName,FIFA_NameNormalized,ReversedName,ReversedNameNormalized,PlayerID,PlayerName,TeamID,Position,JerseyNumber,LocalNameNormalized
0,Bae Junho,BAE JUNHO,Junho Bae,JUNHO BAE,693,Junho Bae,31,MF,17,JUNHO BAE
1,Castrop Jens,CASTROP JENS,Jens Castrop,JENS CASTROP,699,Jens Castrop,31,DF,23,JENS CASTROP
2,Cho Guesung,CHO GUESUNG,Guesung Cho,GUESUNG CHO,685,Guesung Cho,31,FW,9,GUESUNG CHO
3,Cho Wije,CHO WIJE,Wije Cho,WIJE CHO,690,Wije Cho,31,DF,14,WIJE CHO
4,Eom Jisung,EOM JISUNG,Jisung Eom,JISUNG EOM,701,Jisung Eom,31,MF,25,JISUNG EOM



Unmatched Korean FIFA players:


,PlayerName,Nation,Position
778,Bae Junho,KOR,MF
953,Castrop Jens,KOR,DF
934,Cho Guesung,KOR,FW
790,Cho Wije,KOR,DF
957,Eom Jisung,KOR,MF
1024,Hwang Heechan,KOR,MF
62,Hwang Inbeom,KOR,MF
818,Jo Hyeonwoo,KOR,GK
966,Kim Jingyu,KOR,MF
1119,Kim Minjae,KOR,DF



FINAL FIFA → PLAYERID MAPPING VALIDATION
Unique FIFA player identities: 1242
Matched PlayerIDs: 1216
Unmatched PlayerIDs: 26
Unique PlayerIDs: 1216
Duplicate PlayerID mappings: 0

Remaining unmatched players:


,PlayerName,Nation,Position
62,Hwang Inbeom,KOR,MF
75,Lee Kangin,KOR,MF
723,Oh Hyeongyu,KOR,FW
778,Bae Junho,KOR,MF
790,Cho Wije,KOR,DF
818,Jo Hyeonwoo,KOR,GK
822,Kim Taehyeon,KOR,DF
826,Lee Donggyeong,KOR,MF
855,Song Bumkeun,KOR,GK
904,Yang Hyunjun,KOR,MF



Korea Republic mappings:


,PlayerName,Nation,Position,PlayerID
778,Bae Junho,KOR,MF,NaN
953,Castrop Jens,KOR,DF,NaN
934,Cho Guesung,KOR,FW,NaN
790,Cho Wije,KOR,DF,NaN
957,Eom Jisung,KOR,MF,NaN
1024,Hwang Heechan,KOR,MF,NaN
62,Hwang Inbeom,KOR,MF,NaN
818,Jo Hyeonwoo,KOR,GK,NaN
966,Kim Jingyu,KOR,MF,NaN
1119,Kim Minjae,KOR,DF,NaN


## Step 47 — Resolve Korea Republic PlayerIDs

The FIFA website uses `Surname GivenName` for Korea Republic players,
while Players.csv uses `GivenName Surname`.

The previously validated `korean_mapping` contains the exact
FIFA name → PlayerID relationship.

We will apply that mapping directly.

In [87]:
# ================================================================
# STEP 47 — FINAL KOREA REPUBLIC MAPPING
# ================================================================

# Copy the master mapping
fifa_player_master_fixed = fifa_player_master.copy()

# ------------------------------------------------
# Create exact FIFA name → PlayerID lookup
# ------------------------------------------------

korea_id_lookup = (
    korean_mapping
    .set_index("FIFA_PlayerName")["PlayerID"]
    .to_dict()
)

# ------------------------------------------------
# Identify unmatched Korea Republic players
# ------------------------------------------------

korea_mask = (
    (fifa_player_master_fixed["Nation"] == "KOR") &
    (fifa_player_master_fixed["PlayerID"].isna())
)

print(f"Unmatched Korea Republic players: {korea_mask.sum()}")

# ------------------------------------------------
# Apply exact mapping
# ------------------------------------------------

fifa_player_master_fixed.loc[korea_mask, "PlayerID"] = (
    fifa_player_master_fixed.loc[korea_mask, "PlayerName"]
    .map(korea_id_lookup)
)

# ------------------------------------------------
# Final validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FIFA → PLAYERID MAPPING VALIDATION")
print("=" * 70)

print(
    f"Unique FIFA player identities: "
    f"{len(fifa_player_master_fixed)}"
)

print(
    f"Matched PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].notna().sum()}"
)

print(
    f"Unmatched PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].isna().sum()}"
)

print(
    f"Unique PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].dropna().nunique()}"
)

duplicate_playerids = (
    fifa_player_master_fixed["PlayerID"]
    .dropna()
    .duplicated()
    .sum()
)

print(
    f"Duplicate PlayerID mappings: "
    f"{duplicate_playerids}"
)

# ------------------------------------------------
# Show Korean mappings
# ------------------------------------------------

print("\nKorea Republic mappings:")

korea_validation = (
    fifa_player_master_fixed[
        fifa_player_master_fixed["Nation"] == "KOR"
    ][
        ["PlayerName", "Nation", "Position", "PlayerID"]
    ]
    .sort_values("PlayerName")
)

display(korea_validation)

# ------------------------------------------------
# Show remaining unmatched players
# ------------------------------------------------

remaining_unmatched = fifa_player_master_fixed[
    fifa_player_master_fixed["PlayerID"].isna()
][
    ["PlayerName", "Nation", "Position"]
]

print("\nRemaining unmatched players:")

if remaining_unmatched.empty:
    print("NONE — ALL FIFA PLAYERS SUCCESSFULLY MAPPED!")
else:
    display(remaining_unmatched)

Unmatched Korea Republic players: 26

FINAL FIFA → PLAYERID MAPPING VALIDATION
Unique FIFA player identities: 1242
Matched PlayerIDs: 1242
Unmatched PlayerIDs: 0
Unique PlayerIDs: 1242
Duplicate PlayerID mappings: 0

Korea Republic mappings:


,PlayerName,Nation,Position,PlayerID
778,Bae Junho,KOR,MF,693.0
953,Castrop Jens,KOR,DF,699.0
934,Cho Guesung,KOR,FW,685.0
790,Cho Wije,KOR,DF,690.0
957,Eom Jisung,KOR,MF,701.0
1024,Hwang Heechan,KOR,MF,687.0
62,Hwang Inbeom,KOR,MF,682.0
818,Jo Hyeonwoo,KOR,GK,697.0
966,Kim Jingyu,KOR,MF,700.0
1119,Kim Minjae,KOR,DF,680.0



Remaining unmatched players:
NONE — ALL FIFA PLAYERS SUCCESSFULLY MAPPED!


## Step 48 — Save Master FIFA → PlayerID Mapping

The FIFA player identities have been successfully matched to the
local Players.csv PlayerIDs.

We will now save this validated mapping as a CSV checkpoint before
merging the FIFA statistical categories.

In [88]:
# ================================================================
# STEP 48 — SAVE MASTER FIFA → PLAYERID MAPPING
# ================================================================

mapping_path = r"D:\ai course hw\FIFA_WorldCup_2026_Project\Data\FIFA_PlayerID_Mapping.csv"

# Save the validated mapping
fifa_player_master_fixed.to_csv(
    mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("MASTER MAPPING SAVED")
print("=" * 70)

print(f"File: {mapping_path}")
print(f"Rows: {len(fifa_player_master_fixed)}")
print(f"Columns: {len(fifa_player_master_fixed.columns)}")

print("\nValidation before saving:")
print(
    f"Matched PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].notna().sum()}"
)

print(
    f"Unmatched PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].isna().sum()}"
)

print(
    f"Unique PlayerIDs: "
    f"{fifa_player_master_fixed['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerID mappings: "
    f"{fifa_player_master_fixed['PlayerID'].duplicated().sum()}"
)

print("\nFirst 10 rows:")
display(fifa_player_master_fixed.head(10))

MASTER MAPPING SAVED
File: D:\ai course hw\FIFA_WorldCup_2026_Project\Data\FIFA_PlayerID_Mapping.csv
Rows: 1242
Columns: 7

Validation before saving:
Matched PlayerIDs: 1242
Unmatched PlayerIDs: 0
Unique PlayerIDs: 1242
Duplicate PlayerID mappings: 0

First 10 rows:


,PlayerName,Nation,Position,NameNormalized,PlayerID,LocalPlayerName,TeamID
0,Michael Olise,FRA,FW,MICHAEL OLISE,479.0,Michael Olise,6.0
1,Martin Odegaard,NOR,MF,MARTIN ODEGAARD,816.0,Martin Odegaard,22.0
2,Kylian Mbappe,FRA,FW,KYLIAN MBAPPE,478.0,Kylian Mbappe,6.0
3,Brahim Diaz,MAR,FW,BRAHIM DIAZ,738.0,Brahim Diaz,21.0
4,Bruno Guimaraes,BRA,MF,BRUNO GUIMARAES,164.0,Bruno Guimaraes,3.0
5,Lionel Messi,ARG,FW,LIONEL MESSI,36.0,Lionel Messi,1.0
6,Roberto Alvarado,MEX,FW,ROBERTO ALVARADO,727.0,Roberto Alvarado,8.0
7,Anthony Gordon,ENG,FW,ANTHONY GORDON,460.0,Anthony Gordon,5.0
8,Florian Wirtz,GER,MF,FLORIAN WIRTZ,511.0,Florian Wirtz,7.0
9,Andreas Schjelderup,NOR,MF,ANDREAS SCHJELDERUP,827.0,Andreas Schjelderup,22.0


## Step 49 — Prepare FIFA Category Data for Merging

The six FIFA statistical categories will be converted into
PlayerID-based datasets.

Identification columns such as PlayerName, Nation, Position, and Rank
will not be used as the merge key.

Statistical columns will receive category prefixes to prevent
duplicate column names when the datasets are combined.

In [89]:
# ================================================================
# STEP 49 — PREPARE FIFA CATEGORIES FOR MERGING
# ================================================================

# Work from the validated FIFA category datasets
fifa_categories = {
    "GoldenBoot": fifa_final["adidas Golden Boot"].copy(),
    "Attacking": fifa_final["Attacking"].copy(),
    "Distribution": fifa_final["Distribution"].copy(),
    "Defending": fifa_final["Defending"].copy(),
    "Discipline": fifa_final["Discipline"].copy(),
    "Goalkeeping": fifa_final["Goalkeeping"].copy()
}

# ------------------------------------------------
# Add PlayerID to every FIFA category
# ------------------------------------------------

for category, df in fifa_categories.items():

    # Merge the validated FIFA → PlayerID mapping
    df = df.merge(
        fifa_player_master_fixed[
            ["PlayerName", "Nation", "Position", "PlayerID"]
        ],
        on=["PlayerName", "Nation", "Position"],
        how="left"
    )

    fifa_categories[category] = df

# ------------------------------------------------
# Validate PlayerID assignment
# ------------------------------------------------

print("=" * 70)
print("FIFA CATEGORY PLAYERID VALIDATION")
print("=" * 70)

for category, df in fifa_categories.items():

    print(f"\n{category}")
    print("-" * 50)
    print(f"Rows: {len(df)}")
    print(f"PlayerIDs assigned: {df['PlayerID'].notna().sum()}")
    print(f"PlayerIDs missing: {df['PlayerID'].isna().sum()}")
    print(f"Unique PlayerIDs: {df['PlayerID'].nunique()}")
    print(f"Duplicate PlayerIDs: {df['PlayerID'].duplicated().sum()}")

print("\n" + "=" * 70)
print("PREPARATION COMPLETE")
print("=" * 70)

FIFA CATEGORY PLAYERID VALIDATION

GoldenBoot
--------------------------------------------------
Rows: 1231
PlayerIDs assigned: 1231
PlayerIDs missing: 0
Unique PlayerIDs: 1231
Duplicate PlayerIDs: 0

Attacking
--------------------------------------------------
Rows: 700
PlayerIDs assigned: 700
PlayerIDs missing: 0
Unique PlayerIDs: 700
Duplicate PlayerIDs: 0

Distribution
--------------------------------------------------
Rows: 1242
PlayerIDs assigned: 1242
PlayerIDs missing: 0
Unique PlayerIDs: 1242
Duplicate PlayerIDs: 0

Defending
--------------------------------------------------
Rows: 1242
PlayerIDs assigned: 1242
PlayerIDs missing: 0
Unique PlayerIDs: 1242
Duplicate PlayerIDs: 0

Discipline
--------------------------------------------------
Rows: 1242
PlayerIDs assigned: 1242
PlayerIDs missing: 0
Unique PlayerIDs: 1242
Duplicate PlayerIDs: 0

Goalkeeping
--------------------------------------------------
Rows: 145
PlayerIDs assigned: 145
PlayerIDs missing: 0
Unique PlayerIDs: 14

## Step 50 — Prefix FIFA Statistical Columns

Each FIFA category contains its own statistical fields.

We will add a category prefix to every statistical column while
keeping PlayerID as the common merge key.

This will prevent column-name conflicts when all six categories
are combined into one player-level FIFA statistics dataset.

In [90]:
# ================================================================
# STEP 50 — PREFIX FIFA STATISTICAL COLUMNS
# ================================================================

# Columns used only to identify the player
id_columns = [
    "PlayerID",
    "PlayerName",
    "Nation",
    "Position"
]

# Create a new dictionary for the prepared datasets
fifa_categories_prefixed = {}

for category, df in fifa_categories.items():

    df = df.copy()

    # Keep PlayerID
    # Prefix every other FIFA statistical column
    rename_columns = {}

    for column in df.columns:

        if column not in id_columns:
            rename_columns[column] = f"{category}_{column}"

    df = df.rename(columns=rename_columns)

    fifa_categories_prefixed[category] = df

# ------------------------------------------------
# Display columns for every category
# ------------------------------------------------

print("=" * 70)
print("PREFIXED FIFA CATEGORY COLUMNS")
print("=" * 70)

for category, df in fifa_categories_prefixed.items():

    print(f"\n{category}")
    print("-" * 50)
    print(df.columns.tolist())

print("\n" + "=" * 70)
print("PREFIXING COMPLETE")
print("=" * 70)

PREFIXED FIFA CATEGORY COLUMNS

GoldenBoot
--------------------------------------------------
['GoldenBoot_Rank', 'GoldenBoot_Goals', 'GoldenBoot_Assists', 'GoldenBoot_Minutes Played', 'PlayerName', 'Nation', 'Position', 'GoldenBoot_NameNormalized', 'PlayerID']

Attacking
--------------------------------------------------
['Attacking_Rank', 'Attacking_Assists', 'Attacking_Attempts On Target', 'Attacking_Attempts At Goal', 'Attacking_Attempts At Goal Conv. Rate (%)', 'Attacking_Attempts Inside the Penalty Area', 'Attacking_Attempts Outside the Penalty Area', 'Attacking_Headed Attempts at Goal', 'Attacking_xG', 'Attacking_xG Efficiency', 'Attacking_Corners', 'PlayerName', 'Nation', 'Position', 'Attacking_NameNormalized', 'PlayerID']

Distribution
--------------------------------------------------
['Distribution_Rank', 'Distribution_Passes', 'Distribution_Passing Accuracy (%)', 'Distribution_Crosses', 'Distribution_Crossing Accuracy (%)', 'Distribution_Defensive Linebreaks Attempted', 'Di

## Step 51 — Build Combined FIFA Player Statistics

The six FIFA statistical categories will be merged into one
player-level dataset using the validated PlayerID.

Player identity fields will come from the master FIFA → PlayerID
mapping.

The statistical fields from all six categories will then be added.

In [91]:
# ================================================================
# STEP 51 — BUILD COMBINED FIFA PLAYER STATISTICS
# ================================================================

# ------------------------------------------------
# 1. Start with the master FIFA player mapping
# ------------------------------------------------

fifa_player_stats_df = fifa_player_master_fixed[
    [
        "PlayerID",
        "PlayerName",
        "Nation",
        "Position"
    ]
].copy()

# ------------------------------------------------
# 2. Merge each FIFA category
# ------------------------------------------------

for category, df in fifa_categories_prefixed.items():

    # Keep only PlayerID + statistical columns
    merge_columns = [
        column
        for column in df.columns
        if column == "PlayerID"
        or column.startswith(f"{category}_")
    ]

    category_df = df[merge_columns].copy()

    # Remove helper columns that are not statistics
    category_df = category_df.drop(
        columns=[
            column
            for column in category_df.columns
            if column.endswith("_NameNormalized")
        ],
        errors="ignore"
    )

    # Merge onto master player table
    fifa_player_stats_df = fifa_player_stats_df.merge(
        category_df,
        on="PlayerID",
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------
# 3. Validation
# ------------------------------------------------

print("=" * 70)
print("COMBINED FIFA PLAYER STATISTICS")
print("=" * 70)

print(f"Rows: {len(fifa_player_stats_df)}")
print(f"Columns: {len(fifa_player_stats_df.columns)}")

print(
    f"Unique PlayerIDs: "
    f"{fifa_player_stats_df['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{fifa_player_stats_df['PlayerID'].duplicated().sum()}"
)

print("\nMissing PlayerIDs:")
print(fifa_player_stats_df["PlayerID"].isna().sum())

# ------------------------------------------------
# 4. Display the final column list
# ------------------------------------------------

print("\nFinal columns:")

for i, column in enumerate(
    fifa_player_stats_df.columns,
    start=1
):
    print(f"{i:2}. {column}")

# ------------------------------------------------
# 5. Preview
# ------------------------------------------------

print("\nFirst 10 rows:")
display(fifa_player_stats_df.head(10))

COMBINED FIFA PLAYER STATISTICS
Rows: 1242
Columns: 44
Unique PlayerIDs: 1242
Duplicate PlayerIDs: 0

Missing PlayerIDs:
0

Final columns:
 1. PlayerID
 2. PlayerName
 3. Nation
 4. Position
 5. GoldenBoot_Rank
 6. GoldenBoot_Goals
 7. GoldenBoot_Assists
 8. GoldenBoot_Minutes Played
 9. Attacking_Rank
10. Attacking_Assists
11. Attacking_Attempts On Target
12. Attacking_Attempts At Goal
13. Attacking_Attempts At Goal Conv. Rate (%)
14. Attacking_Attempts Inside the Penalty Area
15. Attacking_Attempts Outside the Penalty Area
16. Attacking_Headed Attempts at Goal
17. Attacking_xG
18. Attacking_xG Efficiency
19. Attacking_Corners
20. Distribution_Rank
21. Distribution_Passes
22. Distribution_Passing Accuracy (%)
23. Distribution_Crosses
24. Distribution_Crossing Accuracy (%)
25. Distribution_Defensive Linebreaks Attempted
26. Distribution_Defensive Linebreaks Acc (%)
27. Distribution_Switches of Play Attempted
28. Distribution_Switches of Play Acc (%)
29. Defending_Rank
30. Defending_Own

,PlayerID,PlayerName,Nation,Position,GoldenBoot_Rank,GoldenBoot_Goals,GoldenBoot_Assists,GoldenBoot_Minutes Played,Attacking_Rank,Attacking_Assists,...,Discipline_Fouls Against,Discipline_Fouls For,Discipline_Yellow Cards,Discipline_Red Cards,Discipline_Indirect Red Cards,Discipline_Offsides,Goalkeeping_Rank,Goalkeeping_Goalkeeper Saves,Goalkeeping_Goalkeeper Actions Inside the Penalty Area,Goalkeeping_Goalkeeper Actions Outside the Penalty Area
0,479.0,Michael Olise,FRA,FW,179,0,7,719,1,7,...,5,16,1,0,0,0,NaN,NaN,NaN,NaN
1,816.0,Martin Odegaard,NOR,MF,182,0,4,537,2,4,...,3,4,0,0,0,0,NaN,NaN,NaN,NaN
2,478.0,Kylian Mbappe,FRA,FW,1,10,4,769,2,4,...,6,9,1,0,0,7,NaN,NaN,NaN,NaN
3,738.0,Brahim Diaz,MAR,FW,181,0,4,500,2,4,...,3,10,0,0,0,0,NaN,NaN,NaN,NaN
4,164.0,Bruno Guimaraes,BRA,MF,180,0,4,458,2,4,...,4,7,0,0,0,0,NaN,NaN,NaN,NaN
5,36.0,Lionel Messi,ARG,FW,2,8,4,853,2,4,...,3,20,0,0,0,5,NaN,NaN,NaN,NaN
6,727.0,Roberto Alvarado,MEX,FW,184,0,3,476,7,3,...,3,5,0,0,0,1,NaN,NaN,NaN,NaN
7,460.0,Anthony Gordon,ENG,FW,60,1,3,442,7,3,...,4,10,0,0,0,0,NaN,NaN,NaN,NaN
8,511.0,Florian Wirtz,GER,MF,183,0,3,405,7,3,...,1,3,0,0,0,1,NaN,NaN,NaN,NaN
9,827.0,Andreas Schjelderup,NOR,MF,58,1,3,297,7,3,...,2,3,0,0,0,0,NaN,NaN,NaN,NaN


## Step 52 — Clean FIFA Column Names

The combined FIFA dataset contains column names inherited from the
FIFA website.

We will standardize these names by:

- Removing spaces
- Removing parentheses and percentage symbols
- Removing punctuation
- Using clear category prefixes
- Keeping PlayerID, PlayerName, Nation, and Position unchanged

The result will be easier to use in pandas and Power BI.

In [92]:
# ================================================================
# STEP 52 — CLEAN FIFA COLUMN NAMES
# ================================================================

import re

# Columns that should remain unchanged
identity_columns = [
    "PlayerID",
    "PlayerName",
    "Nation",
    "Position"
]

# Create a copy
fifa_player_stats_clean = fifa_player_stats_df.copy()

# ------------------------------------------------
# Clean column names
# ------------------------------------------------

clean_columns = {}

for column in fifa_player_stats_clean.columns:

    if column in identity_columns:
        clean_columns[column] = column
        continue

    # Replace % with Pct
    new_name = column.replace("(%)", "Pct")
    new_name = new_name.replace("%", "Pct")

    # Remove punctuation
    new_name = re.sub(r"[^A-Za-z0-9_ ]", "", new_name)

    # Remove spaces
    new_name = new_name.replace(" ", "")

    # Avoid accidental double underscores
    new_name = re.sub(r"_+", "_", new_name)

    clean_columns[column] = new_name

# Apply names
fifa_player_stats_clean = fifa_player_stats_clean.rename(
    columns=clean_columns
)

# ------------------------------------------------
# Display final column names
# ------------------------------------------------

print("=" * 70)
print("CLEAN FIFA COLUMN NAMES")
print("=" * 70)

for i, column in enumerate(
    fifa_player_stats_clean.columns,
    start=1
):
    print(f"{i:2}. {column}")

print("\n" + "=" * 70)
print("COLUMN NAME CLEANING COMPLETE")
print("=" * 70)

CLEAN FIFA COLUMN NAMES
 1. PlayerID
 2. PlayerName
 3. Nation
 4. Position
 5. GoldenBoot_Rank
 6. GoldenBoot_Goals
 7. GoldenBoot_Assists
 8. GoldenBoot_MinutesPlayed
 9. Attacking_Rank
10. Attacking_Assists
11. Attacking_AttemptsOnTarget
12. Attacking_AttemptsAtGoal
13. Attacking_AttemptsAtGoalConvRatePct
14. Attacking_AttemptsInsidethePenaltyArea
15. Attacking_AttemptsOutsidethePenaltyArea
16. Attacking_HeadedAttemptsatGoal
17. Attacking_xG
18. Attacking_xGEfficiency
19. Attacking_Corners
20. Distribution_Rank
21. Distribution_Passes
22. Distribution_PassingAccuracyPct
23. Distribution_Crosses
24. Distribution_CrossingAccuracyPct
25. Distribution_DefensiveLinebreaksAttempted
26. Distribution_DefensiveLinebreaksAccPct
27. Distribution_SwitchesofPlayAttempted
28. Distribution_SwitchesofPlayAccPct
29. Defending_Rank
30. Defending_OwnGoals
31. Defending_ForcedTurnovers
32. Defending_DefensivePressuresApplied
33. Defending_DefensivePressuresDirectlyApplied
34. Discipline_Rank
35. Discip

## Step 53 — Simplify FIFA Statistics Columns

The FIFA categories are useful for organizing the scraping process,
but the category names are not needed in the final player statistics
table.

We will:

- Remove category prefixes from statistic names
- Keep only one `Assists` column
- Use the Golden Boot `Assists` value as the main FIFA Assists field
- Remove the duplicate Attacking Assists field
- Keep PlayerID, PlayerName, Nation, and Position
- Keep Rank fields with their category-specific meaning

This creates a cleaner player-level FIFA statistics table for
analysis and Power BI.

In [93]:
# ================================================================
# STEP 53 — SIMPLIFY FIFA STATISTICS COLUMNS
# ================================================================

# Start from the cleaned FIFA dataset
fifa_player_stats_simple = fifa_player_stats_clean.copy()

# ------------------------------------------------
# 1. Remove the duplicate Attacking Assists
# ------------------------------------------------

# Golden Boot Assists will be our single FIFA Assists field
fifa_player_stats_simple = fifa_player_stats_simple.drop(
    columns=["Attacking_Assists"],
    errors="ignore"
)

# Rename Golden Boot Assists → Assists
fifa_player_stats_simple = fifa_player_stats_simple.rename(
    columns={
        "GoldenBoot_Assists": "Assists"
    }
)

# ------------------------------------------------
# 2. Remove category prefixes from remaining columns
# ------------------------------------------------

identity_columns = [
    "PlayerID",
    "PlayerName",
    "Nation",
    "Position"
]

rename_columns = {}

for column in fifa_player_stats_simple.columns:

    if column in identity_columns:
        continue

    # Remove known category prefixes
    new_name = re.sub(
        r"^(GoldenBoot|Attacking|Distribution|Defending|Discipline|Goalkeeping)_",
        "",
        column
    )

    rename_columns[column] = new_name

fifa_player_stats_simple = (
    fifa_player_stats_simple.rename(columns=rename_columns)
)

# ------------------------------------------------
# 3. Display final columns
# ------------------------------------------------

print("=" * 70)
print("SIMPLIFIED FIFA PLAYER STATISTICS")
print("=" * 70)

for i, column in enumerate(
    fifa_player_stats_simple.columns,
    start=1
):
    print(f"{i:2}. {column}")

# ------------------------------------------------
# 4. Check duplicate column names
# ------------------------------------------------

duplicate_columns = (
    fifa_player_stats_simple.columns[
        fifa_player_stats_simple.columns.duplicated()
    ]
    .tolist()
)

print("\nDuplicate column names:")

if duplicate_columns:
    print(duplicate_columns)
else:
    print("NONE")

# ------------------------------------------------
# 5. Basic validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(f"Rows: {len(fifa_player_stats_simple)}")
print(f"Columns: {len(fifa_player_stats_simple.columns)}")
print(f"Unique PlayerIDs: {fifa_player_stats_simple['PlayerID'].nunique()}")
print(
    f"Duplicate PlayerIDs: "
    f"{fifa_player_stats_simple['PlayerID'].duplicated().sum()}"
)

print("\nAssists column exists:", "Assists" in fifa_player_stats_simple.columns)

# ------------------------------------------------
# 6. Preview
# ------------------------------------------------

print("\nFirst 10 rows:")
display(fifa_player_stats_simple.head(10))

SIMPLIFIED FIFA PLAYER STATISTICS
 1. PlayerID
 2. PlayerName
 3. Nation
 4. Position
 5. Rank
 6. Goals
 7. Assists
 8. MinutesPlayed
 9. Rank
10. AttemptsOnTarget
11. AttemptsAtGoal
12. AttemptsAtGoalConvRatePct
13. AttemptsInsidethePenaltyArea
14. AttemptsOutsidethePenaltyArea
15. HeadedAttemptsatGoal
16. xG
17. xGEfficiency
18. Corners
19. Rank
20. Passes
21. PassingAccuracyPct
22. Crosses
23. CrossingAccuracyPct
24. DefensiveLinebreaksAttempted
25. DefensiveLinebreaksAccPct
26. SwitchesofPlayAttempted
27. SwitchesofPlayAccPct
28. Rank
29. OwnGoals
30. ForcedTurnovers
31. DefensivePressuresApplied
32. DefensivePressuresDirectlyApplied
33. Rank
34. FoulsAgainst
35. FoulsFor
36. YellowCards
37. RedCards
38. IndirectRedCards
39. Offsides
40. Rank
41. GoalkeeperSaves
42. GoalkeeperActionsInsidethePenaltyArea
43. GoalkeeperActionsOutsidethePenaltyArea

Duplicate column names:
['Rank', 'Rank', 'Rank', 'Rank', 'Rank']

VALIDATION
Rows: 1242
Columns: 43
Unique PlayerIDs: 1242
Duplicate Pla

,PlayerID,PlayerName,Nation,Position,Rank,Goals,Assists,MinutesPlayed,Rank,AttemptsOnTarget,...,FoulsAgainst,FoulsFor,YellowCards,RedCards,IndirectRedCards,Offsides,Rank,GoalkeeperSaves,GoalkeeperActionsInsidethePenaltyArea,GoalkeeperActionsOutsidethePenaltyArea
0,479.0,Michael Olise,FRA,FW,179,0,7,719,1,5,...,5,16,1,0,0,0,NaN,NaN,NaN,NaN
1,816.0,Martin Odegaard,NOR,MF,182,0,4,537,2,3,...,3,4,0,0,0,0,NaN,NaN,NaN,NaN
2,478.0,Kylian Mbappe,FRA,FW,1,10,4,769,2,23,...,6,9,1,0,0,7,NaN,NaN,NaN,NaN
3,738.0,Brahim Diaz,MAR,FW,181,0,4,500,2,1,...,3,10,0,0,0,0,NaN,NaN,NaN,NaN
4,164.0,Bruno Guimaraes,BRA,MF,180,0,4,458,2,2,...,4,7,0,0,0,0,NaN,NaN,NaN,NaN
5,36.0,Lionel Messi,ARG,FW,2,8,4,853,2,19,...,3,20,0,0,0,5,NaN,NaN,NaN,NaN
6,727.0,Roberto Alvarado,MEX,FW,184,0,3,476,7,0,...,3,5,0,0,0,1,NaN,NaN,NaN,NaN
7,460.0,Anthony Gordon,ENG,FW,60,1,3,442,7,4,...,4,10,0,0,0,0,NaN,NaN,NaN,NaN
8,511.0,Florian Wirtz,GER,MF,183,0,3,405,7,1,...,1,3,0,0,0,1,NaN,NaN,NaN,NaN
9,827.0,Andreas Schjelderup,NOR,MF,58,1,3,297,7,2,...,2,3,0,0,0,0,NaN,NaN,NaN,NaN


## Step 54 — Remove FIFA Rank Columns

The FIFA rank is specific to each statistical category and does not
represent a player statistic.

The combined player dataset will therefore exclude all category Rank
columns.

The final table will contain:

- PlayerID
- PlayerName
- Nation
- Position
- FIFA statistical values

There will be no duplicate column names.

In [94]:
# ================================================================
# STEP 54 — REMOVE RANK COLUMNS
# ================================================================

# Start from the simplified FIFA dataset
fifa_player_stats_final = fifa_player_stats_simple.copy()

# ------------------------------------------------
# 1. Find all Rank columns
# ------------------------------------------------

rank_columns = [
    column
    for column in fifa_player_stats_final.columns
    if column == "Rank"
]

print("Rank columns found:", len(rank_columns))

# ------------------------------------------------
# 2. Remove all Rank columns
# ------------------------------------------------

fifa_player_stats_final = fifa_player_stats_final.drop(
    columns=rank_columns
)

# ------------------------------------------------
# 3. Check for duplicate column names
# ------------------------------------------------

duplicate_columns = (
    fifa_player_stats_final.columns[
        fifa_player_stats_final.columns.duplicated()
    ]
    .tolist()
)

# ------------------------------------------------
# 4. Validate
# ------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FIFA PLAYER STATISTICS VALIDATION")
print("=" * 70)

print(f"Rows: {len(fifa_player_stats_final)}")
print(f"Columns: {len(fifa_player_stats_final.columns)}")

print(
    f"Unique PlayerIDs: "
    f"{fifa_player_stats_final['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{fifa_player_stats_final['PlayerID'].duplicated().sum()}"
)

print(
    f"Duplicate column names: "
    f"{len(duplicate_columns)}"
)

print(
    f"Assists columns: "
    f"{sum(column == 'Assists' for column in fifa_player_stats_final.columns)}"
)

# ------------------------------------------------
# 5. Show duplicate columns if any
# ------------------------------------------------

if duplicate_columns:
    print("\nDuplicate columns found:")
    print(duplicate_columns)
else:
    print("\nNo duplicate column names.")

# ------------------------------------------------
# 6. Show final columns
# ------------------------------------------------

print("\nFinal FIFA columns:")

for i, column in enumerate(
    fifa_player_stats_final.columns,
    start=1
):
    print(f"{i:2}. {column}")

# ------------------------------------------------
# 7. Preview
# ------------------------------------------------

print("\nFirst 10 rows:")
display(fifa_player_stats_final.head(10))

Rank columns found: 6

FINAL FIFA PLAYER STATISTICS VALIDATION
Rows: 1242
Columns: 37
Unique PlayerIDs: 1242
Duplicate PlayerIDs: 0
Duplicate column names: 0
Assists columns: 1

No duplicate column names.

Final FIFA columns:
 1. PlayerID
 2. PlayerName
 3. Nation
 4. Position
 5. Goals
 6. Assists
 7. MinutesPlayed
 8. AttemptsOnTarget
 9. AttemptsAtGoal
10. AttemptsAtGoalConvRatePct
11. AttemptsInsidethePenaltyArea
12. AttemptsOutsidethePenaltyArea
13. HeadedAttemptsatGoal
14. xG
15. xGEfficiency
16. Corners
17. Passes
18. PassingAccuracyPct
19. Crosses
20. CrossingAccuracyPct
21. DefensiveLinebreaksAttempted
22. DefensiveLinebreaksAccPct
23. SwitchesofPlayAttempted
24. SwitchesofPlayAccPct
25. OwnGoals
26. ForcedTurnovers
27. DefensivePressuresApplied
28. DefensivePressuresDirectlyApplied
29. FoulsAgainst
30. FoulsFor
31. YellowCards
32. RedCards
33. IndirectRedCards
34. Offsides
35. GoalkeeperSaves
36. GoalkeeperActionsInsidethePenaltyArea
37. GoalkeeperActionsOutsidethePenaltyArea

,PlayerID,PlayerName,Nation,Position,Goals,Assists,MinutesPlayed,AttemptsOnTarget,AttemptsAtGoal,AttemptsAtGoalConvRatePct,...,DefensivePressuresDirectlyApplied,FoulsAgainst,FoulsFor,YellowCards,RedCards,IndirectRedCards,Offsides,GoalkeeperSaves,GoalkeeperActionsInsidethePenaltyArea,GoalkeeperActionsOutsidethePenaltyArea
0,479.0,Michael Olise,FRA,FW,0,7,719,5,20,0,...,32,5,16,1,0,0,0,NaN,NaN,NaN
1,816.0,Martin Odegaard,NOR,MF,0,4,537,3,9,0,...,18,3,4,0,0,0,0,NaN,NaN,NaN
2,478.0,Kylian Mbappe,FRA,FW,10,4,769,23,41,24,...,15,6,9,1,0,0,7,NaN,NaN,NaN
3,738.0,Brahim Diaz,MAR,FW,0,4,500,1,6,0,...,24,3,10,0,0,0,0,NaN,NaN,NaN
4,164.0,Bruno Guimaraes,BRA,MF,0,4,458,2,8,0,...,15,4,7,0,0,0,0,NaN,NaN,NaN
5,36.0,Lionel Messi,ARG,FW,8,4,853,19,35,23,...,23,3,20,0,0,0,5,NaN,NaN,NaN
6,727.0,Roberto Alvarado,MEX,FW,0,3,476,0,5,0,...,13,3,5,0,0,0,1,NaN,NaN,NaN
7,460.0,Anthony Gordon,ENG,FW,1,3,442,4,6,17,...,15,4,10,0,0,0,0,NaN,NaN,NaN
8,511.0,Florian Wirtz,GER,MF,0,3,405,1,10,0,...,18,1,3,0,0,0,1,NaN,NaN,NaN
9,827.0,Andreas Schjelderup,NOR,MF,1,3,297,2,3,33,...,14,2,3,0,0,0,0,NaN,NaN,NaN


## Step 55 — Analyze FIFA Missing Values

Before replacing any missing values, we will inspect the FIFA
statistics column by column.

Missing values will be classified based on the FIFA category and
whether the statistic is applicable to the player.

No values will be replaced in this step.

In [95]:
# ================================================================
# STEP 55 — FIFA MISSING VALUE ANALYSIS
# ================================================================

# ------------------------------------------------
# 1. Missing values by column
# ------------------------------------------------

missing_fifa = pd.DataFrame({
    "Column": fifa_player_stats_final.columns,
    "MissingCount": fifa_player_stats_final.isna().sum(),
    "MissingPct": (
        fifa_player_stats_final.isna().mean() * 100
    ).round(2)
})

# Keep only columns with missing values
missing_fifa = (
    missing_fifa[
        missing_fifa["MissingCount"] > 0
    ]
    .sort_values(
        "MissingCount",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("FIFA MISSING VALUE ANALYSIS")
print("=" * 70)

print(f"Total rows: {len(fifa_player_stats_final)}")
print(
    f"Columns with missing values: "
    f"{len(missing_fifa)}"
)

print(
    f"Total missing cells: "
    f"{fifa_player_stats_final.isna().sum().sum()}"
)

print("\nMissing values by column:")
display(missing_fifa)

# ------------------------------------------------
# 2. Missing values by FIFA statistical group
# ------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUES BY STATISTICAL GROUP")
print("=" * 70)

groups = {
    "Goals / Assists / Minutes": [
        "Goals",
        "Assists",
        "MinutesPlayed"
    ],

    "Attacking": [
        column for column in fifa_player_stats_final.columns
        if column in [
            "AttemptsOnTarget",
            "AttemptsAtGoal",
            "AttemptsAtGoalConvRatePct",
            "AttemptsInsidethePenaltyArea",
            "AttemptsOutsidethePenaltyArea",
            "HeadedAttemptsatGoal",
            "xG",
            "xGEfficiency",
            "Corners"
        ]
    ],

    "Distribution": [
        column for column in fifa_player_stats_final.columns
        if column in [
            "Passes",
            "PassingAccuracyPct",
            "Crosses",
            "CrossingAccuracyPct",
            "DefensiveLinebreaksAttempted",
            "DefensiveLinebreaksAccPct",
            "SwitchesofPlayAttempted",
            "SwitchesofPlayAccPct"
        ]
    ],

    "Defending": [
        column for column in fifa_player_stats_final.columns
        if column in [
            "OwnGoals",
            "ForcedTurnovers",
            "DefensivePressuresApplied",
            "DefensivePressuresDirectlyApplied"
        ]
    ],

    "Discipline": [
        column for column in fifa_player_stats_final.columns
        if column in [
            "FoulsAgainst",
            "FoulsFor",
            "YellowCards",
            "RedCards",
            "IndirectRedCards",
            "Offsides"
        ]
    ],

    "Goalkeeping": [
        column for column in fifa_player_stats_final.columns
        if column in [
            "GoalkeeperSaves",
            "GoalkeeperActionsInsidethePenaltyArea",
            "GoalkeeperActionsOutsidethePenaltyArea"
        ]
    ]
}

for group_name, columns in groups.items():

    existing_columns = [
        column
        for column in columns
        if column in fifa_player_stats_final.columns
    ]

    if existing_columns:

        missing_count = (
            fifa_player_stats_final[existing_columns]
            .isna()
            .sum()
            .sum()
        )

        print(
            f"{group_name}: "
            f"{missing_count:,} missing cells"
        )

# ------------------------------------------------
# 3. Count players with any missing FIFA statistic
# ------------------------------------------------

stat_columns = [
    column
    for column in fifa_player_stats_final.columns
    if column not in [
        "PlayerID",
        "PlayerName",
        "Nation",
        "Position"
    ]
]

players_with_missing = (
    fifa_player_stats_final[stat_columns]
    .isna()
    .any(axis=1)
    .sum()
)

players_complete = len(fifa_player_stats_final) - players_with_missing

print("\n" + "=" * 70)
print("PLAYER-LEVEL COMPLETENESS")
print("=" * 70)

print(
    f"Players with at least one missing statistic: "
    f"{players_with_missing}"
)

print(
    f"Players with no missing statistics: "
    f"{players_complete}"
)

FIFA MISSING VALUE ANALYSIS
Total rows: 1242
Columns with missing values: 15
Total missing cells: 8202

Missing values by column:


,Column,MissingCount,MissingPct
0,GoalkeeperSaves,1097,88.33
1,GoalkeeperActionsOutsidethePenaltyArea,1097,88.33
2,GoalkeeperActionsInsidethePenaltyArea,1097,88.33
3,AttemptsInsidethePenaltyArea,542,43.64
4,AttemptsAtGoalConvRatePct,542,43.64
5,AttemptsAtGoal,542,43.64
6,AttemptsOnTarget,542,43.64
7,xGEfficiency,542,43.64
8,xG,542,43.64
9,HeadedAttemptsatGoal,542,43.64



MISSING VALUES BY STATISTICAL GROUP
Goals / Assists / Minutes: 33 missing cells
Attacking: 4,878 missing cells
Distribution: 0 missing cells
Defending: 0 missing cells
Discipline: 0 missing cells
Goalkeeping: 3,291 missing cells

PLAYER-LEVEL COMPLETENESS
Players with at least one missing statistic: 1178
Players with no missing statistics: 64


## Step 56 — Handle FIFA Missing Values

FIFA missing values have different meanings depending on the statistic.

We will safely replace missing goal/assist/minute values with zero and
replace missing goalkeeper statistics with zero because goalkeeper
statistics are not applicable to the majority of outfield players.

Attacking statistics will remain missing for now because only 700
complete attacking player records were successfully extracted.

No other missing values will be changed in this step.

In [96]:
# ================================================================
# STEP 56 — HANDLE SAFE FIFA MISSING VALUES
# ================================================================

# Make a backup before modifying values
fifa_player_stats_handled = fifa_player_stats_final.copy()

# ------------------------------------------------
# 1. Statistics where missing means zero
# ------------------------------------------------

zero_fill_columns = [
    "Goals",
    "Assists",
    "MinutesPlayed",

    # Goalkeeping
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea"
]

# Fill only these columns
for column in zero_fill_columns:

    if column in fifa_player_stats_handled.columns:
        fifa_player_stats_handled[column] = (
            fifa_player_stats_handled[column].fillna(0)
        )

# ------------------------------------------------
# 2. Validate remaining missing values
# ------------------------------------------------

remaining_missing = pd.DataFrame({
    "Column": fifa_player_stats_handled.columns,
    "MissingCount": fifa_player_stats_handled.isna().sum(),
    "MissingPct": (
        fifa_player_stats_handled.isna().mean() * 100
    ).round(2)
})

remaining_missing = (
    remaining_missing[
        remaining_missing["MissingCount"] > 0
    ]
    .sort_values(
        "MissingCount",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("FIFA MISSING VALUE HANDLING")
print("=" * 70)

print(
    f"Missing cells before: "
    f"{fifa_player_stats_final.isna().sum().sum():,}"
)

print(
    f"Missing cells after: "
    f"{fifa_player_stats_handled.isna().sum().sum():,}"
)

print("\nRemaining missing values:")
display(remaining_missing)

# ------------------------------------------------
# 3. Verify zero-filled columns
# ------------------------------------------------

print("\n" + "=" * 70)
print("ZERO-FILLED COLUMN VALIDATION")
print("=" * 70)

for column in zero_fill_columns:

    if column in fifa_player_stats_handled.columns:

        print(
            f"{column}: "
            f"{fifa_player_stats_handled[column].isna().sum()} missing"
        )

# ------------------------------------------------
# 4. Confirm dataset integrity
# ------------------------------------------------

print("\n" + "=" * 70)
print("DATASET INTEGRITY")
print("=" * 70)

print(f"Rows: {len(fifa_player_stats_handled)}")
print(f"Columns: {len(fifa_player_stats_handled.columns)}")
print(
    f"Unique PlayerIDs: "
    f"{fifa_player_stats_handled['PlayerID'].nunique()}"
)
print(
    f"Duplicate PlayerIDs: "
    f"{fifa_player_stats_handled['PlayerID'].duplicated().sum()}"
)

FIFA MISSING VALUE HANDLING
Missing cells before: 8,202
Missing cells after: 4,878

Remaining missing values:


,Column,MissingCount,MissingPct
0,AttemptsOnTarget,542,43.64
1,AttemptsAtGoal,542,43.64
2,AttemptsAtGoalConvRatePct,542,43.64
3,AttemptsInsidethePenaltyArea,542,43.64
4,AttemptsOutsidethePenaltyArea,542,43.64
5,HeadedAttemptsatGoal,542,43.64
6,xG,542,43.64
7,xGEfficiency,542,43.64
8,Corners,542,43.64



ZERO-FILLED COLUMN VALIDATION
Goals: 0 missing
Assists: 0 missing
MinutesPlayed: 0 missing
GoalkeeperSaves: 0 missing
GoalkeeperActionsInsidethePenaltyArea: 0 missing
GoalkeeperActionsOutsidethePenaltyArea: 0 missing

DATASET INTEGRITY
Rows: 1242
Columns: 37
Unique PlayerIDs: 1242
Duplicate PlayerIDs: 0


## Step 57 — Prepare Local Player Statistics

The existing player_stats_df contains player information, goal events,
and physical statistics.

Before merging it with the FIFA statistics, goal-related fields will
be renamed with a `Local_` prefix.

This prevents conflicts with the FIFA Goals, Assists, and OwnGoals
fields and allows us to compare the two data sources later.

In [97]:
# ================================================================
# STEP 57 — PREPARE LOCAL PLAYER STATISTICS
# ================================================================

# Create a copy
local_player_stats = player_stats_df.copy()

# ------------------------------------------------
# Rename local goal statistics
# ------------------------------------------------

local_goal_rename = {
    "Goals": "Local_Goals",
    "RegularGoals": "Local_RegularGoals",
    "PenaltyGoals": "Local_PenaltyGoals",
    "OwnGoals": "Local_OwnGoals",
    "TotalGoalEvents": "Local_TotalGoalEvents"
}

local_player_stats = local_player_stats.rename(
    columns=local_goal_rename
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("LOCAL PLAYER STATISTICS PREPARATION")
print("=" * 70)

print(f"Rows: {len(local_player_stats)}")
print(f"Columns: {len(local_player_stats.columns)}")

print(
    f"Unique PlayerIDs: "
    f"{local_player_stats['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{local_player_stats['PlayerID'].duplicated().sum()}"
)

print("\nGoal columns:")

goal_columns = [
    column
    for column in local_player_stats.columns
    if "Goal" in column
]

print(goal_columns)

# ------------------------------------------------
# Check that physical statistics are still present
# ------------------------------------------------

print("\nPhysical statistics present:")

physical_columns = [
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning",
    "TopSpeedKmh"
]

for column in physical_columns:

    print(
        f"{column}: "
        f"{'YES' if column in local_player_stats.columns else 'NO'}"
    )

# ------------------------------------------------
# Preview
# ------------------------------------------------

print("\nFirst 5 rows:")
display(local_player_stats.head())

LOCAL PLAYER STATISTICS PREPARATION
Rows: 1248
Columns: 30
Unique PlayerIDs: 1248
Duplicate PlayerIDs: 0

Goal columns:
['Local_Goals', 'Local_RegularGoals', 'Local_PenaltyGoals', 'Local_OwnGoals', 'Local_TotalGoalEvents']

Physical statistics present:
Matches: YES
DistanceMeters: YES
Sprints: YES
HighSpeedRunning: YES
TopSpeedKmh: YES

First 5 rows:


,PlayerID,PlayerName,TeamID,Position,JerseyNumber,Local_Goals,Local_RegularGoals,Local_PenaltyGoals,Local_OwnGoals,Local_TotalGoalEvents,...,LineBreaksAttempted,LineBreaksCompleted,LineBreaksAccuracyPct,Through,Around,Over,Pass,Crosses,Prog,TotalCrosses
0,1,Melvin Mastil,25,GK,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Aissa Mandi,25,DF,2,0,0,0,0,0,...,82.0,66.0,80.49,16.0,43.0,23.0,82.0,0.0,0.0,0.0
2,3,Achref Abada,25,DF,3,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Mohamed Amine Tougai,25,DF,4,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Zineddine Belaid,25,DF,5,0,0,0,0,0,...,3.0,2.0,66.67,1.0,0.0,2.0,3.0,0.0,0.0,0.0


## Step 58 — Build Master Player Statistics Table

The local player statistics and FIFA player statistics will now be
combined using PlayerID.

The local player dataset is the master table, so all 1,248 players
from Players.csv will be retained.

FIFA statistics will be added where available.

Players without FIFA statistics will remain in the dataset with
missing FIFA values.

In [98]:
# ================================================================
# STEP 58 — BUILD MASTER PLAYER STATISTICS TABLE
# ================================================================

# ------------------------------------------------
# 1. Select FIFA statistics only
# ------------------------------------------------

fifa_statistics_only = fifa_player_stats_handled[
    [
        column
        for column in fifa_player_stats_handled.columns
        if column not in [
            "PlayerName",
            "Nation",
            "Position"
        ]
    ]
].copy()

# ------------------------------------------------
# 2. Merge local player statistics with FIFA data
# ------------------------------------------------

master_player_stats_df = local_player_stats.merge(
    fifa_statistics_only,
    on="PlayerID",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------
# 3. Validation
# ------------------------------------------------

print("=" * 70)
print("MASTER PLAYER STATISTICS")
print("=" * 70)

print(f"Rows: {len(master_player_stats_df)}")
print(f"Columns: {len(master_player_stats_df.columns)}")

print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_df['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{master_player_stats_df['PlayerID'].duplicated().sum()}"
)

# ------------------------------------------------
# 4. Check FIFA coverage
# ------------------------------------------------

fifa_goals_available = (
    master_player_stats_df["Goals"].notna().sum()
)

fifa_missing = (
    master_player_stats_df["Goals"].isna().sum()
)

print("\n" + "=" * 70)
print("FIFA COVERAGE")
print("=" * 70)

print(
    f"Players with FIFA statistics: "
    f"{fifa_goals_available}"
)

print(
    f"Players without FIFA statistics: "
    f"{fifa_missing}"
)

print(
    f"FIFA coverage: "
    f"{(fifa_goals_available / len(master_player_stats_df) * 100):.2f}%"
)

# ------------------------------------------------
# 5. Check local statistics
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOCAL DATA COVERAGE")
print("=" * 70)

print(
    f"Players with local statistics: "
    f"{master_player_stats_df['PlayerID'].notna().sum()}"
)

# ------------------------------------------------
# 6. Show final shape and columns
# ------------------------------------------------

print("\n" + "=" * 70)
print("MASTER DATASET COLUMNS")
print("=" * 70)

for i, column in enumerate(
    master_player_stats_df.columns,
    start=1
):
    print(f"{i:2}. {column}")

# ------------------------------------------------
# 7. Preview
# ------------------------------------------------

print("\nFirst 10 rows:")
display(master_player_stats_df.head(10))

MASTER PLAYER STATISTICS
Rows: 1248
Columns: 63
Unique PlayerIDs: 1248
Duplicate PlayerIDs: 0

FIFA COVERAGE
Players with FIFA statistics: 1242
Players without FIFA statistics: 6
FIFA coverage: 99.52%

LOCAL DATA COVERAGE
Players with local statistics: 1248

MASTER DATASET COLUMNS
 1. PlayerID
 2. PlayerName
 3. TeamID
 4. Position
 5. JerseyNumber
 6. Local_Goals
 7. Local_RegularGoals
 8. Local_PenaltyGoals
 9. Local_OwnGoals
10. Local_TotalGoalEvents
11. Matches
12. DistanceMeters
13. Sprints
14. HighSpeedRunning
15. TopSpeedKmh
16. SpeedZone1
17. SpeedZone2
18. SpeedZone3
19. SpeedZone4
20. SpeedZone5
21. LineBreaksAttempted
22. LineBreaksCompleted
23. LineBreaksAccuracyPct
24. Through
25. Around
26. Over
27. Pass
28. Crosses_x
29. Prog
30. TotalCrosses
31. Goals
32. Assists
33. MinutesPlayed
34. AttemptsOnTarget
35. AttemptsAtGoal
36. AttemptsAtGoalConvRatePct
37. AttemptsInsidethePenaltyArea
38. AttemptsOutsidethePenaltyArea
39. HeadedAttemptsatGoal
40. xG
41. xGEfficiency
42. Co

,PlayerID,PlayerName,TeamID,Position,JerseyNumber,Local_Goals,Local_RegularGoals,Local_PenaltyGoals,Local_OwnGoals,Local_TotalGoalEvents,...,DefensivePressuresDirectlyApplied,FoulsAgainst,FoulsFor,YellowCards,RedCards,IndirectRedCards,Offsides,GoalkeeperSaves,GoalkeeperActionsInsidethePenaltyArea,GoalkeeperActionsOutsidethePenaltyArea
0,1,Melvin Mastil,25,GK,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Aissa Mandi,25,DF,2,0,0,0,0,0,...,6,0,5,0,0,0,0,0,0,0
2,3,Achref Abada,25,DF,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Mohamed Amine Tougai,25,DF,4,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Zineddine Belaid,25,DF,5,0,0,0,0,0,...,2,0,0,0,0,0,1,0,0,0
5,6,Ramiz Zerrouki,25,MF,6,0,0,0,0,0,...,5,4,0,1,0,0,0,0,0,0
6,7,Riyad Mahrez,25,FW,7,2,2,0,0,2,...,13,1,7,0,0,0,1,0,0,0
7,8,Houssem Aouar,25,MF,8,0,0,0,0,0,...,7,1,2,0,0,0,0,0,0,0
8,9,Amine Gouiri,25,FW,9,1,1,0,0,1,...,12,1,1,0,0,0,0,0,0,0
9,10,Fares Chaibi,25,MF,10,0,0,0,0,0,...,16,3,2,1,0,0,1,0,0,0


## Step 59 — Validate FIFA Goals and Own Goals

FIFA Goals and local attacking goals have already been confirmed
to match exactly.

The FIFA OwnGoals field is stored as text, so it will be converted
to numeric before comparing it with the local own-goal calculation.

In [100]:
# ================================================================
# STEP 59 — CORRECTED GOAL VALIDATION
# ================================================================

# ------------------------------------------------
# 1. Convert FIFA goal fields to numeric
# ------------------------------------------------

master_player_stats_df["Goals"] = pd.to_numeric(
    master_player_stats_df["Goals"],
    errors="coerce"
)

master_player_stats_df["OwnGoals"] = pd.to_numeric(
    master_player_stats_df["OwnGoals"],
    errors="coerce"
)

# ------------------------------------------------
# 2. FIFA vs Local attacking goals
# ------------------------------------------------

goal_validation_df = master_player_stats_df[
    master_player_stats_df["Goals"].notna()
][
    [
        "PlayerID",
        "PlayerName",
        "TeamID",
        "Local_Goals",
        "Local_RegularGoals",
        "Local_PenaltyGoals",
        "Local_OwnGoals",
        "Goals",
        "OwnGoals"
    ]
].copy()

goal_validation_df["GoalDifference"] = (
    goal_validation_df["Goals"]
    - goal_validation_df["Local_Goals"]
)

goal_discrepancies = goal_validation_df[
    goal_validation_df["GoalDifference"] != 0
].copy()

# ------------------------------------------------
# 3. Goal validation
# ------------------------------------------------

print("=" * 70)
print("FIFA GOALS VS LOCAL GOALS")
print("=" * 70)

print(
    f"Players compared: "
    f"{len(goal_validation_df)}"
)

print(
    f"Players with matching goals: "
    f"{len(goal_validation_df) - len(goal_discrepancies)}"
)

print(
    f"Players with different goals: "
    f"{len(goal_discrepancies)}"
)

print(
    f"Total local attacking goals: "
    f"{goal_validation_df['Local_Goals'].sum():.0f}"
)

print(
    f"Total FIFA goals: "
    f"{goal_validation_df['Goals'].sum():.0f}"
)

# ------------------------------------------------
# 4. Display discrepancies
# ------------------------------------------------

print("\n" + "=" * 70)
print("GOAL DISCREPANCIES")
print("=" * 70)

if goal_discrepancies.empty:
    print("NONE — FIFA and local attacking goals match!")
else:
    display(
        goal_discrepancies.sort_values(
            "GoalDifference",
            key=abs,
            ascending=False
        )
    )

# ------------------------------------------------
# 5. Own goal validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("OWN GOAL VALIDATION")
print("=" * 70)

local_own_goals = (
    master_player_stats_df["Local_OwnGoals"]
    .sum()
)

fifa_own_goals = (
    master_player_stats_df["OwnGoals"]
    .sum()
)

print(f"Local own goals: {local_own_goals:.0f}")
print(f"FIFA own goals: {fifa_own_goals:.0f}")

print(
    f"Own goal difference: "
    f"{fifa_own_goals - local_own_goals:.0f}"
)

# ------------------------------------------------
# 6. Data types
# ------------------------------------------------

print("\n" + "=" * 70)
print("GOAL FIELD DATA TYPES")
print("=" * 70)

print(
    master_player_stats_df[
        [
            "Local_Goals",
            "Local_RegularGoals",
            "Local_PenaltyGoals",
            "Local_OwnGoals",
            "Goals",
            "OwnGoals"
        ]
    ].dtypes
)

FIFA GOALS VS LOCAL GOALS
Players compared: 1242
Players with matching goals: 1242
Players with different goals: 0
Total local attacking goals: 294
Total FIFA goals: 294

GOAL DISCREPANCIES
NONE — FIFA and local attacking goals match!

OWN GOAL VALIDATION
Local own goals: 14
FIFA own goals: 14
Own goal difference: 0

GOAL FIELD DATA TYPES
Local_Goals             int64
Local_RegularGoals      int64
Local_PenaltyGoals      int64
Local_OwnGoals          int64
Goals                 float64
OwnGoals              float64
dtype: object


## Step 60 — Standardize Master Dataset Data Types

The master player dataset combines local and FIFA statistics.

We will standardize numeric fields so that:

- Count statistics use integer-compatible types
- Percentages and xG statistics use float types
- PlayerID remains an integer
- No unexpected text values remain in statistical fields
- Infinite numeric values are checked

No statistical values will be modified.

In [101]:
# ================================================================
# STEP 60 — STANDARDIZE MASTER DATA TYPES
# ================================================================

# Work from the validated master dataset
master_player_stats_clean = master_player_stats_df.copy()

# ------------------------------------------------
# 1. Integer / count columns
# ------------------------------------------------

integer_columns = [
    "PlayerID",
    "JerseyNumber",

    # Local goals
    "Local_Goals",
    "Local_RegularGoals",
    "Local_PenaltyGoals",
    "Local_OwnGoals",
    "Local_TotalGoalEvents",

    # FIFA goals
    "Goals",
    "Assists",
    "MinutesPlayed",

    # FIFA attacking
    "AttemptsOnTarget",
    "AttemptsAtGoal",
    "AttemptsInsidethePenaltyArea",
    "AttemptsOutsidethePenaltyArea",
    "HeadedAttemptsatGoal",
    "Corners",

    # FIFA distribution
    "Passes",
    "Crosses",
    "DefensiveLinebreaksAttempted",
    "SwitchesofPlayAttempted",

    # FIFA defending
    "OwnGoals",
    "ForcedTurnovers",
    "DefensivePressuresApplied",
    "DefensivePressuresDirectlyApplied",

    # FIFA discipline
    "FoulsAgainst",
    "FoulsFor",
    "YellowCards",
    "RedCards",
    "IndirectRedCards",
    "Offsides",

    # FIFA goalkeeping
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea",

    # Local physical statistics
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning"
]

# ------------------------------------------------
# 2. Convert integer columns
# ------------------------------------------------

for column in integer_columns:

    if column in master_player_stats_clean.columns:

        master_player_stats_clean[column] = pd.to_numeric(
            master_player_stats_clean[column],
            errors="coerce"
        ).round().astype("Int64")

# ------------------------------------------------
# 3. Float columns
# ------------------------------------------------

float_columns = [
    "TopSpeedKmh",
    "SpeedZone1",
    "SpeedZone2",
    "SpeedZone3",
    "SpeedZone4",
    "SpeedZone5",
    "LineBreaksAccuracyPct",

    # FIFA attacking
    "AttemptsAtGoalConvRatePct",
    "xG",
    "xGEfficiency",

    # FIFA distribution
    "PassingAccuracyPct",
    "CrossingAccuracyPct",
    "DefensiveLinebreaksAccPct",
    "SwitchesofPlayAccPct"
]

for column in float_columns:

    if column in master_player_stats_clean.columns:

        master_player_stats_clean[column] = pd.to_numeric(
            master_player_stats_clean[column],
            errors="coerce"
        )

# ------------------------------------------------
# 4. Check for infinite values
# ------------------------------------------------

numeric_columns = master_player_stats_clean.select_dtypes(
    include="number"
).columns

infinite_values = (
    master_player_stats_clean[numeric_columns]
    .isin([float("inf"), float("-inf")])
    .sum()
    .sum()
)

# ------------------------------------------------
# 5. Validate data types
# ------------------------------------------------

print("=" * 70)
print("MASTER DATA TYPE VALIDATION")
print("=" * 70)

print(f"Rows: {len(master_player_stats_clean)}")
print(f"Columns: {len(master_player_stats_clean.columns)}")

print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_clean['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{master_player_stats_clean['PlayerID'].duplicated().sum()}"
)

print(f"Infinite numeric values: {infinite_values}")

# ------------------------------------------------
# 6. Show selected data types
# ------------------------------------------------

print("\nSelected data types:")

display(
    master_player_stats_clean[
        [
            "PlayerID",
            "Local_Goals",
            "Local_OwnGoals",
            "Goals",
            "Assists",
            "MinutesPlayed",
            "xG",
            "PassingAccuracyPct",
            "OwnGoals",
            "GoalkeeperSaves"
        ]
    ].dtypes.to_frame("DataType")
)

# ------------------------------------------------
# 7. Check remaining missing values
# ------------------------------------------------

remaining_missing = (
    master_player_stats_clean.isna().sum()
)

remaining_missing = (
    remaining_missing[
        remaining_missing > 0
    ]
    .sort_values(ascending=False)
)

print("\nRemaining missing values:")

if remaining_missing.empty:
    print("NONE")
else:
    display(
        remaining_missing.to_frame("MissingCount")
    )

MASTER DATA TYPE VALIDATION
Rows: 1248
Columns: 63
Unique PlayerIDs: 1248
Duplicate PlayerIDs: 0
Infinite numeric values: 0

Selected data types:


,DataType
PlayerID,Int64
Local_Goals,Int64
Local_OwnGoals,Int64
Goals,Int64
Assists,Int64
MinutesPlayed,Int64
xG,float64
PassingAccuracyPct,float64
OwnGoals,Int64
GoalkeeperSaves,Int64



Remaining missing values:


,MissingCount
xGEfficiency,1248
AttemptsOnTarget,548
AttemptsInsidethePenaltyArea,548
AttemptsAtGoal,548
AttemptsOutsidethePenaltyArea,548
AttemptsAtGoalConvRatePct,548
xG,548
HeadedAttemptsatGoal,548
Corners,548
SpeedZone5,219


## Step 61 — Resolve Duplicate Statistic Names

The master merge contains a few automatically generated `_x` and `_y`
columns because the local physical statistics and FIFA statistics
share names such as `Crosses` and `PassingAccuracyPct`.

We will explicitly identify these columns and rename them according to
their data source.

Local physical statistics receive the `Local_` prefix where needed,
while FIFA statistics keep their clean names.

No statistical values will be changed or dropped.

In [102]:
# ================================================================
# STEP 61 — RESOLVE DUPLICATE STATISTIC NAMES
# ================================================================

# Work from the cleaned master dataset
master_player_stats_final = master_player_stats_clean.copy()

# ------------------------------------------------
# 1. Show columns containing _x or _y
# ------------------------------------------------

duplicate_style_columns = [
    column
    for column in master_player_stats_final.columns
    if column.endswith("_x") or column.endswith("_y")
]

print("=" * 70)
print("DUPLICATE-STYLE COLUMNS FOUND")
print("=" * 70)

print(duplicate_style_columns)

# ------------------------------------------------
# 2. Resolve known local/FIFA collisions
# ------------------------------------------------

rename_duplicates = {
    "Crosses_x": "Local_Crosses",
    "Crosses_y": "Crosses"
}

# Only rename columns that actually exist
rename_duplicates = {
    old: new
    for old, new in rename_duplicates.items()
    if old in master_player_stats_final.columns
}

master_player_stats_final = master_player_stats_final.rename(
    columns=rename_duplicates
)

# ------------------------------------------------
# 3. Check remaining _x / _y columns
# ------------------------------------------------

remaining_duplicate_style_columns = [
    column
    for column in master_player_stats_final.columns
    if column.endswith("_x") or column.endswith("_y")
]

# ------------------------------------------------
# 4. Check duplicate column names
# ------------------------------------------------

duplicate_columns = (
    master_player_stats_final.columns[
        master_player_stats_final.columns.duplicated()
    ]
    .tolist()
)

# ------------------------------------------------
# 5. Validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE COLUMN RESOLUTION")
print("=" * 70)

print("Renamed columns:")
print(rename_duplicates)

print("\nRemaining _x / _y columns:")

if remaining_duplicate_style_columns:
    print(remaining_duplicate_style_columns)
else:
    print("NONE")

print("\nDuplicate column names:")

if duplicate_columns:
    print(duplicate_columns)
else:
    print("NONE")

# ------------------------------------------------
# 6. Check Crosses specifically
# ------------------------------------------------

print("\n" + "=" * 70)
print("CROSSES VALIDATION")
print("=" * 70)

cross_columns = [
    column
    for column in master_player_stats_final.columns
    if "Crosses" in column
]

print(cross_columns)

# ------------------------------------------------
# 7. Dataset integrity
# ------------------------------------------------

print("\n" + "=" * 70)
print("MASTER DATASET INTEGRITY")
print("=" * 70)

print(f"Rows: {len(master_player_stats_final)}")
print(f"Columns: {len(master_player_stats_final.columns)}")

print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].nunique()}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].duplicated().sum()}"
)

print(
    f"Total missing cells: "
    f"{master_player_stats_final.isna().sum().sum():,}"
)

DUPLICATE-STYLE COLUMNS FOUND
['Crosses_x', 'Crosses_y']

DUPLICATE COLUMN RESOLUTION
Renamed columns:
{'Crosses_x': 'Local_Crosses', 'Crosses_y': 'Crosses'}

Remaining _x / _y columns:
NONE

Duplicate column names:
NONE

CROSSES VALIDATION
['Local_Crosses', 'TotalCrosses', 'Crosses']

MASTER DATASET INTEGRITY
Rows: 1248
Columns: 63
Unique PlayerIDs: 1248
Duplicate PlayerIDs: 0
Total missing cells: 10,156


## Step 62 — Final Missing-Value Audit

Before preparing the final master dataset, we will audit every
remaining missing-value field.

The purpose is to distinguish:

- genuine zero values that should be filled,
- unavailable statistics that should remain NaN,
- players without FIFA statistics,
- players without local physical statistics.

No values will be changed in this step.

In [103]:
# ================================================================
# STEP 62 — FINAL MISSING-VALUE AUDIT
# ================================================================

missing_audit = (
    master_player_stats_final
    .isna()
    .sum()
    .reset_index()
)

missing_audit.columns = ["Column", "MissingCount"]

missing_audit["MissingPct"] = (
    missing_audit["MissingCount"]
    / len(master_player_stats_final)
    * 100
)

missing_audit = (
    missing_audit[missing_audit["MissingCount"] > 0]
    .sort_values("MissingCount", ascending=False)
    .reset_index(drop=True)
)

print("=" * 70)
print("FINAL MISSING-VALUE AUDIT")
print("=" * 70)

if len(missing_audit) == 0:
    print("No missing values found.")
else:
    print(missing_audit.to_string(index=False))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Total missing cells: {master_player_stats_final.isna().sum().sum():,}")
print(f"Columns containing missing values: {len(missing_audit)}")
print(
    f"Players with at least one missing value: "
    f"{master_player_stats_final.isna().any(axis=1).sum()}"
)

# ------------------------------------------------
# Missing values by major data source
# ------------------------------------------------

local_columns = [
    column for column in master_player_stats_final.columns
    if column.startswith("Local_")
    or column in [
        "PlayerID",
        "PlayerName",
        "TeamID",
        "Position",
        "JerseyNumber",
        "Matches",
        "DistanceMeters",
        "Sprints",
        "HighSpeedRunning",
        "TopSpeedKmh",
        "SpeedZone1",
        "SpeedZone2",
        "SpeedZone3",
        "SpeedZone4",
        "SpeedZone5",
        "LineBreaksAttempted",
        "LineBreaksCompleted",
        "LineBreaksAccuracyPct",
        "Through",
        "Around",
        "Over",
        "Pass",
        "Local_Crosses",
        "Prog",
        "TotalCrosses",
    ]
    and column in master_player_stats_final.columns
]

fifa_columns = [
    column for column in master_player_stats_final.columns
    if column not in local_columns
]

print("\n" + "=" * 70)
print("MISSING VALUES BY SOURCE")
print("=" * 70)

print(
    f"Local/statistics-side missing cells: "
    f"{master_player_stats_final[local_columns].isna().sum().sum():,}"
)

print(
    f"FIFA-side missing cells: "
    f"{master_player_stats_final[fifa_columns].isna().sum().sum():,}"
)

FINAL MISSING-VALUE AUDIT
                                Column  MissingCount  MissingPct
                          xGEfficiency          1248  100.000000
                      AttemptsOnTarget           548   43.910256
          AttemptsInsidethePenaltyArea           548   43.910256
                        AttemptsAtGoal           548   43.910256
         AttemptsOutsidethePenaltyArea           548   43.910256
             AttemptsAtGoalConvRatePct           548   43.910256
                                    xG           548   43.910256
                  HeadedAttemptsatGoal           548   43.910256
                               Corners           548   43.910256
                            SpeedZone5           219   17.548077
                   LineBreaksAttempted           219   17.548077
                   LineBreaksCompleted           219   17.548077
                            SpeedZone4           219   17.548077
                            SpeedZone1           219   17.548077

## Step 63 — Investigate xG Efficiency

The FIFA Attacking data originally contained an "xG Efficiency"
field, but the final master dataset shows `xGEfficiency` as 100% missing.

Before filling any missing values, we will inspect the original FIFA
Attacking columns and the corresponding master columns to identify
whether the statistic was renamed incorrectly or duplicated.

No data will be changed in this step.

In [104]:
# ================================================================
# STEP 63 — INVESTIGATE xG EFFICIENCY
# ================================================================

print("=" * 70)
print("FIFA ATTACKING COLUMNS")
print("=" * 70)

for i, column in enumerate(fifa_final["Attacking"].columns, start=1):
    print(f"{i:2}. {column}")

print("\n" + "=" * 70)
print("FIFA ATTACKING xG-RELATED COLUMNS")
print("=" * 70)

attacking_xg_columns = [
    column
    for column in fifa_final["Attacking"].columns
    if "xG" in column or "xg" in column
]

print(attacking_xg_columns)

print("\n" + "=" * 70)
print("MASTER xG-RELATED COLUMNS")
print("=" * 70)

master_xg_columns = [
    column
    for column in master_player_stats_final.columns
    if "xG" in column or "xg" in column
]

print(master_xg_columns)

print("\n" + "=" * 70)
print("xG-RELATED DATA PREVIEW")
print("=" * 70)

for column in attacking_xg_columns:
    print(f"\n--- {column} ---")
    print(fifa_final["Attacking"][column].head(10).to_list())

print("\n" + "=" * 70)
print("MASTER xG DATA")
print("=" * 70)

for column in master_xg_columns:
    print(f"\n--- {column} ---")
    print(master_player_stats_final[column].head(10).to_list())

FIFA ATTACKING COLUMNS
 1. Rank
 2. Assists
 3. Attempts On Target
 4. Attempts At Goal
 5. Attempts At Goal Conv. Rate (%)
 6. Attempts Inside the Penalty Area
 7. Attempts Outside the Penalty Area
 8. Headed Attempts at Goal
 9. xG
10. xG Efficiency
11. Corners
12. PlayerName
13. Nation
14. Position
15. NameNormalized

FIFA ATTACKING xG-RELATED COLUMNS
['xG', 'xG Efficiency']

MASTER xG-RELATED COLUMNS
['xG', 'xGEfficiency']

xG-RELATED DATA PREVIEW

--- xG ---
['2.34', '0.72', '6.54', '0.31', '1.02', '6.18', '0.28', '0.7', '0.97', '0.2']

--- xG Efficiency ---
['0x', '0x', '1.53x', '0x', '0x', '1.29x', '0x', '1.43x', '0x', '5.12x']

MASTER xG DATA

--- xG ---
[nan, nan, nan, nan, nan, nan, 1.49, 0.24, nan, nan]

--- xGEfficiency ---
[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]


## Step 64 — Restore xG Efficiency

The FIFA Attacking dataset stores xG Efficiency values with a trailing
"x", for example "1.53x" and "0x".

During numeric conversion, these values were incorrectly converted
to NaN.

We will restore the original FIFA xG Efficiency values by removing
the trailing "x" and converting them to numeric values.

The values will be matched to players using PlayerID.

No other statistics will be changed.

In [105]:
# ================================================================
# STEP 64 — RESTORE xG EFFICIENCY
# ================================================================

# ------------------------------------------------
# 1. Prepare original FIFA attacking xG Efficiency
# ------------------------------------------------

attacking_xg_efficiency = fifa_final["Attacking"][
    ["PlayerName", "Nation", "Position", "xG Efficiency"]
].copy()

# Convert values such as "1.53x" or "0x" -> 1.53 or 0
attacking_xg_efficiency["xGEfficiency"] = (
    attacking_xg_efficiency["xG Efficiency"]
    .astype(str)
    .str.strip()
    .str.replace("x", "", regex=False)
    .replace("nan", pd.NA)
)

attacking_xg_efficiency["xGEfficiency"] = pd.to_numeric(
    attacking_xg_efficiency["xGEfficiency"],
    errors="coerce"
)

# ------------------------------------------------
# 2. Attach PlayerID using the validated FIFA mapping
# ------------------------------------------------

attacking_xg_efficiency = attacking_xg_efficiency.merge(
    fifa_player_master_fixed[
        ["PlayerName", "Nation", "Position", "PlayerID"]
    ],
    on=["PlayerName", "Nation", "Position"],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------
# 3. Validate PlayerID matching
# ------------------------------------------------

print("=" * 70)
print("xG EFFICIENCY PLAYER MATCHING")
print("=" * 70)

print(f"FIFA attacking rows: {len(attacking_xg_efficiency):,}")
print(
    "Missing PlayerIDs: "
    f"{attacking_xg_efficiency['PlayerID'].isna().sum():,}"
)
print(
    "Unique PlayerIDs: "
    f"{attacking_xg_efficiency['PlayerID'].nunique():,}"
)

# ------------------------------------------------
# 4. Show original and restored values
# ------------------------------------------------

print("\n" + "=" * 70)
print("RESTORED xG EFFICIENCY SAMPLE")
print("=" * 70)

print(
    attacking_xg_efficiency[
        ["PlayerName", "Nation", "Position",
         "xG Efficiency", "xGEfficiency", "PlayerID"]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------
# 5. Update master dataset
# ------------------------------------------------

# Remove current broken xGEfficiency column
master_player_stats_final = master_player_stats_final.drop(
    columns=["xGEfficiency"]
)

# Merge restored values
master_player_stats_final = master_player_stats_final.merge(
    attacking_xg_efficiency[["PlayerID", "xGEfficiency"]],
    on="PlayerID",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------
# 6. Move xGEfficiency next to xG
# ------------------------------------------------

columns = master_player_stats_final.columns.tolist()

columns.remove("xGEfficiency")

xg_position = columns.index("xG") + 1

columns.insert(xg_position, "xGEfficiency")

master_player_stats_final = master_player_stats_final[columns]

# ------------------------------------------------
# 7. Final validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("xG EFFICIENCY RESTORATION VALIDATION")
print("=" * 70)

print(
    f"xGEfficiency missing: "
    f"{master_player_stats_final['xGEfficiency'].isna().sum():,}"
)

print(
    f"xGEfficiency non-missing: "
    f"{master_player_stats_final['xGEfficiency'].notna().sum():,}"
)

print(
    f"xGEfficiency dtype: "
    f"{master_player_stats_final['xGEfficiency'].dtype}"
)

print("\nSample restored values:")
print(
    master_player_stats_final[
        ["PlayerName", "xG", "xGEfficiency"]
    ]
    .head(15)
    .to_string(index=False)
)

xG EFFICIENCY PLAYER MATCHING
FIFA attacking rows: 700
Missing PlayerIDs: 0
Unique PlayerIDs: 700

RESTORED xG EFFICIENCY SAMPLE
         PlayerName Nation Position xG Efficiency  xGEfficiency  PlayerID
      Michael Olise    FRA       FW            0x          0.00     479.0
    Martin Odegaard    NOR       MF            0x          0.00     816.0
      Kylian Mbappe    FRA       FW         1.53x          1.53     478.0
        Brahim Diaz    MAR       FW            0x          0.00     738.0
    Bruno Guimaraes    BRA       MF            0x          0.00     164.0
       Lionel Messi    ARG       FW         1.29x          1.29      36.0
   Roberto Alvarado    MEX       FW            0x          0.00     727.0
     Anthony Gordon    ENG       FW         1.43x          1.43     460.0
      Florian Wirtz    GER       MF            0x          0.00     511.0
Andreas Schjelderup    NOR       MF         5.12x          5.12     827.0
        Bukayo Saka    ENG       FW         1.74x        

## Step 65 — Master Dataset Integrity Check

After restoring xG Efficiency, we will perform a final structural
validation of the master player dataset.

We will verify:

- row count
- column count
- unique PlayerIDs
- duplicate PlayerIDs
- duplicate column names
- xG and xG Efficiency availability
- remaining missing values

No values will be modified in this step.

In [106]:
# ================================================================
# STEP 65 — MASTER DATASET INTEGRITY CHECK
# ================================================================

print("=" * 70)
print("MASTER DATASET FINAL INTEGRITY CHECK")
print("=" * 70)

print(f"Rows: {len(master_player_stats_final):,}")
print(f"Columns: {len(master_player_stats_final.columns):,}")

print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].nunique():,}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].duplicated().sum():,}"
)

duplicate_columns = (
    master_player_stats_final.columns[
        master_player_stats_final.columns.duplicated()
    ]
    .tolist()
)

print(
    f"Duplicate column names: "
    f"{len(duplicate_columns)}"
)

if duplicate_columns:
    print(duplicate_columns)

print("\n" + "=" * 70)
print("xG VALIDATION")
print("=" * 70)

print(
    f"xG non-missing: "
    f"{master_player_stats_final['xG'].notna().sum():,}"
)

print(
    f"xG missing: "
    f"{master_player_stats_final['xG'].isna().sum():,}"
)

print(
    f"xGEfficiency non-missing: "
    f"{master_player_stats_final['xGEfficiency'].notna().sum():,}"
)

print(
    f"xGEfficiency missing: "
    f"{master_player_stats_final['xGEfficiency'].isna().sum():,}"
)

print("\n" + "=" * 70)
print("TOTAL MISSING VALUES")
print("=" * 70)

print(
    f"Total missing cells: "
    f"{master_player_stats_final.isna().sum().sum():,}"
)

print(
    f"Columns containing missing values: "
    f"{(master_player_stats_final.isna().sum() > 0).sum():,}"
)

print(
    f"Players with at least one missing value: "
    f"{master_player_stats_final.isna().any(axis=1).sum():,}"
)

MASTER DATASET FINAL INTEGRITY CHECK
Rows: 1,248
Columns: 63
Unique PlayerIDs: 1,248
Duplicate PlayerIDs: 0
Duplicate column names: 0

xG VALIDATION
xG non-missing: 700
xG missing: 548
xGEfficiency non-missing: 700
xGEfficiency missing: 548

TOTAL MISSING VALUES
Total missing cells: 9,456
Columns containing missing values: 53
Players with at least one missing value: 656


## Step 66 — Classify Missing Values

We will classify the remaining missing values before making any
changes.

There are three different situations:

1. Statistics where a missing value means zero
   - Goals
   - Assists
   - Minutes Played
   - Goalkeeping statistics

2. Statistics that are unavailable because the player does not have
   an extracted FIFA attacking record
   - xG
   - xG Efficiency
   - Attempts
   - Attacking statistics

3. Local physical statistics that are unavailable because the player
   has no physical-stat record.

Only category 1 will be converted to zero.

Categories 2 and 3 will remain NaN so that unavailable data is not
mistaken for zero performance.

In [107]:
# ================================================================
# STEP 66 — CLASSIFY MISSING VALUES
# ================================================================

df = master_player_stats_final.copy()

# ------------------------------------------------
# 1. Columns where missing means zero
# ------------------------------------------------

zero_fill_columns = [
    "Goals",
    "Assists",
    "MinutesPlayed",
    "OwnGoals",
    "FoulsAgainst",
    "FoulsFor",
    "YellowCards",
    "RedCards",
    "IndirectRedCards",
    "Offsides",
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea",
]

zero_fill_columns = [
    column
    for column in zero_fill_columns
    if column in df.columns
]

# ------------------------------------------------
# 2. FIFA attacking statistics that should remain NaN
# ------------------------------------------------

attacking_columns = [
    "AttemptsOnTarget",
    "AttemptsAtGoal",
    "AttemptsAtGoalConvRatePct",
    "AttemptsInsidethePenaltyArea",
    "AttemptsOutsidethePenaltyArea",
    "HeadedAttemptsatGoal",
    "xG",
    "xGEfficiency",
    "Corners",
]

attacking_columns = [
    column
    for column in attacking_columns
    if column in df.columns
]

# ------------------------------------------------
# 3. Local physical statistics that should remain NaN
# ------------------------------------------------

physical_columns = [
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning",
    "TopSpeedKmh",
    "SpeedZone1",
    "SpeedZone2",
    "SpeedZone3",
    "SpeedZone4",
    "SpeedZone5",
    "LineBreaksAttempted",
    "LineBreaksCompleted",
    "LineBreaksAccuracyPct",
    "Through",
    "Around",
    "Over",
    "Pass",
    "Local_Crosses",
    "Prog",
    "TotalCrosses",
]

physical_columns = [
    column
    for column in physical_columns
    if column in df.columns
]

# ------------------------------------------------
# 4. Print classification
# ------------------------------------------------

print("=" * 70)
print("MISSING-VALUE CLASSIFICATION")
print("=" * 70)

print("\nZERO-FILL COLUMNS:")
for column in zero_fill_columns:
    print(
        f"{column:<45} "
        f"Missing: {df[column].isna().sum():>4}"
    )

print("\nATTACKING COLUMNS — KEEP NaN:")
for column in attacking_columns:
    print(
        f"{column:<45} "
        f"Missing: {df[column].isna().sum():>4}"
    )

print("\nPHYSICAL COLUMNS — KEEP NaN:")
for column in physical_columns:
    print(
        f"{column:<45} "
        f"Missing: {df[column].isna().sum():>4}"
    )

# ------------------------------------------------
# 5. Calculate missing cells by classification
# ------------------------------------------------

zero_missing = df[zero_fill_columns].isna().sum().sum()
attacking_missing = df[attacking_columns].isna().sum().sum()
physical_missing = df[physical_columns].isna().sum().sum()

print("\n" + "=" * 70)
print("MISSING CELLS BY CLASSIFICATION")
print("=" * 70)

print(f"Zero-fill candidates:       {zero_missing:,}")
print(f"Attacking unavailable:      {attacking_missing:,}")
print(f"Physical unavailable:       {physical_missing:,}")

MISSING-VALUE CLASSIFICATION

ZERO-FILL COLUMNS:
Goals                                         Missing:    6
Assists                                       Missing:    6
MinutesPlayed                                 Missing:    6
OwnGoals                                      Missing:    6
FoulsAgainst                                  Missing:    6
FoulsFor                                      Missing:    6
YellowCards                                   Missing:    6
RedCards                                      Missing:    6
IndirectRedCards                              Missing:    6
Offsides                                      Missing:    6
GoalkeeperSaves                               Missing:    6
GoalkeeperActionsInsidethePenaltyArea         Missing:    6
GoalkeeperActionsOutsidethePenaltyArea        Missing:    6

ATTACKING COLUMNS — KEEP NaN:
AttemptsOnTarget                              Missing:  548
AttemptsAtGoal                                Missing:  548
AttemptsAtGoalConvRa

## Step 67 — Apply Safe Zero Filling

The missing-value audit confirmed that 14 FIFA statistics have exactly
6 missing values each, corresponding to the 6 players without FIFA
statistics.

For these fields, missing values will be replaced with zero.

All unavailable attacking statistics and unavailable physical
statistics will remain NaN.

No existing non-missing values will be changed.

In [108]:
# ================================================================
# STEP 67 — APPLY SAFE ZERO FILLING
# ================================================================

# Create final working copy
master_player_stats_final = master_player_stats_final.copy()

# ------------------------------------------------
# Columns approved for zero filling
# ------------------------------------------------

zero_fill_columns = [
    "Goals",
    "Assists",
    "MinutesPlayed",
    "OwnGoals",
    "FoulsAgainst",
    "FoulsFor",
    "YellowCards",
    "RedCards",
    "IndirectRedCards",
    "Offsides",
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea",
]

# ------------------------------------------------
# Record missing counts BEFORE filling
# ------------------------------------------------

missing_before = (
    master_player_stats_final[zero_fill_columns]
    .isna()
    .sum()
)

# ------------------------------------------------
# Fill only missing values with zero
# ------------------------------------------------

for column in zero_fill_columns:
    master_player_stats_final[column] = (
        master_player_stats_final[column]
        .fillna(0)
    )

# ------------------------------------------------
# Record missing counts AFTER filling
# ------------------------------------------------

missing_after = (
    master_player_stats_final[zero_fill_columns]
    .isna()
    .sum()
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("ZERO-FILL VALIDATION")
print("=" * 70)

print(
    f"Missing cells before: "
    f"{missing_before.sum():,}"
)

print(
    f"Missing cells after:  "
    f"{missing_after.sum():,}"
)

print("\nColumn results:")

for column in zero_fill_columns:
    print(
        f"{column:<45} "
        f"{missing_before[column]:>3} → "
        f"{missing_after[column]:>3}"
    )

# ------------------------------------------------
# Dataset integrity
# ------------------------------------------------

print("\n" + "=" * 70)
print("MASTER DATASET INTEGRITY")
print("=" * 70)

print(f"Rows: {len(master_player_stats_final):,}")
print(f"Columns: {len(master_player_stats_final.columns):,}")

print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].nunique():,}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].duplicated().sum():,}"
)

print(
    f"Duplicate column names: "
    f"{master_player_stats_final.columns.duplicated().sum():,}"
)

print(
    f"Total missing cells remaining: "
    f"{master_player_stats_final.isna().sum().sum():,}"
)

ZERO-FILL VALIDATION
Missing cells before: 78
Missing cells after:  0

Column results:
Goals                                           6 →   0
Assists                                         6 →   0
MinutesPlayed                                   6 →   0
OwnGoals                                        6 →   0
FoulsAgainst                                    6 →   0
FoulsFor                                        6 →   0
YellowCards                                     6 →   0
RedCards                                        6 →   0
IndirectRedCards                                6 →   0
Offsides                                        6 →   0
GoalkeeperSaves                                 6 →   0
GoalkeeperActionsInsidethePenaltyArea           6 →   0
GoalkeeperActionsOutsidethePenaltyArea          6 →   0

MASTER DATASET INTEGRITY
Rows: 1,248
Columns: 63
Unique PlayerIDs: 1,248
Duplicate PlayerIDs: 0
Duplicate column names: 0
Total missing cells remaining: 9,378


## Step 68 — Validate FIFA Coverage

The master dataset contains 1,248 local players, while FIFA statistics
were successfully matched for 1,242 players.

We will identify the 6 players without FIFA statistics and verify that
their FIFA fields are consistently unavailable.

This is a validation step only. No values will be changed.

In [109]:
# ================================================================
# STEP 68 — VALIDATE FIFA COVERAGE
# ================================================================

# FIFA statistic columns
fifa_stat_columns = [
    "Goals",
    "Assists",
    "MinutesPlayed",
    "AttemptsOnTarget",
    "AttemptsAtGoal",
    "AttemptsAtGoalConvRatePct",
    "AttemptsInsidethePenaltyArea",
    "AttemptsOutsidethePenaltyArea",
    "HeadedAttemptsatGoal",
    "xG",
    "xGEfficiency",
    "Corners",
    "Passes",
    "PassingAccuracyPct",
    "Crosses",
    "CrossingAccuracyPct",
    "DefensiveLinebreaksAttempted",
    "DefensiveLinebreaksAccPct",
    "SwitchesofPlayAttempted",
    "SwitchesofPlayAccPct",
    "OwnGoals",
    "ForcedTurnovers",
    "DefensivePressuresApplied",
    "DefensivePressuresDirectlyApplied",
    "FoulsAgainst",
    "FoulsFor",
    "YellowCards",
    "RedCards",
    "IndirectRedCards",
    "Offsides",
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea",
]

fifa_stat_columns = [
    column
    for column in fifa_stat_columns
    if column in master_player_stats_final.columns
]

# ------------------------------------------------
# Identify players with no FIFA statistics
# ------------------------------------------------

no_fifa_stats_mask = (
    master_player_stats_final[fifa_stat_columns]
    .isna()
    .all(axis=1)
)

no_fifa_stats = master_player_stats_final.loc[
    no_fifa_stats_mask,
    ["PlayerID", "PlayerName", "TeamID", "Position"]
].copy()

# ------------------------------------------------
# Identify players with FIFA statistics
# ------------------------------------------------

players_with_fifa = (
    ~no_fifa_stats_mask
).sum()

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("FIFA COVERAGE VALIDATION")
print("=" * 70)

print(f"Total players: {len(master_player_stats_final):,}")
print(f"Players with FIFA statistics: {players_with_fifa:,}")
print(f"Players without FIFA statistics: {len(no_fifa_stats):,}")

print("\n" + "=" * 70)
print("PLAYERS WITHOUT FIFA STATISTICS")
print("=" * 70)

print(
    no_fifa_stats.to_string(index=False)
)

# ------------------------------------------------
# Verify those players have no FIFA values
# ------------------------------------------------

remaining_fifa_values = (
    master_player_stats_final.loc[
        no_fifa_stats_mask,
        fifa_stat_columns
    ]
    .notna()
    .sum()
    .sum()
)

print("\n" + "=" * 70)
print("NO-FIFA PLAYER VALIDATION")
print("=" * 70)

print(
    f"Non-missing FIFA cells among these players: "
    f"{remaining_fifa_values}"
)

# ------------------------------------------------
# Coverage percentage
# ------------------------------------------------

coverage_pct = (
    players_with_fifa
    / len(master_player_stats_final)
    * 100
)

print(
    f"FIFA coverage: {coverage_pct:.2f}%"
)

FIFA COVERAGE VALIDATION
Total players: 1,248
Players with FIFA statistics: 1,248
Players without FIFA statistics: 0

PLAYERS WITHOUT FIFA STATISTICS
Empty DataFrame
Columns: [PlayerID, PlayerName, TeamID, Position]
Index: []

NO-FIFA PLAYER VALIDATION
Non-missing FIFA cells among these players: 0
FIFA coverage: 100.00%


## Step 69 — Correct FIFA Coverage Validation

Zero-filling selected FIFA statistics made it impossible to identify
players without FIFA data using NaN values alone.

Therefore, FIFA coverage must be determined from the validated
FIFA PlayerID mapping created earlier.

We will compare the master PlayerIDs against the official FIFA
PlayerID mapping.

No values will be changed.

In [110]:
# ================================================================
# STEP 69 — CORRECT FIFA COVERAGE VALIDATION
# ================================================================

# FIFA PlayerIDs from the validated master mapping
fifa_player_ids = set(
    fifa_player_master_fixed["PlayerID"]
    .dropna()
    .astype(int)
)

# Master PlayerIDs
master_player_ids = set(
    master_player_stats_final["PlayerID"]
    .dropna()
    .astype(int)
)

# ------------------------------------------------
# Players with FIFA statistics
# ------------------------------------------------

players_with_fifa_ids = master_player_ids.intersection(
    fifa_player_ids
)

# ------------------------------------------------
# Players without FIFA statistics
# ------------------------------------------------

players_without_fifa_ids = master_player_ids.difference(
    fifa_player_ids
)

# ------------------------------------------------
# Get player details
# ------------------------------------------------

no_fifa_stats = master_player_stats_final[
    master_player_stats_final["PlayerID"].isin(
        players_without_fifa_ids
    )
][
    ["PlayerID", "PlayerName", "TeamID", "Position"]
].copy()

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("CORRECT FIFA COVERAGE VALIDATION")
print("=" * 70)

print(
    f"Total master players: "
    f"{len(master_player_stats_final):,}"
)

print(
    f"FIFA mapping players: "
    f"{len(fifa_player_ids):,}"
)

print(
    f"Players with FIFA statistics: "
    f"{len(players_with_fifa_ids):,}"
)

print(
    f"Players without FIFA statistics: "
    f"{len(players_without_fifa_ids):,}"
)

print("\n" + "=" * 70)
print("PLAYERS WITHOUT FIFA STATISTICS")
print("=" * 70)

print(
    no_fifa_stats.to_string(index=False)
)

# ------------------------------------------------
# Coverage
# ------------------------------------------------

coverage_pct = (
    len(players_with_fifa_ids)
    / len(master_player_stats_df)
    * 100
)

print("\n" + "=" * 70)
print("FIFA COVERAGE")
print("=" * 70)

print(f"FIFA coverage: {coverage_pct:.2f}%")

# ------------------------------------------------
# Mapping integrity
# ------------------------------------------------

print("\n" + "=" * 70)
print("MAPPING INTEGRITY")
print("=" * 70)

print(
    "Master PlayerIDs not found in FIFA mapping: "
    f"{len(players_without_fifa_ids)}"
)

print(
    "FIFA mapping PlayerIDs not found in master: "
    f"{len(fifa_player_ids.difference(master_player_ids))}"
)

CORRECT FIFA COVERAGE VALIDATION
Total master players: 1,248
FIFA mapping players: 1,242
Players with FIFA statistics: 1,242
Players without FIFA statistics: 6

PLAYERS WITHOUT FIFA STATISTICS
 PlayerID        PlayerName  TeamID Position
      154       Arjan Malic      26       DF
      629     Yuto Nagatomo      20       DF
      733   Marwane Saadane      21       DF
      745        Amine Sbai      21       FW
      806       Tommy Smith      42       DF
     1232 Ruslanbek Jiyanov      48       MF

FIFA COVERAGE
FIFA coverage: 99.52%

MAPPING INTEGRITY
Master PlayerIDs not found in FIFA mapping: 6
FIFA mapping PlayerIDs not found in master: 0


## Step 70 — Final Source Validation

Before exporting the master player dataset, we will validate the
important statistics against the original source datasets.

We will verify:

- Master player count
- FIFA player coverage
- Local physical-stat coverage
- FIFA Attacking coverage
- FIFA Goalkeeping coverage
- Goals
- Own Goals
- Duplicate PlayerIDs
- Duplicate column names

No data will be modified in this step.

In [111]:
# ================================================================
# STEP 70 — FINAL SOURCE VALIDATION
# ================================================================

print("=" * 70)
print("FINAL SOURCE VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Player counts
# ------------------------------------------------

master_players = master_player_stats_final["PlayerID"].nunique()

fifa_players = fifa_player_master_fixed["PlayerID"].nunique()

local_physical_players = player_stats_df[
    "PlayerID"
].nunique()

attacking_players = fifa_final["Attacking"][
    ["PlayerName", "Nation", "Position"]
].drop_duplicates().shape[0]

goalkeeping_players = fifa_final["Goalkeeping"][
    ["PlayerName", "Nation", "Position"]
].drop_duplicates().shape[0]

print("\nPLAYER COVERAGE")
print("-" * 70)
print(f"Master players:              {master_players:,}")
print(f"FIFA mapped players:         {fifa_players:,}")
print(f"Local physical players:      {local_physical_players:,}")
print(f"FIFA Attacking players:      {attacking_players:,}")
print(f"FIFA Goalkeeping players:    {goalkeeping_players:,}")

# ------------------------------------------------
# 2. Goals validation
# ------------------------------------------------

local_attacking_goals = (
    master_player_stats_final["Local_RegularGoals"].sum()
    + master_player_stats_final["Local_PenaltyGoals"].sum()
)

fifa_goals = master_player_stats_final["Goals"].sum()

local_own_goals = master_player_stats_final[
    "Local_OwnGoals"
].sum()

fifa_own_goals = master_player_stats_final[
    "OwnGoals"
].sum()

print("\nGOAL VALIDATION")
print("-" * 70)
print(f"Local attacking goals:      {local_attacking_goals:,}")
print(f"FIFA attacking goals:       {fifa_goals:,}")
print(f"Goal difference:             {local_attacking_goals - fifa_goals:,}")

print(f"Local own goals:             {local_own_goals:,}")
print(f"FIFA own goals:              {fifa_own_goals:,}")
print(f"Own-goal difference:         {local_own_goals - fifa_own_goals:,}")

# ------------------------------------------------
# 3. Structural validation
# ------------------------------------------------

duplicate_player_ids = (
    master_player_stats_final["PlayerID"]
    .duplicated()
    .sum()
)

duplicate_columns = (
    master_player_stats_final.columns
    .duplicated()
    .sum()
)

print("\nSTRUCTURAL VALIDATION")
print("-" * 70)
print(f"Master rows:                 {len(master_player_stats_final):,}")
print(f"Master columns:              {len(master_player_stats_final.columns):,}")
print(f"Duplicate PlayerIDs:         {duplicate_player_ids:,}")
print(f"Duplicate column names:      {duplicate_columns:,}")

# ------------------------------------------------
# 4. Overall validation status
# ------------------------------------------------

all_checks_passed = (
    master_players == 1248
    and fifa_players == 1242
    and local_physical_players == 1029
    and attacking_players == 700
    and goalkeeping_players == 145
    and local_attacking_goals == fifa_goals
    and local_own_goals == fifa_own_goals
    and duplicate_player_ids == 0
    and duplicate_columns == 0
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL VALIDATION CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE VALIDATION CHECKS FAILED")

FINAL SOURCE VALIDATION

PLAYER COVERAGE
----------------------------------------------------------------------
Master players:              1,248
FIFA mapped players:         1,242
Local physical players:      1,248
FIFA Attacking players:      700
FIFA Goalkeeping players:    145

GOAL VALIDATION
----------------------------------------------------------------------
Local attacking goals:      294
FIFA attacking goals:       294
Goal difference:             0
Local own goals:             14
FIFA own goals:              14
Own-goal difference:         0

STRUCTURAL VALIDATION
----------------------------------------------------------------------
Master rows:                 1,248
Master columns:              63
Duplicate PlayerIDs:         0
Duplicate column names:      0

FINAL VALIDATION STATUS


## Step 71 — Correct Physical Statistics Source Validation

The previous validation counted PlayerIDs from the merged local player
dataset instead of the original physical-stat dataset.

Because the local merge intentionally keeps all 1,248 players, that
count was 1,248.

The original PlayerPhysicalStats dataset contains 1,029 actual
physical-stat records.

We will correct this validation without changing any data.

In [112]:
# ================================================================
# STEP 71 — CORRECT PHYSICAL SOURCE VALIDATION
# ================================================================

# Count actual physical-stat records from the original source
local_physical_players = physical_df["PlayerID"].nunique()

# Recalculate all validation values
master_players = master_player_stats_final["PlayerID"].nunique()
fifa_players = fifa_player_master_fixed["PlayerID"].nunique()

attacking_players = fifa_final["Attacking"][
    ["PlayerName", "Nation", "Position"]
].drop_duplicates().shape[0]

goalkeeping_players = fifa_final["Goalkeeping"][
    ["PlayerName", "Nation", "Position"]
].drop_duplicates().shape[0]

local_attacking_goals = (
    master_player_stats_final["Local_RegularGoals"].sum()
    + master_player_stats_final["Local_PenaltyGoals"].sum()
)

fifa_goals = master_player_stats_final["Goals"].sum()

local_own_goals = master_player_stats_final["Local_OwnGoals"].sum()
fifa_own_goals = master_player_stats_final["OwnGoals"].sum()

duplicate_player_ids = (
    master_player_stats_final["PlayerID"].duplicated().sum()
)

duplicate_columns = (
    master_player_stats_final.columns.duplicated().sum()
)

# ------------------------------------------------
# Display validation
# ------------------------------------------------

print("=" * 70)
print("CORRECTED FINAL SOURCE VALIDATION")
print("=" * 70)

print("\nPLAYER COVERAGE")
print("-" * 70)
print(f"Master players:              {master_players:,}")
print(f"FIFA mapped players:         {fifa_players:,}")
print(f"Local physical records:      {local_physical_players:,}")
print(f"FIFA Attacking players:      {attacking_players:,}")
print(f"FIFA Goalkeeping players:    {goalkeeping_players:,}")

print("\nGOAL VALIDATION")
print("-" * 70)
print(f"Local attacking goals:       {local_attacking_goals:,}")
print(f"FIFA attacking goals:        {fifa_goals:,}")
print(f"Goal difference:             {local_attacking_goals - fifa_goals:,}")

print(f"Local own goals:             {local_own_goals:,}")
print(f"FIFA own goals:              {fifa_own_goals:,}")
print(f"Own-goal difference:         {local_own_goals - fifa_own_goals:,}")

print("\nSTRUCTURAL VALIDATION")
print("-" * 70)
print(f"Master rows:                 {len(master_player_stats_final):,}")
print(f"Master columns:              {len(master_player_stats_final.columns):,}")
print(f"Duplicate PlayerIDs:         {duplicate_player_ids:,}")
print(f"Duplicate column names:      {duplicate_columns:,}")

# ------------------------------------------------
# Final status
# ------------------------------------------------

all_checks_passed = (
    master_players == 1248
    and fifa_players == 1242
    and local_physical_players == 1029
    and attacking_players == 700
    and goalkeeping_players == 145
    and local_attacking_goals == fifa_goals
    and local_own_goals == fifa_own_goals
    and duplicate_player_ids == 0
    and duplicate_columns == 0
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL VALIDATION CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE VALIDATION CHECKS FAILED")

CORRECTED FINAL SOURCE VALIDATION

PLAYER COVERAGE
----------------------------------------------------------------------
Master players:              1,248
FIFA mapped players:         1,242
Local physical records:      1,029
FIFA Attacking players:      700
FIFA Goalkeeping players:    145

GOAL VALIDATION
----------------------------------------------------------------------
Local attacking goals:       294
FIFA attacking goals:        294
Goal difference:             0
Local own goals:             14
FIFA own goals:              14
Own-goal difference:         0

STRUCTURAL VALIDATION
----------------------------------------------------------------------
Master rows:                 1,248
Master columns:              63
Duplicate PlayerIDs:         0
Duplicate column names:      0

FINAL VALIDATION STATUS
ALL VALIDATION CHECKS PASSED


## Step 72 — Export Final Master Player Statistics

The master player dataset has passed all source, mapping, structural,
goal, and coverage validations.

We will now export the validated dataset as the final master CSV.

The exported file will contain one row per PlayerID and all available
local and FIFA player statistics.

Unavailable statistics will remain blank rather than being incorrectly
converted to zero.

In [113]:
# ================================================================
# STEP 72 — EXPORT FINAL MASTER PLAYER STATISTICS
# ================================================================

import os

# ------------------------------------------------
# Output path
# ------------------------------------------------

output_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Master_Player_Stats.csv"
)

# ------------------------------------------------
# Export
# ------------------------------------------------

master_player_stats_final.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------
# Confirm file
# ------------------------------------------------

print("=" * 70)
print("FINAL MASTER CSV EXPORT")
print("=" * 70)

print(f"File created:")
print(output_path)

print(f"\nRows exported: {len(master_player_stats_final):,}")
print(f"Columns exported: {len(master_player_stats_final.columns):,}")

print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].nunique():,}"
)

print(
    f"Duplicate PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].duplicated().sum():,}"
)

print(
    f"File exists: "
    f"{os.path.exists(output_path)}"
)

if os.path.exists(output_path):
    file_size_kb = os.path.getsize(output_path) / 1024
    print(f"File size: {file_size_kb:,.2f} KB")

FINAL MASTER CSV EXPORT
File created:
..\Data\FIFA_WorldCup_2026_Master_Player_Stats.csv

Rows exported: 1,248
Columns exported: 63
Unique PlayerIDs: 1,248
Duplicate PlayerIDs: 0
File exists: True
File size: 254.59 KB


## Step 73A — Load Match Datasets

The match validation requires the `Matches.csv` and
`MatchStatistics.csv` datasets.

The previous validation cell referenced variable names that do not
exist in the current notebook, so we will load the two source files
explicitly before continuing.

No data is modified.

In [115]:
# ================================================================
# STEP 73A — LOAD MATCH DATASETS
# ================================================================

import pandas as pd
import os

# ------------------------------------------------
# File paths
# ------------------------------------------------

matches_path = os.path.join(
    DATA_DIR,
    "Matches.csv"
)

match_statistics_path = os.path.join(
    DATA_DIR,
    "MatchStatistics.csv"
)

# ------------------------------------------------
# Load datasets
# ------------------------------------------------

matches_df = pd.read_csv(matches_path)
match_statistics_df = pd.read_csv(match_statistics_path)

# ------------------------------------------------
# Basic validation
# ------------------------------------------------

print("=" * 70)
print("MATCH DATASETS LOADED")
print("=" * 70)

print("\nMatches.csv")
print(f"Rows: {len(matches_df):,}")
print(f"Columns: {len(matches_df.columns):,}")
print(f"Columns: {matches_df.columns.tolist()}")

print("\nMatchStatistics.csv")
print(f"Rows: {len(match_statistics_df):,}")
print(f"Columns: {len(match_statistics_df.columns):,}")
print(f"Columns: {match_statistics_df.columns.tolist()}")

print("\nFiles loaded successfully!")

MATCH DATASETS LOADED

Matches.csv
Rows: 104
Columns: 11
Columns: ['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result']

MatchStatistics.csv
Rows: 208
Columns: 11
Columns: ['StatisticID', 'MatchID', 'TeamID', 'Shots', 'ShotsOnTarget', 'Possession', 'Corners', 'Fouls', 'YellowCards', 'RedCards', 'Offsides']

Files loaded successfully!


## Step 73B — Validate Match & Team Statistics

We will validate the relationship between `matches_df` and
`match_statistics_df`.

Each match should have exactly two team-statistics records.

We will also verify that:

- every statistics MatchID exists in Matches,
- every match has statistics,
- MatchID + TeamID is unique,
- every statistics TeamID participated in that match,
- there are no unexpected missing values.

No data will be modified.

In [116]:
# ================================================================
# STEP 73B — VALIDATE MATCH & TEAM STATISTICS
# ================================================================

print("=" * 70)
print("MATCH & TEAM STATISTICS VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Basic dataset information
# ------------------------------------------------

print("\nMATCHES DATASET")
print("-" * 70)

print(f"Rows: {len(matches_df):,}")
print(f"Columns: {len(matches_df.columns):,}")
print(f"Unique MatchIDs: {matches_df['MatchID'].nunique():,}")
print(
    f"Duplicate MatchIDs: "
    f"{matches_df['MatchID'].duplicated().sum():,}"
)

print("\nMATCH STATISTICS DATASET")
print("-" * 70)

print(f"Rows: {len(match_statistics_df):,}")
print(f"Columns: {len(match_statistics_df.columns):,}")
print(
    f"Unique MatchIDs: "
    f"{match_statistics_df['MatchID'].nunique():,}"
)

# ------------------------------------------------
# 2. Statistics records per match
# ------------------------------------------------

stats_per_match = (
    match_statistics_df
    .groupby("MatchID")
    .size()
)

print("\n" + "=" * 70)
print("STATISTICS RECORDS PER MATCH")
print("=" * 70)

print(
    stats_per_match
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    f"\nMatches with exactly 2 team-stat records: "
    f"{(stats_per_match == 2).sum():,}"
)

print(
    f"Matches with != 2 team-stat records: "
    f"{(stats_per_match != 2).sum():,}"
)

# ------------------------------------------------
# 3. MatchID coverage
# ------------------------------------------------

match_ids = set(matches_df["MatchID"])
stat_match_ids = set(match_statistics_df["MatchID"])

stats_without_match = stat_match_ids - match_ids
matches_without_stats = match_ids - stat_match_ids

print("\n" + "=" * 70)
print("MATCHID COVERAGE")
print("=" * 70)

print(
    f"Statistics MatchIDs not in Matches: "
    f"{len(stats_without_match):,}"
)

print(
    f"Matches without statistics: "
    f"{len(matches_without_stats):,}"
)

# ------------------------------------------------
# 4. MatchID + TeamID uniqueness
# ------------------------------------------------

duplicate_match_team = (
    match_statistics_df
    .duplicated(subset=["MatchID", "TeamID"])
    .sum()
)

print("\n" + "=" * 70)
print("MATCH + TEAM UNIQUENESS")
print("=" * 70)

print(
    f"Duplicate MatchID + TeamID records: "
    f"{duplicate_match_team:,}"
)

# ------------------------------------------------
# 5. Validate team participation
# ------------------------------------------------

matches_team_pairs = (
    set(zip(matches_df["MatchID"], matches_df["Team1ID"]))
    |
    set(zip(matches_df["MatchID"], matches_df["Team2ID"]))
)

statistics_team_pairs = set(
    zip(
        match_statistics_df["MatchID"],
        match_statistics_df["TeamID"]
    )
)

invalid_team_match_pairs = (
    statistics_team_pairs - matches_team_pairs
)

print("\n" + "=" * 70)
print("TEAM PARTICIPATION VALIDATION")
print("=" * 70)

print(
    f"Invalid MatchID + TeamID combinations: "
    f"{len(invalid_team_match_pairs):,}"
)

# ------------------------------------------------
# 6. Missing values
# ------------------------------------------------

missing_matches = matches_df.isna().sum()
missing_stats = match_statistics_df.isna().sum()

print("\n" + "=" * 70)
print("MISSING VALUES — MATCHES")
print("=" * 70)

if (missing_matches > 0).any():
    print(
        missing_matches[missing_matches > 0]
        .to_string()
    )
else:
    print("No missing values.")

print("\n" + "=" * 70)
print("MISSING VALUES — MATCH STATISTICS")
print("=" * 70)

if (missing_stats > 0).any():
    print(
        missing_stats[missing_stats > 0]
        .to_string()
    )
else:
    print("No missing values.")

# ------------------------------------------------
# 7. Final validation
# ------------------------------------------------

all_checks_passed = (
    matches_df["MatchID"].nunique() == len(matches_df)
    and
    match_statistics_df["MatchID"].nunique()
    == len(matches_df)
    and
    (stats_per_match == 2).all()
    and
    len(stats_without_match) == 0
    and
    len(matches_without_stats) == 0
    and
    duplicate_match_team == 0
    and
    len(invalid_team_match_pairs) == 0
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL MATCH & TEAM STATISTICS CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE CHECKS FAILED")

MATCH & TEAM STATISTICS VALIDATION

MATCHES DATASET
----------------------------------------------------------------------
Rows: 104
Columns: 11
Unique MatchIDs: 104
Duplicate MatchIDs: 0

MATCH STATISTICS DATASET
----------------------------------------------------------------------
Rows: 208
Columns: 11
Unique MatchIDs: 104

STATISTICS RECORDS PER MATCH
2    104

Matches with exactly 2 team-stat records: 104
Matches with != 2 team-stat records: 0

MATCHID COVERAGE
Statistics MatchIDs not in Matches: 0
Matches without statistics: 0

MATCH + TEAM UNIQUENESS
Duplicate MatchID + TeamID records: 0

TEAM PARTICIPATION VALIDATION
Invalid MatchID + TeamID combinations: 0

MISSING VALUES — MATCHES
GroupName       32
WinnerTeamID    20

MISSING VALUES — MATCH STATISTICS
No missing values.

FINAL VALIDATION STATUS
ALL MATCH & TEAM STATISTICS CHECKS PASSED


## Step 74 — Validate Goal Events Against Matches

The match and team-statistics datasets have passed validation.

We will now validate every goal event against `Matches.csv`.

Each goal must:

- reference a valid MatchID,
- have a valid TeamID,
- belong to a team participating in that match,
- have a valid PlayerID,
- use a valid goal minute representation.

Own goals are retained as goal events because they are legitimate
match events, but they are handled separately from attacking goals
in the player statistics.

In [117]:
# ================================================================
# STEP 74 — VALIDATE GOALS AGAINST MATCHES
# ================================================================

print("=" * 70)
print("GOAL EVENTS vs MATCHES VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Load / use existing Goals dataframe
# ------------------------------------------------

goals_df = Goals.copy() if "Goals" in globals() else pd.read_csv(
    os.path.join(DATA_DIR, "Goals.csv")
)

# ------------------------------------------------
# 2. Basic information
# ------------------------------------------------

print("\nGOALS DATASET")
print("-" * 70)

print(f"Rows: {len(goals_df):,}")
print(f"Columns: {len(goals_df.columns):,}")
print(f"Unique GoalIDs: {goals_df['GoalID'].nunique():,}")
print(
    f"Duplicate GoalIDs: "
    f"{goals_df['GoalID'].duplicated().sum():,}"
)

print(
    f"Unique MatchIDs: "
    f"{goals_df['MatchID'].nunique():,}"
)

print(
    f"Unique PlayerIDs: "
    f"{goals_df['PlayerID'].nunique():,}"
)

print(
    f"Unique TeamIDs: "
    f"{goals_df['TeamID'].nunique():,}"
)

# ------------------------------------------------
# 3. Validate MatchID
# ------------------------------------------------

match_ids = set(matches_df["MatchID"])

invalid_match_ids = set(
    goals_df["MatchID"]
) - match_ids

print("\n" + "=" * 70)
print("MATCHID VALIDATION")
print("=" * 70)

print(
    f"Goal events with invalid MatchID: "
    f"{len(invalid_match_ids):,}"
)

# ------------------------------------------------
# 4. Validate TeamID + MatchID
# ------------------------------------------------

matches_team_pairs = (
    set(zip(matches_df["MatchID"], matches_df["Team1ID"]))
    |
    set(zip(matches_df["MatchID"], matches_df["Team2ID"]))
)

goal_team_pairs = set(
    zip(
        goals_df["MatchID"],
        goals_df["TeamID"]
    )
)

invalid_goal_team_pairs = (
    goal_team_pairs - matches_team_pairs
)

print("\n" + "=" * 70)
print("GOAL TEAM PARTICIPATION VALIDATION")
print("=" * 70)

print(
    f"Invalid MatchID + TeamID combinations: "
    f"{len(invalid_goal_team_pairs):,}"
)

# ------------------------------------------------
# 5. Validate PlayerID
# ------------------------------------------------

player_ids = set(players_df["PlayerID"])

invalid_player_ids = set(
    goals_df["PlayerID"]
) - player_ids

print("\n" + "=" * 70)
print("PLAYERID VALIDATION")
print("=" * 70)

print(
    f"Goal events with invalid PlayerID: "
    f"{len(invalid_player_ids):,}"
)

# ------------------------------------------------
# 6. Goal types
# ------------------------------------------------

print("\n" + "=" * 70)
print("GOAL TYPE DISTRIBUTION")
print("=" * 70)

print(
    goals_df["GoalType"]
    .fillna("Regular Goal")
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 7. Goal minute validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("GOAL MINUTE VALIDATION")
print("=" * 70)

# Use existing numeric minute if available
if "GoalMinuteNumeric" in goals_df.columns:
    minute_values = pd.to_numeric(
        goals_df["GoalMinuteNumeric"],
        errors="coerce"
    )
else:
    minute_values = pd.to_numeric(
        goals_df["Minute"].astype(str).str.extract(
            r"^(\d+)"
        )[0],
        errors="coerce"
    )

invalid_minutes = (
    minute_values.isna()
    | (minute_values < 1)
    | (minute_values > 130)
)

print(
    f"Invalid goal minutes: "
    f"{invalid_minutes.sum():,}"
)

print(
    f"Minimum goal minute: "
    f"{minute_values.min()}"
)

print(
    f"Maximum goal minute: "
    f"{minute_values.max()}"
)

# ------------------------------------------------
# 8. Final validation
# ------------------------------------------------

all_checks_passed = (
    goals_df["GoalID"].nunique() == len(goals_df)
    and
    len(invalid_match_ids) == 0
    and
    len(invalid_goal_team_pairs) == 0
    and
    len(invalid_player_ids) == 0
    and
    invalid_minutes.sum() == 0
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL GOAL EVENT CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE GOAL EVENT CHECKS FAILED")

GOAL EVENTS vs MATCHES VALIDATION

GOALS DATASET
----------------------------------------------------------------------
Rows: 308
Columns: 6
Unique GoalIDs: 308
Duplicate GoalIDs: 0
Unique MatchIDs: 96
Unique PlayerIDs: 190
Unique TeamIDs: 47

MATCHID VALIDATION
Goal events with invalid MatchID: 0

GOAL TEAM PARTICIPATION VALIDATION
Invalid MatchID + TeamID combinations: 0

PLAYERID VALIDATION
Goal events with invalid PlayerID: 0

GOAL TYPE DISTRIBUTION
GoalType
Regular Goal    278
Penalty          16
Own Goal         14

GOAL MINUTE VALIDATION
Invalid goal minutes: 0
Minimum goal minute: 2
Maximum goal minute: 120

FINAL VALIDATION STATUS
ALL GOAL EVENT CHECKS PASSED


## Step 75 — Build Match-Team Fact Table

The match and team-statistics sources have passed validation.

We will now combine them into a single analytical table where each
row represents one team in one match.

This structure will make the dataset easier to analyze in Power BI.

The table will contain:

- Match information
- Team and opponent
- Team goals and opponent goals
- Match result from the team's perspective
- Team match statistics

No source values will be modified.

In [118]:
# ================================================================
# STEP 75 — BUILD MATCH-TEAM FACT TABLE
# ================================================================

# ------------------------------------------------
# 1. Create team perspective rows
# ------------------------------------------------

team1_rows = matches_df[
    [
        "MatchID",
        "MatchDate",
        "Stage",
        "GroupName",
        "StadiumID",
        "Team1ID",
        "Team2ID",
        "Team1Goals",
        "Team2Goals",
    ]
].copy()

team1_rows = team1_rows.rename(
    columns={
        "Team1ID": "TeamID",
        "Team2ID": "OpponentTeamID",
        "Team1Goals": "TeamGoals",
        "Team2Goals": "OpponentGoals",
    }
)

team1_rows["TeamResult"] = team1_rows.apply(
    lambda row:
        "Win" if row["TeamGoals"] > row["OpponentGoals"]
        else "Loss" if row["TeamGoals"] < row["OpponentGoals"]
        else "Draw",
    axis=1
)

team2_rows = matches_df[
    [
        "MatchID",
        "MatchDate",
        "Stage",
        "GroupName",
        "StadiumID",
        "Team1ID",
        "Team2ID",
        "Team1Goals",
        "Team2Goals",
    ]
].copy()

team2_rows = team2_rows.rename(
    columns={
        "Team2ID": "TeamID",
        "Team1ID": "OpponentTeamID",
        "Team2Goals": "TeamGoals",
        "Team1Goals": "OpponentGoals",
    }
)

team2_rows["TeamResult"] = team2_rows.apply(
    lambda row:
        "Win" if row["TeamGoals"] > row["OpponentGoals"]
        else "Loss" if row["TeamGoals"] < row["OpponentGoals"]
        else "Draw",
    axis=1
)

# ------------------------------------------------
# 2. Combine both perspectives
# ------------------------------------------------

match_team_base = pd.concat(
    [team1_rows, team2_rows],
    ignore_index=True
)

# ------------------------------------------------
# 3. Merge team statistics
# ------------------------------------------------

match_team_stats_df = match_team_base.merge(
    match_statistics_df,
    on=["MatchID", "TeamID"],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------
# 4. Remove unnecessary StatisticID
# ------------------------------------------------

if "StatisticID" in match_team_stats_df.columns:
    match_team_stats_df = match_team_stats_df.drop(
        columns=["StatisticID"]
    )

# ------------------------------------------------
# 5. Reorder columns
# ------------------------------------------------

desired_columns = [
    "MatchID",
    "MatchDate",
    "Stage",
    "GroupName",
    "StadiumID",
    "TeamID",
    "OpponentTeamID",
    "TeamGoals",
    "OpponentGoals",
    "TeamResult",
    "Shots",
    "ShotsOnTarget",
    "Possession",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

match_team_stats_df = match_team_stats_df[
    [
        column
        for column in desired_columns
        if column in match_team_stats_df.columns
    ]
]

# ------------------------------------------------
# 6. Validation
# ------------------------------------------------

print("=" * 70)
print("MATCH-TEAM FACT TABLE")
print("=" * 70)

print(f"Rows: {len(match_team_stats_df):,}")
print(f"Columns: {len(match_team_stats_df.columns):,}")

print(
    f"Unique MatchIDs: "
    f"{match_team_stats_df['MatchID'].nunique():,}"
)

print(
    f"Unique TeamIDs: "
    f"{match_team_stats_df['TeamID'].nunique():,}"
)

print(
    f"Duplicate MatchID + TeamID: "
    f"{match_team_stats_df.duplicated(['MatchID', 'TeamID']).sum():,}"
)

print(
    f"Missing statistic rows: "
    f"{match_team_stats_df[['Shots', 'Possession']].isna().any(axis=1).sum():,}"
)

print("\n" + "=" * 70)
print("TEAM RESULT DISTRIBUTION")
print("=" * 70)

print(
    match_team_stats_df["TeamResult"]
    .value_counts()
    .to_string()
)

print("\n" + "=" * 70)
print("SAMPLE")
print("=" * 70)

print(
    match_team_stats_df.head(10).to_string(index=False)
)

MATCH-TEAM FACT TABLE
Rows: 208
Columns: 18
Unique MatchIDs: 104
Unique TeamIDs: 48
Duplicate MatchID + TeamID: 0
Missing statistic rows: 0

TEAM RESULT DISTRIBUTION
TeamResult
Win     80
Loss    80
Draw    48

SAMPLE
 MatchID  MatchDate       Stage GroupName  StadiumID  TeamID  OpponentTeamID  TeamGoals  OpponentGoals TeamResult  Shots  ShotsOnTarget  Possession  Corners  Fouls  YellowCards  RedCards  Offsides
       1 2026-06-11 First Stage   Group A          3       8              24          2              0        Win     16              4          61        3     12            1         1         1
       2 2026-06-11 First Stage   Group A          4      31              38          2              1        Win     15              6          62        4      9            1         0         2
       3 2026-06-12 First Stage   Group B          1      16              26          1              1       Draw     13              4          61        9     10            2         0     

## Step 76 — Validate Team Goals Against Goal Events

The Match-Team Fact Table has been successfully created.

We will now compare the goals recorded in `Matches.csv` with the
individual goal events in `Goals.csv`.

For every MatchID + TeamID combination, the number of goal events
should match the team's recorded goals in the match.

Own goals are included in the match goal total because they are
legitimate match scoring events.

No data will be modified.

In [119]:
# ================================================================
# STEP 76 — VALIDATE TEAM GOALS AGAINST GOAL EVENTS
# ================================================================

# ------------------------------------------------
# 1. Count goal events by MatchID + TeamID
# ------------------------------------------------

goal_events_by_team = (
    goals_df
    .groupby(["MatchID", "TeamID"])
    .size()
    .reset_index(name="GoalEvents")
)

# ------------------------------------------------
# 2. Merge event counts with Match-Team table
# ------------------------------------------------

goal_validation_df = match_team_stats_df[
    [
        "MatchID",
        "TeamID",
        "TeamGoals",
    ]
].merge(
    goal_events_by_team,
    on=["MatchID", "TeamID"],
    how="left"
)

# Teams with no goals have zero goal events
goal_validation_df["GoalEvents"] = (
    goal_validation_df["GoalEvents"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------
# 3. Calculate difference
# ------------------------------------------------

goal_validation_df["GoalDifference"] = (
    goal_validation_df["TeamGoals"]
    - goal_validation_df["GoalEvents"]
)

# ------------------------------------------------
# 4. Validation summary
# ------------------------------------------------

print("=" * 70)
print("TEAM GOALS vs GOAL EVENTS")
print("=" * 70)

print(
    f"Team-match rows checked: "
    f"{len(goal_validation_df):,}"
)

print(
    f"Rows with matching goals: "
    f"{(goal_validation_df['GoalDifference'] == 0).sum():,}"
)

print(
    f"Rows with different goals: "
    f"{(goal_validation_df['GoalDifference'] != 0).sum():,}"
)

print(
    f"Total match goals: "
    f"{match_team_stats_df['TeamGoals'].sum():,}"
)

print(
    f"Total goal events: "
    f"{goals_df.shape[0]:,}"
)

print(
    f"Total goal difference: "
    f"{match_team_stats_df['TeamGoals'].sum() - len(goals_df):,}"
)

# ------------------------------------------------
# 5. Display mismatches if any
# ------------------------------------------------

mismatches = goal_validation_df[
    goal_validation_df["GoalDifference"] != 0
]

print("\n" + "=" * 70)
print("GOAL MISMATCHES")
print("=" * 70)

if len(mismatches) == 0:
    print("No mismatches found.")
else:
    print(
        mismatches.to_string(index=False)
    )

# ------------------------------------------------
# 6. Goal event type totals
# ------------------------------------------------

print("\n" + "=" * 70)
print("GOAL EVENT BREAKDOWN")
print("=" * 70)

goal_type_counts = (
    goals_df["GoalType"]
    .fillna("Regular Goal")
    .value_counts()
)

print(
    goal_type_counts.to_string()
)

# ------------------------------------------------
# 7. Final validation
# ------------------------------------------------

all_checks_passed = (
    len(mismatches) == 0
    and
    match_team_stats_df["TeamGoals"].sum()
    == len(goals_df)
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL TEAM GOAL VALIDATION CHECKS PASSED")
else:
    print("WARNING: GOAL VALIDATION FAILED")

TEAM GOALS vs GOAL EVENTS
Team-match rows checked: 208
Rows with matching goals: 208
Rows with different goals: 0
Total match goals: 308
Total goal events: 308
Total goal difference: 0

GOAL MISMATCHES
No mismatches found.

GOAL EVENT BREAKDOWN
GoalType
Regular Goal    278
Penalty          16
Own Goal         14

FINAL VALIDATION STATUS
ALL TEAM GOAL VALIDATION CHECKS PASSED


## Step 77 — Add Team Performance Metrics

The Match-Team Fact Table has passed goal-event validation.

We will now add standard team-performance metrics:

- Goal Difference
- Points
- Clean Sheet

These metrics are calculated from the already validated match goals
and team result.

No source values will be overwritten.

In [120]:
# ================================================================
# STEP 77 — ADD TEAM PERFORMANCE METRICS
# ================================================================

# ------------------------------------------------
# 1. Goal Difference
# ------------------------------------------------

match_team_stats_df["GoalDifference"] = (
    match_team_stats_df["TeamGoals"]
    - match_team_stats_df["OpponentGoals"]
)

# ------------------------------------------------
# 2. Points
# ------------------------------------------------

match_team_stats_df["Points"] = (
    match_team_stats_df["TeamResult"]
    .map({
        "Win": 3,
        "Draw": 1,
        "Loss": 0
    })
)

# ------------------------------------------------
# 3. Clean Sheet
# ------------------------------------------------

match_team_stats_df["CleanSheet"] = (
    match_team_stats_df["OpponentGoals"] == 0
)

# ------------------------------------------------
# 4. Validate calculated metrics
# ------------------------------------------------

print("=" * 70)
print("TEAM PERFORMANCE METRICS")
print("=" * 70)

print("\nPoints distribution:")
print(
    match_team_stats_df["Points"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nGoal Difference range:")
print(
    f"Minimum: {match_team_stats_df['GoalDifference'].min()}"
)

print(
    f"Maximum: {match_team_stats_df['GoalDifference'].max()}"
)

print("\nClean sheets:")
print(
    match_team_stats_df["CleanSheet"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 5. Validate Points
# ------------------------------------------------

expected_points = (
    match_team_stats_df["TeamResult"]
    .map({
        "Win": 3,
        "Draw": 1,
        "Loss": 0
    })
)

points_errors = (
    match_team_stats_df["Points"]
    != expected_points
).sum()

# ------------------------------------------------
# 6. Validate Goal Difference
# ------------------------------------------------

goal_difference_errors = (
    match_team_stats_df["GoalDifference"]
    != (
        match_team_stats_df["TeamGoals"]
        - match_team_stats_df["OpponentGoals"]
    )
).sum()

# ------------------------------------------------
# 7. Final validation
# ------------------------------------------------

print("\n" + "=" * 70)
print("METRIC VALIDATION")
print("=" * 70)

print(f"Goal Difference errors: {goal_difference_errors}")
print(f"Points errors:          {points_errors}")

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if points_errors == 0 and goal_difference_errors == 0:
    print("ALL TEAM PERFORMANCE METRICS PASSED")
else:
    print("WARNING: ONE OR MORE METRIC CHECKS FAILED")

TEAM PERFORMANCE METRICS

Points distribution:
Points
0    80
1    48
3    80

Goal Difference range:
Minimum: -6
Maximum: 6

Clean sheets:
CleanSheet
False    153
True      55

METRIC VALIDATION
Goal Difference errors: 0
Points errors:          0

FINAL VALIDATION STATUS
ALL TEAM PERFORMANCE METRICS PASSED


## Step 78 — Validate Match Statistics Ranges

The Match-Team Fact Table now contains calculated performance metrics.

Before exporting it, we will validate the numerical ranges of the
underlying match statistics.

This step checks for impossible or logically inconsistent values.

No data will be modified.

In [121]:
# ================================================================
# STEP 78 — VALIDATE MATCH STATISTICS RANGES
# ================================================================

print("=" * 70)
print("MATCH STATISTICS RANGE VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Possession
# ------------------------------------------------

invalid_possession = (
    (match_team_stats_df["Possession"] < 0)
    |
    (match_team_stats_df["Possession"] > 100)
).sum()

# ------------------------------------------------
# 2. Shots vs shots on target
# ------------------------------------------------

invalid_shots = (
    match_team_stats_df["ShotsOnTarget"]
    > match_team_stats_df["Shots"]
).sum()

# ------------------------------------------------
# 3. Non-negative statistics
# ------------------------------------------------

non_negative_columns = [
    "TeamGoals",
    "OpponentGoals",
    "Shots",
    "ShotsOnTarget",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

negative_counts = (
    match_team_stats_df[non_negative_columns] < 0
).sum()

total_negative_values = negative_counts.sum()

# ------------------------------------------------
# 4. Possession balance
# ------------------------------------------------

# Every match has two teams, so their possession should sum to 100.
possession_by_match = (
    match_team_stats_df
    .groupby("MatchID")["Possession"]
    .sum()
)

invalid_possession_totals = (
    (possession_by_match - 100).abs() > 0.01
).sum()

# ------------------------------------------------
# 5. Display results
# ------------------------------------------------

print("\nPOSSESSION")
print("-" * 70)

print(
    f"Invalid individual possession values: "
    f"{invalid_possession}"
)

print(
    f"Matches where possession does not sum to 100: "
    f"{invalid_possession_totals}"
)

print("\nSHOTS")
print("-" * 70)

print(
    f"Rows where ShotsOnTarget > Shots: "
    f"{invalid_shots}"
)

print("\nNEGATIVE VALUES")
print("-" * 70)

for column in non_negative_columns:
    print(
        f"{column:<25} "
        f"{negative_counts[column]}"
    )

print(
    f"\nTotal negative values: "
    f"{total_negative_values}"
)

# ------------------------------------------------
# 6. Final validation
# ------------------------------------------------

all_checks_passed = (
    invalid_possession == 0
    and
    invalid_shots == 0
    and
    total_negative_values == 0
    and
    invalid_possession_totals == 0
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL MATCH STATISTICS RANGE CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE RANGE CHECKS FAILED")

MATCH STATISTICS RANGE VALIDATION

POSSESSION
----------------------------------------------------------------------
Invalid individual possession values: 0
Matches where possession does not sum to 100: 8

SHOTS
----------------------------------------------------------------------
Rows where ShotsOnTarget > Shots: 0

NEGATIVE VALUES
----------------------------------------------------------------------
TeamGoals                 0
OpponentGoals             0
Shots                     0
ShotsOnTarget             0
Corners                   0
Fouls                     0
YellowCards               0
RedCards                  0
Offsides                  0

Total negative values: 0

FINAL VALIDATION STATUS


## Step 79 — Investigate Possession Imbalance

The match-statistics range validation found 8 matches where the two
team possession values do not sum exactly to 100%.

The individual possession values are all within the valid 0–100 range.

Before changing anything, we will inspect the affected matches and
calculate the exact possession difference.

No data will be modified in this step.

In [122]:
# ================================================================
# STEP 79 — INVESTIGATE POSSESSION IMBALANCE
# ================================================================

# ------------------------------------------------
# 1. Calculate possession total per match
# ------------------------------------------------

possession_check = (
    match_team_stats_df
    .groupby("MatchID")
    .agg(
        PossessionTotal=("Possession", "sum"),
        TeamCount=("TeamID", "count")
    )
    .reset_index()
)

possession_check["DifferenceFrom100"] = (
    possession_check["PossessionTotal"] - 100
)

# ------------------------------------------------
# 2. Get affected matches
# ------------------------------------------------

affected_match_ids = possession_check.loc[
    possession_check["DifferenceFrom100"].abs() > 0.01,
    "MatchID"
].tolist()

affected_matches = match_team_stats_df[
    match_team_stats_df["MatchID"].isin(affected_match_ids)
][
    [
        "MatchID",
        "MatchDate",
        "TeamID",
        "OpponentTeamID",
        "Possession"
    ]
].sort_values(
    ["MatchID", "TeamID"]
)

# ------------------------------------------------
# 3. Display affected matches
# ------------------------------------------------

print("=" * 70)
print("AFFECTED MATCHES")
print("=" * 70)

print(
    affected_matches.to_string(index=False)
)

# ------------------------------------------------
# 4. Display possession totals
# ------------------------------------------------

print("\n" + "=" * 70)
print("POSSESSION TOTALS")
print("=" * 70)

print(
    possession_check[
        possession_check["MatchID"].isin(affected_match_ids)
    ].to_string(index=False)
)

# ------------------------------------------------
# 5. Summary
# ------------------------------------------------

print("\n" + "=" * 70)
print("POSSESSION IMBALANCE SUMMARY")
print("=" * 70)

print(
    f"Affected matches: {len(affected_match_ids):,}"
)

print(
    f"Minimum possession total: "
    f"{possession_check['PossessionTotal'].min():.2f}"
)

print(
    f"Maximum possession total: "
    f"{possession_check['PossessionTotal'].max():.2f}"
)

print(
    f"Maximum absolute difference from 100: "
    f"{possession_check['DifferenceFrom100'].abs().max():.2f}"
)

AFFECTED MATCHES
 MatchID  MatchDate  TeamID  OpponentTeamID  Possession
       1 2026-06-11       8              24          61
       1 2026-06-11      24               8          40
       2 2026-06-11      31              38          62
       2 2026-06-11      38              31          39
      15 2026-06-15      30              42          49
      15 2026-06-15      42              30          52
      31 2026-06-19      32              47          22
      31 2026-06-19      47              32          79
      33 2026-06-20       7              17          60
      33 2026-06-20      17               7          41
      48 2026-06-23       4              27          64
      48 2026-06-23      27               4          37
      64 2026-06-26       2              42          55
      64 2026-06-26      42               2          46
      85 2026-07-02      11              25          45
      85 2026-07-02      25              11          56

POSSESSION TOTALS
 MatchID  Po

## Step 80 — Finalize Match-Team Fact Table

The possession investigation confirmed that the 8 affected matches
contain only a 1-percentage-point rounding difference.

The original FIFA possession values will be preserved.

A possession total of 100 or 101 is accepted because the values are
reported as rounded whole percentages.

All other range and consistency checks have passed.

No source values will be modified.

In [123]:
# ================================================================
# STEP 80 — FINALIZE MATCH-TEAM FACT TABLE
# ================================================================

# ------------------------------------------------
# 1. Recalculate possession totals
# ------------------------------------------------

possession_check = (
    match_team_stats_df
    .groupby("MatchID")["Possession"]
    .sum()
)

# FIFA whole-number rounding can produce 100 or 101
valid_possession_totals = possession_check.isin([100, 101])

invalid_possession_totals = (
    ~valid_possession_totals
).sum()

# ------------------------------------------------
# 2. Revalidate shots
# ------------------------------------------------

invalid_shots = (
    match_team_stats_df["ShotsOnTarget"]
    > match_team_stats_df["Shots"]
).sum()

# ------------------------------------------------
# 3. Revalidate non-negative statistics
# ------------------------------------------------

non_negative_columns = [
    "TeamGoals",
    "OpponentGoals",
    "Shots",
    "ShotsOnTarget",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

negative_values = (
    match_team_stats_df[non_negative_columns] < 0
).sum().sum()

# ------------------------------------------------
# 4. Structural validation
# ------------------------------------------------

duplicate_match_team = (
    match_team_stats_df
    .duplicated(["MatchID", "TeamID"])
    .sum()
)

missing_values = (
    match_team_stats_df.isna().sum().sum()
)

# ------------------------------------------------
# 5. Final validation
# ------------------------------------------------

all_checks_passed = (
    len(match_team_stats_df) == 208
    and
    match_team_stats_df["MatchID"].nunique() == 104
    and
    duplicate_match_team == 0
    and
    invalid_possession_totals == 0
    and
    invalid_shots == 0
    and
    negative_values == 0
    and
    missing_values == 0
)

# ------------------------------------------------
# 6. Results
# ------------------------------------------------

print("=" * 70)
print("FINAL MATCH-TEAM FACT TABLE VALIDATION")
print("=" * 70)

print(f"Rows: {len(match_team_stats_df):,}")
print(
    f"Unique MatchIDs: "
    f"{match_team_stats_df['MatchID'].nunique():,}"
)
print(
    f"Unique TeamIDs: "
    f"{match_team_stats_df['TeamID'].nunique():,}"
)
print(
    f"Duplicate MatchID + TeamID: "
    f"{duplicate_match_team:,}"
)

print("\nPOSSESSION")
print("-" * 70)

print(
    f"Matches with possession total 100: "
    f"{(possession_check == 100).sum():,}"
)

print(
    f"Matches with possession total 101: "
    f"{(possession_check == 101).sum():,}"
)

print(
    f"Invalid possession totals: "
    f"{invalid_possession_totals:,}"
)

print("\nSTATISTICS")
print("-" * 70)

print(
    f"ShotsOnTarget > Shots: "
    f"{invalid_shots:,}"
)

print(
    f"Negative statistic values: "
    f"{negative_values:,}"
)

print(
    f"Missing values: "
    f"{missing_values:,}"
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("MATCH-TEAM FACT TABLE READY FOR EXPORT")
else:
    print("WARNING: ONE OR MORE CHECKS FAILED")

FINAL MATCH-TEAM FACT TABLE VALIDATION
Rows: 208
Unique MatchIDs: 104
Unique TeamIDs: 48
Duplicate MatchID + TeamID: 0

POSSESSION
----------------------------------------------------------------------
Matches with possession total 100: 96
Matches with possession total 101: 8
Invalid possession totals: 0

STATISTICS
----------------------------------------------------------------------
ShotsOnTarget > Shots: 0
Negative statistic values: 0
Missing values: 64

FINAL VALIDATION STATUS


## Step 81 — Investigate Missing Match Statistics

The Match-Team Fact Table contains 64 missing statistic cells even
though the original `MatchStatistics.csv` passed validation with no
missing statistic rows.

Therefore, these missing values were introduced during the merge.

We will identify the affected MatchID + TeamID combinations and
compare them directly with the original MatchStatistics dataset.

No data will be modified.

In [124]:
# ================================================================
# STEP 81 — INVESTIGATE MISSING MATCH STATISTICS
# ================================================================

# ------------------------------------------------
# 1. Identify rows with missing match statistics
# ------------------------------------------------

stat_columns = [
    "Shots",
    "ShotsOnTarget",
    "Possession",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

missing_stat_rows = match_team_stats_df[
    match_team_stats_df[stat_columns]
    .isna()
    .any(axis=1)
].copy()

print("=" * 70)
print("MISSING MATCH STATISTIC ROWS")
print("=" * 70)

print(
    f"Rows with missing statistics: "
    f"{len(missing_stat_rows):,}"
)

print("\nAffected MatchID + TeamID:")

print(
    missing_stat_rows[
        [
            "MatchID",
            "TeamID",
            "OpponentTeamID",
            "TeamGoals",
            "OpponentGoals"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 2. Compare against original MatchStatistics
# ------------------------------------------------

source_pairs = set(
    zip(
        match_statistics_df["MatchID"],
        match_statistics_df["TeamID"]
    )
)

missing_pairs = set(
    zip(
        missing_stat_rows["MatchID"],
        missing_stat_rows["TeamID"]
    )
)

print("\n" + "=" * 70)
print("PAIR COMPARISON")
print("=" * 70)

print(
    f"Missing pairs found in original source: "
    f"{len(missing_pairs.intersection(source_pairs)):,}"
)

print(
    f"Missing pairs NOT found in original source: "
    f"{len(missing_pairs - source_pairs):,}"
)

# ------------------------------------------------
# 3. Show source records for affected pairs
# ------------------------------------------------

affected_source_rows = match_statistics_df[
    match_statistics_df.apply(
        lambda row:
            (row["MatchID"], row["TeamID"]) in missing_pairs,
        axis=1
    )
].copy()

print("\n" + "=" * 70)
print("ORIGINAL SOURCE RECORDS FOR AFFECTED PAIRS")
print("=" * 70)

print(
    affected_source_rows.to_string(index=False)
)

# ------------------------------------------------
# 4. Compare data types
# ------------------------------------------------

print("\n" + "=" * 70)
print("KEY COLUMN DATA TYPES")
print("=" * 70)

print(
    "match_team_stats_df MatchID:",
    match_team_stats_df["MatchID"].dtype
)

print(
    "match_team_stats_df TeamID:",
    match_team_stats_df["TeamID"].dtype
)

print(
    "match_statistics_df MatchID:",
    match_statistics_df["MatchID"].dtype
)

print(
    "match_statistics_df TeamID:",
    match_statistics_df["TeamID"].dtype
)

MISSING MATCH STATISTIC ROWS
Rows with missing statistics: 0

Affected MatchID + TeamID:
Empty DataFrame
Columns: [MatchID, TeamID, OpponentTeamID, TeamGoals, OpponentGoals]
Index: []

PAIR COMPARISON
Missing pairs found in original source: 0
Missing pairs NOT found in original source: 0

ORIGINAL SOURCE RECORDS FOR AFFECTED PAIRS
Empty DataFrame
Columns: [StatisticID, MatchID, TeamID, Shots, ShotsOnTarget, Possession, Corners, Fouls, YellowCards, RedCards, Offsides]
Index: []

KEY COLUMN DATA TYPES
match_team_stats_df MatchID: int64
match_team_stats_df TeamID: int64
match_statistics_df MatchID: int64
match_statistics_df TeamID: int64


## Step 82 — Locate Missing Values in Match-Team Fact Table

Step 81 confirmed that all 208 MatchID + TeamID combinations
have valid records in the original MatchStatistics.csv.

Therefore, the 64 missing cells in match_team_stats_df are not
missing match statistics.

This step identifies exactly which columns contain the missing
values so that we can correct the construction logic without
altering any source data.

In [125]:
# ================================================================
# STEP 82 — LOCATE ALL MISSING VALUES
# ================================================================

print("=" * 70)
print("MISSING VALUES BY COLUMN")
print("=" * 70)

missing_by_column = (
    match_team_stats_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_by_column = missing_by_column[
    missing_by_column > 0
]

if len(missing_by_column) == 0:
    print("No missing values found.")
else:
    print(missing_by_column.to_string())

print("\n" + "=" * 70)
print("TOTAL MISSING CELLS")
print("=" * 70)

print(
    f"Total missing cells: "
    f"{match_team_stats_df.isna().sum().sum():,}"
)

print("\n" + "=" * 70)
print("ROWS CONTAINING ANY MISSING VALUE")
print("=" * 70)

rows_with_missing = (
    match_team_stats_df
    .isna()
    .any(axis=1)
    .sum()
)

print(
    f"Rows containing missing values: "
    f"{rows_with_missing:,}"
)

print("\n" + "=" * 70)
print("MISSING VALUE LOCATIONS")
print("=" * 70)

if rows_with_missing > 0:
    missing_rows = match_team_stats_df[
        match_team_stats_df.isna().any(axis=1)
    ].copy()

    print(
        missing_rows[
            [
                "MatchID",
                "MatchDate",
                "Stage",
                "GroupName",
                "StadiumID",
                "TeamID",
                "OpponentTeamID",
                "TeamGoals",
                "OpponentGoals",
                "TeamResult"
            ]
        ].to_string(index=False)
    )

MISSING VALUES BY COLUMN
GroupName    64

TOTAL MISSING CELLS
Total missing cells: 64

ROWS CONTAINING ANY MISSING VALUE
Rows containing missing values: 64

MISSING VALUE LOCATIONS
 MatchID  MatchDate         Stage GroupName  StadiumID  TeamID  OpponentTeamID  TeamGoals  OpponentGoals TeamResult
      73 2026-06-28   Round of 32       NaN         11      24              16          0              1       Loss
      74 2026-06-29   Round of 32       NaN          7       7              32          1              1       Draw
      75 2026-06-29   Round of 32       NaN          5       9              21          1              1       Draw
      76 2026-06-29   Round of 32       NaN          9       3              20          2              1        Win
      77 2026-06-30   Round of 32       NaN         13       6              35          3              0        Win
      78 2026-06-30   Round of 32       NaN          8      17              22          1              2       Loss
      7

## Step 83 — Final Match-Team Validation

The missing-value investigation confirmed that the 64 missing cells
are all GroupName values for knockout-stage matches.

GroupName is intentionally nullable because knockout matches do not
belong to a tournament group.

The original source statistics are complete.

The final validation will therefore distinguish expected nullable
GroupName values from genuine missing data.

In [127]:
# ================================================================
# STEP 83 — FINAL MATCH-TEAM VALIDATION
# ================================================================

# ------------------------------------------------
# 1. Basic structure
# ------------------------------------------------

expected_rows = 208
expected_matches = 104
expected_teams = 48

duplicate_match_team = (
    match_team_stats_df
    .duplicated(["MatchID", "TeamID"])
    .sum()
)

# ------------------------------------------------
# 2. Required match/team statistics
# ------------------------------------------------

required_stat_columns = [
    "Shots",
    "ShotsOnTarget",
    "Possession",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

missing_required_stats = (
    match_team_stats_df[required_stat_columns]
    .isna()
    .sum()
    .sum()
)

# ------------------------------------------------
# 3. Expected GroupName behavior
# ------------------------------------------------

group_stage_missing_group = (
    match_team_stats_df.loc[
        match_team_stats_df["Stage"] == "Group stage",
        "GroupName"
    ]
    .isna()
    .sum()
)

knockout_group_present = (
    match_team_stats_df.loc[
        match_team_stats_df["Stage"] != "Group stage",
        "GroupName"
    ]
    .notna()
    .sum()
)

# ------------------------------------------------
# 4. Possession validation
# ------------------------------------------------

possession_totals = (
    match_team_stats_df
    .groupby("MatchID")["Possession"]
    .sum()
)

invalid_possession = (
    ~possession_totals.isin([100, 101])
).sum()

# ------------------------------------------------
# 5. Shots validation
# ------------------------------------------------

invalid_shots = (
    match_team_stats_df["ShotsOnTarget"]
    > match_team_stats_df["Shots"]
).sum()

# ------------------------------------------------
# 6. Non-negative statistics
# ------------------------------------------------

non_negative_columns = [
    "TeamGoals",
    "OpponentGoals",
    "Shots",
    "ShotsOnTarget",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

negative_values = (
    match_team_stats_df[non_negative_columns] < 0
).sum().sum()

# ------------------------------------------------
# 7. Goal difference validation
# ------------------------------------------------

goal_difference_errors = (
    match_team_stats_df["GoalDifference"]
    != (
        match_team_stats_df["TeamGoals"]
        - match_team_stats_df["OpponentGoals"]
    )
).sum()

# ------------------------------------------------
# 8. Points validation
# ------------------------------------------------

expected_points = (
    match_team_stats_df["TeamResult"]
    .map({
        "Win": 3,
        "Draw": 1,
        "Loss": 0
    })
)

points_errors = (
    match_team_stats_df["Points"]
    != expected_points
).sum()

# ------------------------------------------------
# 9. Clean sheet validation
# ------------------------------------------------

expected_clean_sheet = (
    match_team_stats_df["OpponentGoals"] == 0
)

clean_sheet_errors = (
    match_team_stats_df["CleanSheet"]
    != expected_clean_sheet
).sum()

# ------------------------------------------------
# 10. Final status
# ------------------------------------------------

all_checks_passed = (
    len(match_team_stats_df) == expected_rows
    and
    match_team_stats_df["MatchID"].nunique() == expected_matches
    and
    match_team_stats_df["TeamID"].nunique() == expected_teams
    and
    duplicate_match_team == 0
    and
    missing_required_stats == 0
    and
    group_stage_missing_group == 0
    and
    knockout_group_present == 0
    and
    invalid_possession == 0
    and
    invalid_shots == 0
    and
    negative_values == 0
    and
    goal_difference_errors == 0
    and
    points_errors == 0
    and
    clean_sheet_errors == 0
)

# ------------------------------------------------
# 11. Results
# ------------------------------------------------

print("=" * 70)
print("FINAL MATCH-TEAM VALIDATION")
print("=" * 70)

print(f"Rows: {len(match_team_stats_df):,}")
print(
    f"Unique MatchIDs: "
    f"{match_team_stats_df['MatchID'].nunique():,}"
)
print(
    f"Unique TeamIDs: "
    f"{match_team_stats_df['TeamID'].nunique():,}"
)
print(
    f"Duplicate MatchID + TeamID: "
    f"{duplicate_match_team:,}"
)

print("\nREQUIRED STATISTICS")
print("-" * 70)

print(
    f"Missing required statistic cells: "
    f"{missing_required_stats:,}"
)

print("\nGROUPNAME")
print("-" * 70)

print(
    f"Group-stage rows missing GroupName: "
    f"{group_stage_missing_group:,}"
)

print(
    f"Knockout rows incorrectly containing GroupName: "
    f"{knockout_group_present:,}"
)

print("\nPOSSESSION")
print("-" * 70)

print(
    f"Valid possession totals (100 or 101): "
    f"{(invalid_possession == 0)}"
)

print(
    f"Invalid possession totals: "
    f"{invalid_possession:,}"
)

print("\nOTHER VALIDATIONS")
print("-" * 70)

print(f"ShotsOnTarget > Shots: {invalid_shots:,}")
print(f"Negative statistic values: {negative_values:,}")
print(f"GoalDifference errors: {goal_difference_errors:,}")
print(f"Points errors: {points_errors:,}")
print(f"CleanSheet errors: {clean_sheet_errors:,}")

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL MATCH-TEAM VALIDATION CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE CHECKS FAILED")

FINAL MATCH-TEAM VALIDATION
Rows: 208
Unique MatchIDs: 104
Unique TeamIDs: 48
Duplicate MatchID + TeamID: 0

REQUIRED STATISTICS
----------------------------------------------------------------------
Missing required statistic cells: 0

GROUPNAME
----------------------------------------------------------------------
Group-stage rows missing GroupName: 0
Knockout rows incorrectly containing GroupName: 144

POSSESSION
----------------------------------------------------------------------
Valid possession totals (100 or 101): True
Invalid possession totals: 0

OTHER VALIDATIONS
----------------------------------------------------------------------
ShotsOnTarget > Shots: 0
Negative statistic values: 0
GoalDifference errors: 0
Points errors: 0
CleanSheet errors: 0

FINAL VALIDATION STATUS


## Step 84 — Inspect Stage and GroupName Structure

The previous validation assumed that all group-stage matches have
Stage = "Group stage".

The validation shows this assumption is incorrect.

This step will inspect the actual Stage and GroupName combinations
in the match-team fact table.

No data will be modified.

In [128]:
# ================================================================
# STEP 84 — INSPECT STAGE AND GROUPNAME STRUCTURE
# ================================================================

print("=" * 70)
print("UNIQUE STAGE VALUES")
print("=" * 70)

print(
    match_team_stats_df["Stage"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\n" + "=" * 70)
print("STAGE + GROUPNAME COMBINATIONS")
print("=" * 70)

stage_group_summary = (
    match_team_stats_df
    .groupby(
        ["Stage", "GroupName"],
        dropna=False
    )
    .size()
    .reset_index(name="Rows")
    .sort_values(["Stage", "GroupName"], na_position="last")
)

print(
    stage_group_summary.to_string(index=False)
)

print("\n" + "=" * 70)
print("MATCH-LEVEL STAGE + GROUPNAME")
print("=" * 70)

match_level_summary = (
    match_team_stats_df[
        [
            "MatchID",
            "Stage",
            "GroupName"
        ]
    ]
    .drop_duplicates()
    .sort_values("MatchID")
)

print(
    match_level_summary.to_string(index=False)
)

UNIQUE STAGE VALUES
Stage
Bronze final       2
Final              2
First Stage      144
Quarter-final      8
Round of 16       16
Round of 32       32
Semi-final         4

STAGE + GROUPNAME COMBINATIONS
        Stage GroupName  Rows
 Bronze final       NaN     2
        Final       NaN     2
  First Stage   Group A    12
  First Stage   Group B    12
  First Stage   Group C    12
  First Stage   Group D    12
  First Stage   Group E    12
  First Stage   Group F    12
  First Stage   Group G    12
  First Stage   Group H    12
  First Stage   Group I    12
  First Stage   Group J    12
  First Stage   Group K    12
  First Stage   Group L    12
Quarter-final       NaN     8
  Round of 16       NaN    16
  Round of 32       NaN    32
   Semi-final       NaN     4

MATCH-LEVEL STAGE + GROUPNAME
 MatchID         Stage GroupName
       1   First Stage   Group A
       2   First Stage   Group A
       3   First Stage   Group B
       4   First Stage   Group D
       5   First Stage   Grou

## Step 85 — Correct Final Match-Team Validation

The tournament data uses `First Stage` for the group stage.

Therefore:

- `First Stage` matches must have a GroupName.
- All knockout stages must have GroupName = NaN.

No source data will be changed.

In [129]:
# ================================================================
# STEP 85 — CORRECT FINAL MATCH-TEAM VALIDATION
# ================================================================

# Expected group-stage label in the source data
group_stage_name = "First Stage"

# ------------------------------------------------
# 1. Structure
# ------------------------------------------------

duplicate_match_team = (
    match_team_stats_df
    .duplicated(["MatchID", "TeamID"])
    .sum()
)

# ------------------------------------------------
# 2. Required statistics
# ------------------------------------------------

required_stat_columns = [
    "Shots",
    "ShotsOnTarget",
    "Possession",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

missing_required_stats = (
    match_team_stats_df[required_stat_columns]
    .isna()
    .sum()
    .sum()
)

# ------------------------------------------------
# 3. Correct GroupName validation
# ------------------------------------------------

group_stage_missing_group = (
    match_team_stats_df.loc[
        match_team_stats_df["Stage"] == group_stage_name,
        "GroupName"
    ]
    .isna()
    .sum()
)

knockout_group_present = (
    match_team_stats_df.loc[
        match_team_stats_df["Stage"] != group_stage_name,
        "GroupName"
    ]
    .notna()
    .sum()
)

# ------------------------------------------------
# 4. Possession
# ------------------------------------------------

possession_totals = (
    match_team_stats_df
    .groupby("MatchID")["Possession"]
    .sum()
)

invalid_possession = (
    ~possession_totals.isin([100, 101])
).sum()

# ------------------------------------------------
# 5. Shots
# ------------------------------------------------

invalid_shots = (
    match_team_stats_df["ShotsOnTarget"]
    > match_team_stats_df["Shots"]
).sum()

# ------------------------------------------------
# 6. Non-negative statistics
# ------------------------------------------------

non_negative_columns = [
    "TeamGoals",
    "OpponentGoals",
    "Shots",
    "ShotsOnTarget",
    "Corners",
    "Fouls",
    "YellowCards",
    "RedCards",
    "Offsides",
]

negative_values = (
    match_team_stats_df[non_negative_columns] < 0
).sum().sum()

# ------------------------------------------------
# 7. Performance metrics
# ------------------------------------------------

goal_difference_errors = (
    match_team_stats_df["GoalDifference"]
    != (
        match_team_stats_df["TeamGoals"]
        - match_team_stats_df["OpponentGoals"]
    )
).sum()

expected_points = (
    match_team_stats_df["TeamResult"]
    .map({
        "Win": 3,
        "Draw": 1,
        "Loss": 0
    })
)

points_errors = (
    match_team_stats_df["Points"]
    != expected_points
).sum()

expected_clean_sheet = (
    match_team_stats_df["OpponentGoals"] == 0
)

clean_sheet_errors = (
    match_team_stats_df["CleanSheet"]
    != expected_clean_sheet
).sum()

# ------------------------------------------------
# 8. Final validation
# ------------------------------------------------

all_checks_passed = (
    len(match_team_stats_df) == 208
    and
    match_team_stats_df["MatchID"].nunique() == 104
    and
    match_team_stats_df["TeamID"].nunique() == 48
    and
    duplicate_match_team == 0
    and
    missing_required_stats == 0
    and
    group_stage_missing_group == 0
    and
    knockout_group_present == 0
    and
    invalid_possession == 0
    and
    invalid_shots == 0
    and
    negative_values == 0
    and
    goal_difference_errors == 0
    and
    points_errors == 0
    and
    clean_sheet_errors == 0
)

# ------------------------------------------------
# 9. Results
# ------------------------------------------------

print("=" * 70)
print("FINAL MATCH-TEAM VALIDATION")
print("=" * 70)

print(f"Rows: {len(match_team_stats_df):,}")
print(f"Unique MatchIDs: {match_team_stats_df['MatchID'].nunique():,}")
print(f"Unique TeamIDs: {match_team_stats_df['TeamID'].nunique():,}")
print(f"Duplicate MatchID + TeamID: {duplicate_match_team:,}")

print("\nREQUIRED STATISTICS")
print("-" * 70)
print(f"Missing required statistic cells: {missing_required_stats:,}")

print("\nGROUPNAME")
print("-" * 70)
print(f"First Stage rows missing GroupName: {group_stage_missing_group:,}")
print(f"Knockout rows incorrectly containing GroupName: {knockout_group_present:,}")

print("\nPOSSESSION")
print("-" * 70)
print(f"Invalid possession totals: {invalid_possession:,}")

print("\nOTHER VALIDATIONS")
print("-" * 70)
print(f"ShotsOnTarget > Shots: {invalid_shots:,}")
print(f"Negative statistic values: {negative_values:,}")
print(f"GoalDifference errors: {goal_difference_errors:,}")
print(f"Points errors: {points_errors:,}")
print(f"CleanSheet errors: {clean_sheet_errors:,}")

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL MATCH-TEAM VALIDATION CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE CHECKS FAILED")

FINAL MATCH-TEAM VALIDATION
Rows: 208
Unique MatchIDs: 104
Unique TeamIDs: 48
Duplicate MatchID + TeamID: 0

REQUIRED STATISTICS
----------------------------------------------------------------------
Missing required statistic cells: 0

GROUPNAME
----------------------------------------------------------------------
First Stage rows missing GroupName: 0
Knockout rows incorrectly containing GroupName: 0

POSSESSION
----------------------------------------------------------------------
Invalid possession totals: 0

OTHER VALIDATIONS
----------------------------------------------------------------------
ShotsOnTarget > Shots: 0
Negative statistic values: 0
GoalDifference errors: 0
Points errors: 0
CleanSheet errors: 0

FINAL VALIDATION STATUS
ALL MATCH-TEAM VALIDATION CHECKS PASSED


## Step 86 — Export Final Match-Team Fact Table

The Match-Team Fact Table has passed all validation checks.

It contains one row per team per match and is ready for use in
the FIFA World Cup 2026 analytics model.

In [130]:
# ================================================================
# STEP 86 — EXPORT FINAL MATCH-TEAM FACT TABLE
# ================================================================

match_team_output_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Match_Team_Stats.csv"
)

match_team_stats_df.to_csv(
    match_team_output_path,
    index=False
)

print("=" * 70)
print("MATCH-TEAM TABLE EXPORTED")
print("=" * 70)

print(f"File: {match_team_output_path}")
print(f"Rows: {len(match_team_stats_df):,}")
print(f"Columns: {len(match_team_stats_df.columns):,}")
print(
    f"Unique MatchIDs: "
    f"{match_team_stats_df['MatchID'].nunique():,}"
)
print(
    f"Unique TeamIDs: "
    f"{match_team_stats_df['TeamID'].nunique():,}"
)
print(
    f"Duplicate MatchID + TeamID: "
    f"{match_team_stats_df.duplicated(['MatchID', 'TeamID']).sum():,}"
)

print("\nColumns:")
print(match_team_stats_df.columns.tolist())

print("\nFile exists:", os.path.exists(match_team_output_path))

MATCH-TEAM TABLE EXPORTED
File: ..\Data\FIFA_WorldCup_2026_Match_Team_Stats.csv
Rows: 208
Columns: 21
Unique MatchIDs: 104
Unique TeamIDs: 48
Duplicate MatchID + TeamID: 0

Columns:
['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'TeamID', 'OpponentTeamID', 'TeamGoals', 'OpponentGoals', 'TeamResult', 'Shots', 'ShotsOnTarget', 'Possession', 'Corners', 'Fouls', 'YellowCards', 'RedCards', 'Offsides', 'GoalDifference', 'Points', 'CleanSheet']

File exists: True


## Step 87 — Build Match-Level Fact Table

Create one row per FIFA World Cup 2026 match.

This table will contain the match-level information needed for
Power BI analysis, including:

- Match
- Date
- Stage
- Group
- Stadium
- Teams
- Goals
- Winner
- Result
- Total Goals
- Goal Difference

The Match-Team table remains the detailed team-perspective table.
This table provides the single-row-per-match dimension/fact view.

In [131]:
# ================================================================
# STEP 87 — BUILD MATCH-LEVEL FACT TABLE
# ================================================================

# Start from the validated Matches table
match_fact_df = matches_df.copy()

# ------------------------------------------------
# Add match-level calculated metrics
# ------------------------------------------------

match_fact_df["TotalGoals"] = (
    match_fact_df["Team1Goals"]
    + match_fact_df["Team2Goals"]
)

match_fact_df["GoalDifference"] = (
    match_fact_df["Team1Goals"]
    - match_fact_df["Team2Goals"]
).abs()

match_fact_df["TotalTeams"] = 2

# ------------------------------------------------
# Display structure
# ------------------------------------------------

print("=" * 70)
print("MATCH-LEVEL FACT TABLE")
print("=" * 70)

print(f"Rows: {len(match_fact_df):,}")
print(f"Columns: {len(match_fact_df.columns):,}")

print("\nColumns:")
print(match_fact_df.columns.tolist())

print("\nFirst 10 rows:")
print(
    match_fact_df.head(10).to_string(index=False)
)

MATCH-LEVEL FACT TABLE
Rows: 104
Columns: 14

Columns:
['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result', 'TotalGoals', 'GoalDifference', 'TotalTeams']

First 10 rows:
 MatchID  MatchDate       Stage GroupName  StadiumID  Team1ID  Team2ID  Team1Goals  Team2Goals  WinnerTeamID Result  TotalGoals  GoalDifference  TotalTeams
       1 2026-06-11 First Stage   Group A          3        8       24           2           0           8.0      W           2               2           2
       2 2026-06-11 First Stage   Group A          4       31       38           2           1          31.0      W           3               1           2
       3 2026-06-12 First Stage   Group B          1       16       26           1           1           NaN      D           2               0           2
       4 2026-06-12 First Stage   Group D         11       12       32           4           1          12.0      W        

## Step 88 — Validate Match-Level Fact Table

Validate the match-level table for:

- One row per match
- Unique MatchID
- Valid team references
- Correct TotalGoals
- Correct GoalDifference
- Correct Result
- Correct WinnerTeamID
- Valid stage/group relationships
- No unexpected missing values

In [135]:
# ================================================================
# STEP 88 — VALIDATE MATCH-LEVEL FACT TABLE
# ================================================================

print("=" * 70)
print("MATCH-LEVEL FACT TABLE VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 0. Load reference tables needed for validation
# ------------------------------------------------

teams_validation_df = pd.read_csv(
    os.path.join(DATA_DIR, "Teams.csv")
)

stadiums_validation_df = pd.read_csv(
    os.path.join(DATA_DIR, "Stadiums.csv")
)

# ------------------------------------------------
# 1. Basic structure
# ------------------------------------------------

duplicate_matches = (
    match_fact_df["MatchID"].duplicated().sum()
)

print(f"Rows: {len(match_fact_df):,}")
print(
    f"Unique MatchIDs: "
    f"{match_fact_df['MatchID'].nunique():,}"
)
print(
    f"Duplicate MatchIDs: "
    f"{duplicate_matches:,}"
)

# ------------------------------------------------
# 2. Validate TotalGoals
# ------------------------------------------------

expected_total_goals = (
    match_fact_df["Team1Goals"]
    + match_fact_df["Team2Goals"]
)

total_goals_errors = (
    match_fact_df["TotalGoals"]
    != expected_total_goals
).sum()

# ------------------------------------------------
# 3. Validate GoalDifference
# ------------------------------------------------

expected_goal_difference = (
    match_fact_df["Team1Goals"]
    - match_fact_df["Team2Goals"]
).abs()

goal_difference_errors = (
    match_fact_df["GoalDifference"]
    != expected_goal_difference
).sum()

# ------------------------------------------------
# 4. Validate Result
# ------------------------------------------------

expected_result = np.where(
    match_fact_df["Team1Goals"] > match_fact_df["Team2Goals"],
    "W",
    np.where(
        match_fact_df["Team1Goals"] < match_fact_df["Team2Goals"],
        "L",
        "D"
    )
)

result_errors = (
    match_fact_df["Result"].values != expected_result
).sum()

# ------------------------------------------------
# 5. Validate WinnerTeamID
# ------------------------------------------------

winner_errors = 0

for _, row in match_fact_df.iterrows():

    if row["Team1Goals"] > row["Team2Goals"]:

        if row["WinnerTeamID"] != row["Team1ID"]:
            winner_errors += 1

    elif row["Team2Goals"] > row["Team1Goals"]:

        if row["WinnerTeamID"] != row["Team2ID"]:
            winner_errors += 1

    else:

        if not pd.isna(row["WinnerTeamID"]):
            winner_errors += 1

# ------------------------------------------------
# 6. Validate Team IDs
# ------------------------------------------------

valid_team_ids = set(
    teams_validation_df["TeamID"]
)

invalid_team1 = (
    ~match_fact_df["Team1ID"].isin(valid_team_ids)
).sum()

invalid_team2 = (
    ~match_fact_df["Team2ID"].isin(valid_team_ids)
).sum()

# ------------------------------------------------
# 7. Validate Stadium IDs
# ------------------------------------------------

valid_stadium_ids = set(
    stadiums_validation_df["StadiumID"]
)

invalid_stadiums = (
    ~match_fact_df["StadiumID"].isin(valid_stadium_ids)
).sum()

# ------------------------------------------------
# 8. Validate GroupName
# ------------------------------------------------

group_stage_missing = (
    match_fact_df.loc[
        match_fact_df["Stage"] == "First Stage",
        "GroupName"
    ].isna().sum()
)

knockout_group_present = (
    match_fact_df.loc[
        match_fact_df["Stage"] != "First Stage",
        "GroupName"
    ].notna().sum()
)

# ------------------------------------------------
# 9. Validate TotalTeams
# ------------------------------------------------

total_teams_errors = (
    match_fact_df["TotalTeams"] != 2
).sum()

# ------------------------------------------------
# 10. Report
# ------------------------------------------------

print("\n" + "=" * 70)
print("CALCULATED METRICS")
print("=" * 70)

print(f"TotalGoals errors: {total_goals_errors:,}")
print(f"GoalDifference errors: {goal_difference_errors:,}")
print(f"Result errors: {result_errors:,}")
print(f"WinnerTeamID errors: {winner_errors:,}")

print("\n" + "=" * 70)
print("REFERENCE VALIDATION")
print("=" * 70)

print(f"Invalid Team1ID: {invalid_team1:,}")
print(f"Invalid Team2ID: {invalid_team2:,}")
print(f"Invalid StadiumID: {invalid_stadiums:,}")

print("\n" + "=" * 70)
print("GROUP VALIDATION")
print("=" * 70)

print(
    f"First Stage missing GroupName: "
    f"{group_stage_missing:,}"
)

print(
    f"Knockout GroupName present: "
    f"{knockout_group_present:,}"
)

print("\n" + "=" * 70)
print("OTHER VALIDATION")
print("=" * 70)

print(
    f"TotalTeams errors: "
    f"{total_teams_errors:,}"
)

# ------------------------------------------------
# 11. Final status
# ------------------------------------------------

all_checks_passed = (
    len(match_fact_df) == 104
    and match_fact_df["MatchID"].nunique() == 104
    and duplicate_matches == 0
    and total_goals_errors == 0
    and goal_difference_errors == 0
    and result_errors == 0
    and winner_errors == 0
    and invalid_team1 == 0
    and invalid_team2 == 0
    and invalid_stadiums == 0
    and group_stage_missing == 0
    and knockout_group_present == 0
    and total_teams_errors == 0
)

print("\n" + "=" * 70)
print("FINAL VALIDATION STATUS")
print("=" * 70)

if all_checks_passed:
    print("ALL MATCH-LEVEL VALIDATION CHECKS PASSED")
else:
    print("WARNING: ONE OR MORE CHECKS FAILED")

MATCH-LEVEL FACT TABLE VALIDATION
Rows: 104
Unique MatchIDs: 104
Duplicate MatchIDs: 0

CALCULATED METRICS
TotalGoals errors: 0
GoalDifference errors: 0
Result errors: 4
WinnerTeamID errors: 4

REFERENCE VALIDATION
Invalid Team1ID: 0
Invalid Team2ID: 0
Invalid StadiumID: 0

GROUP VALIDATION
First Stage missing GroupName: 0
Knockout GroupName present: 0

OTHER VALIDATION
TotalTeams errors: 0

FINAL VALIDATION STATUS


## Step 89 — Investigate Result and Winner Mismatches

The validation found 4 matches where the calculated Result and
WinnerTeamID do not match the values in Matches.csv.

This may be caused by knockout matches decided by penalty shootouts,
where the match score is tied but one team advances.

We will inspect the affected matches before making any changes.

In [136]:
# ================================================================
# STEP 89 — INVESTIGATE RESULT / WINNER MISMATCHES
# ================================================================

# ------------------------------------------------
# Identify Result mismatches
# ------------------------------------------------

result_mismatch_mask = (
    match_fact_df["Result"] != np.where(
        match_fact_df["Team1Goals"] > match_fact_df["Team2Goals"],
        "W",
        np.where(
            match_fact_df["Team1Goals"] < match_fact_df["Team2Goals"],
            "L",
            "D"
        )
    )
)

# ------------------------------------------------
# Identify WinnerTeamID mismatches
# ------------------------------------------------

mismatch_matches = match_fact_df[
    result_mismatch_mask
].copy()

print("=" * 70)
print("AFFECTED MATCHES")
print("=" * 70)

print(
    mismatch_matches[
        [
            "MatchID",
            "MatchDate",
            "Stage",
            "GroupName",
            "Team1ID",
            "Team2ID",
            "Team1Goals",
            "Team2Goals",
            "WinnerTeamID",
            "Result"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("MISMATCH COUNT")
print("=" * 70)

print(
    f"Result mismatches: "
    f"{result_mismatch_mask.sum():,}"
)

# ------------------------------------------------
# Show whether all are tied scores
# ------------------------------------------------

print("\n" + "=" * 70)
print("SCORE ANALYSIS")
print("=" * 70)

mismatch_matches["ScoreTied"] = (
    mismatch_matches["Team1Goals"]
    == mismatch_matches["Team2Goals"]
)

print(
    mismatch_matches[
        [
            "MatchID",
            "Stage",
            "Team1Goals",
            "Team2Goals",
            "ScoreTied",
            "WinnerTeamID",
            "Result"
        ]
    ].to_string(index=False)
)

AFFECTED MATCHES
 MatchID  MatchDate       Stage GroupName  Team1ID  Team2ID  Team1Goals  Team2Goals  WinnerTeamID Result
      74 2026-06-29 Round of 32       NaN        7       32           1           1          32.0      L
      75 2026-06-29 Round of 32       NaN        9       21           1           1          21.0      L
      88 2026-07-03 Round of 32       NaN       13       19           1           1          19.0      L
      96 2026-07-07 Round of 16       NaN       11        4           0           0          11.0      W

MISMATCH COUNT
Result mismatches: 4

SCORE ANALYSIS
 MatchID       Stage  Team1Goals  Team2Goals  ScoreTied  WinnerTeamID Result
      74 Round of 32           1           1       True          32.0      L
      75 Round of 32           1           1       True          21.0      L
      88 Round of 32           1           1       True          19.0      L
      96 Round of 16           0           0       True          11.0      W


## Step 90 — Finalize Match-Level Fact Table

Use the validated source values from Matches.csv for:

- Result
- WinnerTeamID

Calculate only:

- TotalGoals
- GoalDifference

This preserves penalty-shootout results correctly.

In [137]:
# ================================================================
# STEP 90 — FINALIZE MATCH-LEVEL FACT TABLE
# ================================================================

# Keep source Result and WinnerTeamID
match_fact_df["TotalGoals"] = (
    match_fact_df["Team1Goals"]
    + match_fact_df["Team2Goals"]
)

match_fact_df["GoalDifference"] = (
    match_fact_df["Team1Goals"]
    - match_fact_df["Team2Goals"]
).abs()

match_fact_df["TotalTeams"] = 2

# Final structure check
print("=" * 70)
print("MATCH-LEVEL FACT TABLE READY")
print("=" * 70)

print(f"Rows: {len(match_fact_df):,}")
print(f"Columns: {len(match_fact_df.columns):,}")
print(f"Unique MatchIDs: {match_fact_df['MatchID'].nunique():,}")
print(f"Duplicate MatchIDs: {match_fact_df['MatchID'].duplicated().sum():,}")

print("\nColumns:")
print(match_fact_df.columns.tolist())

print("\nResult distribution:")
print(match_fact_df["Result"].value_counts().to_string())

print("\nWinnerTeamID missing:")
print(match_fact_df["WinnerTeamID"].isna().sum())

print("\nMatch-level table is ready for SQL.")

MATCH-LEVEL FACT TABLE READY
Rows: 104
Columns: 14
Unique MatchIDs: 104
Duplicate MatchIDs: 0

Columns:
['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result', 'TotalGoals', 'GoalDifference', 'TotalTeams']

Result distribution:
Result
W    51
L    33
D    20

WinnerTeamID missing:
20

Match-level table is ready for SQL.


## Step 91 — Export Final Match-Level Table

Export the validated match-level fact table as a CSV for the SQL
database and Power BI model.

In [3]:
# ================================================================
# STEP 91 — EXPORT MATCH-LEVEL TABLE
# ================================================================

match_output_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Match_Fact.csv"
)

match_fact_df.to_csv(
    match_output_path,
    index=False
)

print("=" * 70)
print("MATCH-LEVEL TABLE EXPORTED")
print("=" * 70)

print(f"File: {match_output_path}")
print(f"Rows: {len(match_fact_df):,}")
print(f"Columns: {len(match_fact_df.columns):,}")
print(f"Unique MatchIDs: {match_fact_df['MatchID'].nunique():,}")
print(f"File exists: {os.path.exists(match_output_path)}")

NameError: name 'os' is not defined

## Step 92 — Load Final Master Player Statistics

Load the exported FIFA World Cup 2026 master player statistics
table so we can safely fix the remaining blank statistical fields.

In [4]:
# ================================================================
# STEP 92 — LOAD FINAL MASTER PLAYER STATS
# ================================================================

import os
import pandas as pd
import numpy as np

DATA_DIR = r"D:\ai course hw\FIFA_WorldCup_2026_Project\Data"

master_player_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Master_Player_Stats.csv"
)

master_player_stats_final = pd.read_csv(
    master_player_path
)

print("=" * 70)
print("MASTER PLAYER STATS LOADED")
print("=" * 70)

print(f"Rows: {len(master_player_stats_final):,}")
print(f"Columns: {len(master_player_stats_final.columns):,}")
print(
    f"Unique PlayerIDs: "
    f"{master_player_stats_final['PlayerID'].nunique():,}"
)

print("\nTotal missing cells:")
print(
    f"{master_player_stats_final.isna().sum().sum():,}"
)

print("\nLoaded successfully.")

MASTER PLAYER STATS LOADED
Rows: 1,248
Columns: 63
Unique PlayerIDs: 1,248

Total missing cells:
9,378

Loaded successfully.


## Step 93 — Audit Missing Player Statistics

Inspect missing values by column.

We will identify which blanks should become zero and which blanks
represent statistics that were not available in the scraped source.

In [5]:
# ================================================================
# STEP 93 — AUDIT MISSING PLAYER STATISTICS
# ================================================================

missing_stats = (
    master_player_stats_final
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_stats = missing_stats[missing_stats > 0]

print("=" * 70)
print("MISSING VALUES BY COLUMN")
print("=" * 70)

print(missing_stats.to_string())

print("\n" + "=" * 70)
print("TOTAL MISSING CELLS")
print("=" * 70)

print(
    f"{master_player_stats_final.isna().sum().sum():,}"
)

print("\n" + "=" * 70)
print("COLUMNS WITH MISSING VALUES")
print("=" * 70)

print(f"{len(missing_stats):,}")

MISSING VALUES BY COLUMN
xGEfficiency                         548
HeadedAttemptsatGoal                 548
AttemptsOutsidethePenaltyArea        548
AttemptsAtGoalConvRatePct            548
AttemptsInsidethePenaltyArea         548
AttemptsOnTarget                     548
Corners                              548
xG                                   548
AttemptsAtGoal                       548
HighSpeedRunning                     219
Sprints                              219
DistanceMeters                       219
Matches                              219
Around                               219
TopSpeedKmh                          219
Local_Crosses                        219
Prog                                 219
Over                                 219
Pass                                 219
TotalCrosses                         219
SpeedZone3                           219
SpeedZone2                           219
Through                              219
LineBreaksAccuracyPct           

## Step 94 — Replace Goalkeeping and Defending Blanks with Zero

Goalkeeping statistics are not applicable to players without a
Goalkeeping record, so missing Goalkeeping values will be represented
as zero.

Defending statistics are event/count statistics, so missing values
will also be represented as zero.

Attacking and Physical blanks will remain unchanged because those
datasets have incomplete source coverage.

In [6]:
# ================================================================
# STEP 94 — FIX GOALKEEPING + DEFENDING BLANKS
# ================================================================

goalkeeping_columns = [
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea"
]

defending_columns = [
    "OwnGoals",
    "ForcedTurnovers",
    "DefensivePressuresApplied",
    "DefensivePressuresDirectlyApplied"
]

# ------------------------------------------------
# Convert to numeric and fill missing with zero
# ------------------------------------------------

for col in goalkeeping_columns + defending_columns:
    master_player_stats_final[col] = pd.to_numeric(
        master_player_stats_final[col],
        errors="coerce"
    ).fillna(0)

# Convert count statistics to integers
for col in goalkeeping_columns + defending_columns:
    master_player_stats_final[col] = (
        master_player_stats_final[col]
        .round()
        .astype(int)
    )

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("GOALKEEPING + DEFENDING FIX")
print("=" * 70)

print("\nGOALKEEPING")
print("-" * 70)

for col in goalkeeping_columns:
    print(
        f"{col}: "
        f"{master_player_stats_final[col].isna().sum():,} missing | "
        f"{(master_player_stats_final[col] == 0).sum():,} zeros"
    )

print("\nDEFENDING")
print("-" * 70)

for col in defending_columns:
    print(
        f"{col}: "
        f"{master_player_stats_final[col].isna().sum():,} missing | "
        f"{(master_player_stats_final[col] == 0).sum():,} zeros"
    )

print("\n" + "=" * 70)
print("TOTAL MISSING CELLS AFTER FIX")
print("=" * 70)

print(
    f"{master_player_stats_final.isna().sum().sum():,}"
)

print("\nGoalkeeping and Defending blanks fixed.")

GOALKEEPING + DEFENDING FIX

GOALKEEPING
----------------------------------------------------------------------
GoalkeeperSaves: 0 missing | 1,191 zeros
GoalkeeperActionsInsidethePenaltyArea: 0 missing | 1,187 zeros
GoalkeeperActionsOutsidethePenaltyArea: 0 missing | 1,187 zeros

DEFENDING
----------------------------------------------------------------------
OwnGoals: 0 missing | 1,235 zeros
ForcedTurnovers: 0 missing | 364 zeros
DefensivePressuresApplied: 0 missing | 289 zeros
DefensivePressuresDirectlyApplied: 0 missing | 348 zeros

TOTAL MISSING CELLS AFTER FIX
9,360

Goalkeeping and Defending blanks fixed.


## Step 95 — Classify Remaining Missing Values

Separate the remaining blanks into:

1. FIFA Attacking statistics
2. FIFA Distribution statistics
3. FIFA Discipline statistics
4. FIFA Goalkeeping statistics
5. Local Physical statistics
6. Other player statistics

This will determine which blanks are safe to convert to zero.

In [7]:
# ================================================================
# STEP 95 — CLASSIFY REMAINING MISSING VALUES
# ================================================================

missing_by_column = (
    master_player_stats_final
    .isna()
    .sum()
)

missing_by_column = missing_by_column[
    missing_by_column > 0
]

# ------------------------------------------------
# Define source groups
# ------------------------------------------------

attacking_columns = [
    "AttemptsOnTarget",
    "AttemptsAtGoal",
    "AttemptsAtGoalConvRatePct",
    "AttemptsInsidethePenaltyArea",
    "AttemptsOutsidethePenaltyArea",
    "HeadedAttemptsatGoal",
    "xG",
    "xGEfficiency",
    "Corners"
]

physical_columns = [
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning",
    "TopSpeedKmh",
    "SpeedZone1",
    "SpeedZone2",
    "SpeedZone3",
    "SpeedZone4",
    "SpeedZone5",
    "LineBreaksAttempted",
    "LineBreaksCompleted",
    "LineBreaksAccuracyPct",
    "Through",
    "Around",
    "Over",
    "Pass",
    "Local_Crosses",
    "Prog",
    "TotalCrosses"
]

# ------------------------------------------------
# Print groups
# ------------------------------------------------

print("=" * 70)
print("REMAINING MISSING VALUES — ATTACKING")
print("=" * 70)

for col in attacking_columns:
    print(f"{col}: {missing_by_column.get(col, 0):,}")

print("\n" + "=" * 70)
print("REMAINING MISSING VALUES — PHYSICAL")
print("=" * 70)

for col in physical_columns:
    print(f"{col}: {missing_by_column.get(col, 0):,}")

# ------------------------------------------------
# Everything else
# ------------------------------------------------

known_columns = set(attacking_columns + physical_columns)

other_missing = missing_by_column[
    ~missing_by_column.index.isin(known_columns)
]

print("\n" + "=" * 70)
print("OTHER REMAINING MISSING VALUES")
print("=" * 70)

if len(other_missing) == 0:
    print("None")
else:
    print(other_missing.to_string())

print("\n" + "=" * 70)
print("TOTAL REMAINING MISSING CELLS")
print("=" * 70)

print(
    f"{master_player_stats_final.isna().sum().sum():,}"
)

REMAINING MISSING VALUES — ATTACKING
AttemptsOnTarget: 548
AttemptsAtGoal: 548
AttemptsAtGoalConvRatePct: 548
AttemptsInsidethePenaltyArea: 548
AttemptsOutsidethePenaltyArea: 548
HeadedAttemptsatGoal: 548
xG: 548
xGEfficiency: 548
Corners: 548

REMAINING MISSING VALUES — PHYSICAL
Matches: 219
DistanceMeters: 219
Sprints: 219
HighSpeedRunning: 219
TopSpeedKmh: 219
SpeedZone1: 219
SpeedZone2: 219
SpeedZone3: 219
SpeedZone4: 219
SpeedZone5: 219
LineBreaksAttempted: 219
LineBreaksCompleted: 219
LineBreaksAccuracyPct: 219
Through: 219
Around: 219
Over: 219
Pass: 219
Local_Crosses: 219
Prog: 219
TotalCrosses: 219

OTHER REMAINING MISSING VALUES
Passes                          6
PassingAccuracyPct              6
Crosses                         6
CrossingAccuracyPct             6
DefensiveLinebreaksAttempted    6
DefensiveLinebreaksAccPct       6
SwitchesofPlayAttempted         6
SwitchesofPlayAccPct            6

TOTAL REMAINING MISSING CELLS
9,360


## Step 96 — Verify FIFA Goalkeeping Statistics

The FIFA Goalkeeping category contains 145 player records.

We will verify that these records are correctly represented in the
master player table.

Goalkeeping statistics will NOT be filled blindly with zero.

Only players who genuinely have no FIFA Goalkeeping record will be
considered non-goalkeepers.

In [9]:
# ================================================================
# STEP 96 — VERIFY FIFA GOALKEEPING STATISTICS
# ================================================================

# ------------------------------------------------
# 1. Load FIFA player mapping
# ------------------------------------------------

mapping_path = os.path.join(
    DATA_DIR,
    "FIFA_PlayerID_Mapping.csv"
)

fifa_mapping_df = pd.read_csv(mapping_path)

# ------------------------------------------------
# 2. Goalkeeping columns
# ------------------------------------------------

goalkeeping_columns = [
    "GoalkeeperSaves",
    "GoalkeeperActionsInsidethePenaltyArea",
    "GoalkeeperActionsOutsidethePenaltyArea"
]

print("=" * 70)
print("FIFA GOALKEEPING VERIFICATION")
print("=" * 70)

# ------------------------------------------------
# 3. Players with non-zero goalkeeper statistics
# ------------------------------------------------

goalkeeper_players_master = master_player_stats_final[
    master_player_stats_final[goalkeeping_columns]
    .fillna(0)
    .sum(axis=1) > 0
].copy()

print(
    f"Players with non-zero goalkeeper statistics: "
    f"{len(goalkeeper_players_master):,}"
)

# ------------------------------------------------
# 4. FIFA players marked as GK
# ------------------------------------------------

fifa_gk_players = fifa_mapping_df[
    fifa_mapping_df["Position"].str.upper() == "GK"
].copy()

print(
    f"FIFA players marked GK: "
    f"{len(fifa_gk_players):,}"
)

# ------------------------------------------------
# 5. Match GK players to master
# ------------------------------------------------

gk_master = master_player_stats_final[
    master_player_stats_final["PlayerID"].isin(
        fifa_gk_players["PlayerID"]
    )
].copy()

print(
    f"GK players found in master: "
    f"{len(gk_master):,}"
)

# ------------------------------------------------
# 6. Count zero/non-zero GK statistics among GK players
# ------------------------------------------------

gk_zero_mask = (
    gk_master[goalkeeping_columns]
    .fillna(0)
    .sum(axis=1) == 0
)

gk_nonzero_mask = (
    gk_master[goalkeeping_columns]
    .fillna(0)
    .sum(axis=1) > 0
)

print(
    f"GK players with all goalkeeper statistics = 0: "
    f"{gk_zero_mask.sum():,}"
)

print(
    f"GK players with at least one non-zero statistic: "
    f"{gk_nonzero_mask.sum():,}"
)

# ------------------------------------------------
# 7. Display GK players
# ------------------------------------------------

print("\n" + "=" * 70)
print("GOALKEEPER STATISTICS")
print("=" * 70)

print(
    gk_master[
        [
            "PlayerID",
            "PlayerName",
            "GoalkeeperSaves",
            "GoalkeeperActionsInsidethePenaltyArea",
            "GoalkeeperActionsOutsidethePenaltyArea"
        ]
    ]
    .sort_values(
        "GoalkeeperSaves",
        ascending=False
    )
    .to_string(index=False)
)

FIFA GOALKEEPING VERIFICATION
Players with non-zero goalkeeper statistics: 61
FIFA players marked GK: 145
GK players found in master: 145
GK players with all goalkeeper statistics = 0: 84
GK players with at least one non-zero statistic: 61

GOALKEEPER STATISTICS
 PlayerID               PlayerName  GoalkeeperSaves  GoalkeeperActionsInsidethePenaltyArea  GoalkeeperActionsOutsidethePenaltyArea
      870             Orlando Gill               28                                    141                                     124
      339                Eloy Room               21                                     81                                      72
      885              Diogo Costa               20                                    114                                      93
     1093             Gregor Kobel               20                                    100                                      88
       49        Emiliano Martinez               20                               

## Step 97 — Inspect Exported Master CSV

Load the actual exported FIFA World Cup 2026 Master Player Stats CSV
and identify the remaining blank numeric statistics.

The goal is to produce a Power BI-ready CSV with the required
statistics represented consistently.

In [10]:
# ================================================================
# STEP 97 — INSPECT EXPORTED MASTER CSV
# ================================================================

import os
import pandas as pd
import numpy as np

DATA_DIR = r"D:\ai course hw\FIFA_WorldCup_2026_Project\Data"

master_player_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Master_Player_Stats.csv"
)

csv_check_df = pd.read_csv(master_player_path)

print("=" * 70)
print("EXPORTED MASTER CSV")
print("=" * 70)

print(f"Rows: {len(csv_check_df):,}")
print(f"Columns: {len(csv_check_df.columns):,}")
print(f"Unique PlayerIDs: {csv_check_df['PlayerID'].nunique():,}")

print("\nTotal blank cells:")
print(
    f"{csv_check_df.isna().sum().sum():,}"
)

print("\nColumns containing blanks:")
print(
    csv_check_df.isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
    .to_string()
)

EXPORTED MASTER CSV
Rows: 1,248
Columns: 63
Unique PlayerIDs: 1,248

Total blank cells:
9,378

Columns containing blanks:
AttemptsOutsidethePenaltyArea        548
HeadedAttemptsatGoal                 548
xG                                   548
xGEfficiency                         548
AttemptsInsidethePenaltyArea         548
AttemptsAtGoalConvRatePct            548
AttemptsAtGoal                       548
AttemptsOnTarget                     548
Corners                              548
SpeedZone2                           219
SpeedZone1                           219
TopSpeedKmh                          219
HighSpeedRunning                     219
Sprints                              219
DistanceMeters                       219
Matches                              219
SpeedZone3                           219
Over                                 219
Through                              219
Around                               219
SpeedZone4                           219
SpeedZone5       

## Step 98 — Create Power BI/SQL-Ready Player Statistics

The exported Master Player Stats CSV contains blank numeric cells.

For the final analytics model, all numeric player-statistic fields
will be represented as zero when no value is available.

Player identity fields will not be modified.

This creates a clean numeric table for SQL and Power BI without
blank statistical cells.

In [11]:
# ================================================================
# STEP 98 — FILL NUMERIC STATISTIC BLANKS WITH ZERO
# ================================================================

# ------------------------------------------------
# Load exported CSV
# ------------------------------------------------

master_player_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Master_Player_Stats.csv"
)

player_stats_export_df = pd.read_csv(
    master_player_path
)

# ------------------------------------------------
# Identify numeric columns
# ------------------------------------------------

numeric_columns = (
    player_stats_export_df
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

# ------------------------------------------------
# Count blanks before
# ------------------------------------------------

missing_before = (
    player_stats_export_df[numeric_columns]
    .isna()
    .sum()
    .sum()
)

# ------------------------------------------------
# Fill numeric blanks
# ------------------------------------------------

player_stats_export_df[numeric_columns] = (
    player_stats_export_df[numeric_columns]
    .fillna(0)
)

# ------------------------------------------------
# Count blanks after
# ------------------------------------------------

missing_after = (
    player_stats_export_df[numeric_columns]
    .isna()
    .sum()
    .sum()
)

print("=" * 70)
print("NUMERIC BLANK FIX")
print("=" * 70)

print(
    f"Numeric blank cells before: "
    f"{missing_before:,}"
)

print(
    f"Numeric blank cells after: "
    f"{missing_after:,}"
)

print(
    f"Total blank cells after: "
    f"{player_stats_export_df.isna().sum().sum():,}"
)

print("\nNumeric columns processed:")
print(f"{len(numeric_columns):,}")

if missing_after == 0:
    print("\nALL NUMERIC STATISTIC BLANKS REPLACED WITH ZERO")
else:
    print("\nWARNING: NUMERIC BLANKS REMAIN")

NUMERIC BLANK FIX
Numeric blank cells before: 9,378
Numeric blank cells after: 0
Total blank cells after: 0

Numeric columns processed:
61

ALL NUMERIC STATISTIC BLANKS REPLACED WITH ZERO


## Step 98 — Complete Player Statistics CSV

Replace all remaining blank numeric statistic values with 0.

Player identity fields remain unchanged:

- PlayerID
- PlayerName
- Nation
- Position

The resulting CSV will contain no blank numeric statistics and will
be ready for SQL and Power BI.

In [12]:
# ================================================================
# STEP 98 — COMPLETE PLAYER STATISTICS CSV
# ================================================================

import os
import pandas as pd
import numpy as np

DATA_DIR = r"D:\ai course hw\FIFA_WorldCup_2026_Project\Data"

master_player_path = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Master_Player_Stats.csv"
)

# ------------------------------------------------
# Load exported master CSV
# ------------------------------------------------

player_stats_df = pd.read_csv(master_player_path)

print("=" * 70)
print("BEFORE FIX")
print("=" * 70)

print(f"Rows: {len(player_stats_df):,}")
print(f"Columns: {len(player_stats_df.columns):,}")
print(
    f"Blank cells: "
    f"{player_stats_df.isna().sum().sum():,}"
)

# ------------------------------------------------
# Identity columns — NEVER fill these
# ------------------------------------------------

identity_columns = [
    "PlayerID",
    "PlayerName",
    "TeamID",
    "Position",
]

# ------------------------------------------------
# Find all numeric/statistic columns
# ------------------------------------------------

numeric_columns = [
    col
    for col in player_stats_df.columns
    if col not in identity_columns
    and pd.api.types.is_numeric_dtype(player_stats_df[col])
]

# ------------------------------------------------
# Fill all numeric blanks with zero
# ------------------------------------------------

missing_before = (
    player_stats_df[numeric_columns]
    .isna()
    .sum()
    .sum()
)

player_stats_df[numeric_columns] = (
    player_stats_df[numeric_columns]
    .fillna(0)
)

missing_after = (
    player_stats_df[numeric_columns]
    .isna()
    .sum()
    .sum()
)

# ------------------------------------------------
# Check identity columns
# ------------------------------------------------

identity_missing = (
    player_stats_df[identity_columns]
    .isna()
    .sum()
)

# ------------------------------------------------
# Results
# ------------------------------------------------

print("\n" + "=" * 70)
print("AFTER FIX")
print("=" * 70)

print(
    f"Numeric blanks before: "
    f"{missing_before:,}"
)

print(
    f"Numeric blanks after: "
    f"{missing_after:,}"
)

print(
    f"Total blanks after: "
    f"{player_stats_df.isna().sum().sum():,}"
)

print("\nIdentity column blanks:")
print(identity_missing.to_string())

print("\n" + "=" * 70)
print("PLAYER TABLE")
print("=" * 70)

print(f"Rows: {len(player_stats_df):,}")
print(f"Columns: {len(player_stats_df.columns):,}")
print(
    f"Unique PlayerIDs: "
    f"{player_stats_df['PlayerID'].nunique():,}"
)

if (
    missing_after == 0
    and player_stats_df[identity_columns].isna().sum().sum() == 0
    and player_stats_df["PlayerID"].nunique() == len(player_stats_df)
):
    print("\nALL REMAINING PLAYER STAT BLANKS FIXED")
else:
    print("\nWARNING: ISSUES REMAIN")

BEFORE FIX
Rows: 1,248
Columns: 63
Blank cells: 9,378

AFTER FIX
Numeric blanks before: 9,378
Numeric blanks after: 0
Total blanks after: 0

Identity column blanks:
PlayerID      0
PlayerName    0
TeamID        0
Position      0

PLAYER TABLE
Rows: 1,248
Columns: 63
Unique PlayerIDs: 1,248

ALL REMAINING PLAYER STAT BLANKS FIXED


## Step 100 — Set Goalkeeper Attacking Statistics to Zero

Goalkeepers do not have applicable attacking statistics in the
FIFA player-statistics model.

For every player whose Position is GK, the attacking statistics
will be represented as 0 instead of blank.

Non-goalkeeper attacking statistics will remain unchanged.

In [13]:
# ================================================================
# STEP 100 — SET GOALKEEPER ATTACKING STATS TO ZERO
# ================================================================

# ------------------------------------------------
# Attacking columns
# ------------------------------------------------

attacking_columns = [
    "Assists",
    "AttemptsOnTarget",
    "AttemptsAtGoal",
    "AttemptsAtGoalConvRatePct",
    "AttemptsInsidethePenaltyArea",
    "AttemptsOutsidethePenaltyArea",
    "HeadedAttemptsatGoal",
    "xG",
    "xGEfficiency",
    "Corners"
]

# ------------------------------------------------
# Identify goalkeepers
# ------------------------------------------------

goalkeeper_mask = (
    player_stats_df["Position"]
    .astype(str)
    .str.upper()
    .eq("GK")
)

goalkeeper_count = goalkeeper_mask.sum()

# ------------------------------------------------
# Set attacking stats to zero for GK
# ------------------------------------------------

for col in attacking_columns:
    player_stats_df.loc[
        goalkeeper_mask,
        col
    ] = 0

# ------------------------------------------------
# Validate
# ------------------------------------------------

remaining_gk_blanks = (
    player_stats_df.loc[
        goalkeeper_mask,
        attacking_columns
    ]
    .isna()
    .sum()
    .sum()
)

print("=" * 70)
print("GOALKEEPER ATTACKING STATISTICS")
print("=" * 70)

print(f"Goalkeepers: {goalkeeper_count:,}")
print(
    f"Remaining GK attacking blanks: "
    f"{remaining_gk_blanks:,}"
)

print("\nGK attacking statistics:")
print(
    player_stats_df.loc[
        goalkeeper_mask,
        ["PlayerID", "PlayerName"] + attacking_columns
    ]
    .head(10)
    .to_string(index=False)
)

if remaining_gk_blanks == 0:
    print("\nALL GOALKEEPER ATTACKING VALUES ARE ZERO/VALID")
else:
    print("\nWARNING: GK attacking blanks remain")

GOALKEEPER ATTACKING STATISTICS
Goalkeepers: 145
Remaining GK attacking blanks: 0

GK attacking statistics:
 PlayerID         PlayerName  Assists  AttemptsOnTarget  AttemptsAtGoal  AttemptsAtGoalConvRatePct  AttemptsInsidethePenaltyArea  AttemptsOutsidethePenaltyArea  HeadedAttemptsatGoal  xG  xGEfficiency  Corners
        1      Melvin Mastil        0               0.0             0.0                        0.0                           0.0                            0.0                   0.0 0.0           0.0      0.0
       16     Oussama Benbot        0               0.0             0.0                        0.0                           0.0                            0.0                   0.0 0.0           0.0      0.0
       23        Luca Zidane        0               0.0             0.0                        0.0                           0.0                            0.0                   0.0 0.0           0.0      0.0
       27         Juan Musso        0               0.0 

## Step 101 — Fix Goalkeeper Physical Statistics

Some goalkeeper records contain blank physical statistics.

For the final analytics CSV, missing numeric physical statistics for
goalkeepers will be represented as 0.

Existing physical-stat values will not be changed.

In [14]:
# ================================================================
# STEP 101 — FIX GOALKEEPER PHYSICAL BLANKS
# ================================================================

physical_columns = [
    "Matches",
    "DistanceMeters",
    "Sprints",
    "HighSpeedRunning",
    "TopSpeedKmh",
    "SpeedZone1",
    "SpeedZone2",
    "SpeedZone3",
    "SpeedZone4",
    "SpeedZone5",
    "LineBreaksAttempted",
    "LineBreaksCompleted",
    "LineBreaksAccuracyPct",
    "Through",
    "Around",
    "Over",
    "Pass",
    "Local_Crosses",
    "Prog",
    "TotalCrosses"
]

# ------------------------------------------------
# Identify goalkeepers
# ------------------------------------------------

goalkeeper_mask = (
    player_stats_df["Position"]
    .astype(str)
    .str.upper()
    .eq("GK")
)

print("=" * 70)
print("GOALKEEPER PHYSICAL STATISTICS")
print("=" * 70)

print(f"Goalkeepers: {goalkeeper_mask.sum():,}")

# ------------------------------------------------
# Missing values BEFORE
# ------------------------------------------------

missing_before = (
    player_stats_df.loc[
        goalkeeper_mask,
        physical_columns
    ]
    .isna()
    .sum()
    .sum()
)

print(
    f"Missing goalkeeper physical cells before: "
    f"{missing_before:,}"
)

# ------------------------------------------------
# Fill goalkeeper physical blanks with zero
# ------------------------------------------------

for col in physical_columns:
    player_stats_df.loc[
        goalkeeper_mask,
        col
    ] = player_stats_df.loc[
        goalkeeper_mask,
        col
    ].fillna(0)

# ------------------------------------------------
# Missing values AFTER
# ------------------------------------------------

missing_after = (
    player_stats_df.loc[
        goalkeeper_mask,
        physical_columns
    ]
    .isna()
    .sum()
    .sum()
)

print(
    f"Missing goalkeeper physical cells after: "
    f"{missing_after:,}"
)

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

if missing_after == 0:
    print("ALL GOALKEEPER PHYSICAL BLANKS FIXED")
else:
    print("WARNING: SOME GOALKEEPER PHYSICAL BLANKS REMAIN")

GOALKEEPER PHYSICAL STATISTICS
Goalkeepers: 145
Missing goalkeeper physical cells before: 0
Missing goalkeeper physical cells after: 0

RESULT
ALL GOALKEEPER PHYSICAL BLANKS FIXED


## Step 102 — Save Corrected Player Statistics

Save the corrected player statistics table after applying the
zero-value rules.

The final CSV must contain:

- 1,248 players
- 63 columns
- 0 blank numeric/statistic cells
- Goalkeeper attacking statistics = 0 where not applicable
- Goalkeeper physical statistics = 0 where unavailable
- Goalkeeper statistics preserved where actual values exist
- Defending statistics = 0 where no event/statistic exists

In [20]:
# ================================================================
# SAVE FINAL PLAYER CSV — ORIGINAL NAME
# ================================================================

final_player_output = os.path.join(
    DATA_DIR,
    "FIFA_WorldCup_2026_Master_Player_Stats.csv"
)

player_stats_df.to_csv(
    final_player_output,
    index=False
)

# Verify
check_df = pd.read_csv(final_player_output)

print("=" * 70)
print("FINAL PLAYER CSV SAVED")
print("=" * 70)

print(f"File: {final_player_output}")
print(f"Rows: {len(check_df):,}")
print(f"Columns: {len(check_df.columns):,}")
print(f"Unique PlayerIDs: {check_df['PlayerID'].nunique():,}")
print(f"Blank cells: {check_df.isna().sum().sum():,}")

if check_df.isna().sum().sum() == 0:
    print("\nFINAL CSV VERIFIED — 0 BLANK CELLS")
else:
    print("\nWARNING: BLANK CELLS REMAIN")

FINAL PLAYER CSV SAVED
File: D:\ai course hw\FIFA_WorldCup_2026_Project\Data\FIFA_WorldCup_2026_Master_Player_Stats.csv
Rows: 1,248
Columns: 63
Unique PlayerIDs: 1,248
Blank cells: 0

FINAL CSV VERIFIED — 0 BLANK CELLS
